# ML Study Tracker 2: Classical ML

Split from `ml_study_tracker-clauded.ipynb`.


# Table of Contents

- [Algorithms](#algorithms)
  - [Linear Regression](#linear-regression)
  - [Logistic Regression](#logistic-regression)
  - [K-Nearest Neighbors](#k-nearest-neighbors)
  - [Naive Bayes](#naive-bayes)
  - [Decision Trees](#decision-trees)
  - [Random Forest](#random-forest)
  - [Gradient Boosting](#gradient-boosting)
  - [XGBoost / LightGBM / CatBoost](#xgboost--lightgbm--catboost)
  - [Support Vector Machines](#support-vector-machines)
  - [K-Means](#k-means)
  - [PCA](#pca)
- [Advanced Machine Learning](#advanced-machine-learning)
  - [Imbalanced Classification](#imbalanced-classification)
  - [Time Series Forecasting](#time-series-forecasting)
  - [Recommender Systems](#recommender-systems)
  - [Anomaly Detection](#anomaly-detection)
  - [Explainable AI](#explainable-ai)


In [1]:
# Common imports

try:
    import numpy as np
except ImportError:
    np = None

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        confusion_matrix, classification_report,
        mean_absolute_error, mean_squared_error, r2_score
    )
except ImportError:
    train_test_split = cross_val_score = GridSearchCV = None
    accuracy_score = precision_score = recall_score = f1_score = None
    confusion_matrix = classification_report = None
    mean_absolute_error = mean_squared_error = r2_score = None

# Add more imports as needed.


# Algorithms

## Linear Regression

### Study checklist
- [ ] Simple linear regression
- [ ] Multiple linear regression
- [ ] Assumptions
- [ ] Coefficient interpretation
- [ ] Regularization: Ridge and Lasso
- [ ] Residual analysis

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Simple linear regression

**Approach:**
- Protects model fit and interpretability: a single-feature line is the simplest way to confirm the response is roughly linear before trusting more complex models on the same data.
- Why this is the mathematically right approach: linear regression finds the slope and intercept that minimize the sum of squared errors between predicted and actual y values; setting the derivative of that squared-error loss to zero gives a closed-form solution, and under the Gauss-Markov assumptions this is provably the Best Linear Unbiased Estimator - no other linear, unbiased method has lower variance. In ML notation, the slope (m) is called the weight (w) and the intercept (b) is called the bias - y = mx + b and y = wx + b are literally the same equation; ML just renames them because the idea generalizes to many inputs (y = w1*x1 + w2*x2 + ... + b), where each input needs its own weight.
- For this checklist item: Fit a line y = mx + b to one feature; verify the slope and intercept match domain intuition before adding complexity.
- Code walkthrough: Check that model.coef_[0] and model.intercept_ recover slope=2, intercept=0 exactly (since y=2x here), and that predict([[5]]) returns 10.

**Learn more:**
- Website: [scikit-learn: Ordinary Least Squares](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Linear+Regression+Simple+linear+regression+machine+learning+theory)

**Trade-offs:**
- A single-feature fit is faster to sanity-check than multiple regression, but it can't tell you whether an omitted second feature is confounding the relationship you're seeing.
- A single feature rarely captures the full signal, so use it as a sanity-check baseline (does the sign/magnitude match domain intuition?) rather than a production model.

**Practical software engineering use cases:**
- When to use it: Use this when you need a fast, auditable first pass on a single strong predictor (e.g., price vs. size) before justifying a more complex model to stakeholders.
- When not to use it: Don't use it as your final model once you know two or more features jointly drive the outcome - you'll be leaving predictable variance on the table.


In [118]:
# Linear Regression - Simple linear regression (using scikit-learn)

import numpy as np                                   # NumPy: for creating and handling numeric arrays
from sklearn.linear_model import LinearRegression    # The linear regression model from scikit-learn

# Step 1: Create training data.
# scikit-learn expects X (features) as a 2D array: each row = one sample, each column = one feature.
# .reshape(-1, 1) converts [1, 2, 3, 4] into [[1], [2], [3], [4]]  (4 rows, 1 column).
X = np.array([1, 2, 3, 4]).reshape(-1, 1)
y = np.array([2, 4, 6, 8])                           # y (target) stays 1D: one value per sample

# Step 2: Create the model object. Nothing is learned yet - this just sets it up.
model = LinearRegression()

# Step 3: Train ("fit") the model. This is where it learns the best slope and intercept
# by minimizing the sum of squared errors between predictions and actual y values.
model.fit(X, y)

# Step 4: Inspect what was learned.
# model.coef_      -> the slope(s), one per feature (here just one)
# model.intercept_ -> where the line crosses the y-axis when X = 0
print("slope (coef_):", model.coef_[0])
print("intercept:", model.intercept_)

# Step 5: Use the trained model to predict y for the training points (and a new point, 5).
predictions = model.predict(X)                       # predicts y = slope * x + intercept for each row
print("predictions on training X:", predictions)
print("prediction for x = 5:", model.predict([[5]])[0])   # new input must also be 2D: [[5]]


slope (coef_): 2.0
intercept: 0.0
predictions on training X: [2. 4. 6. 8.]
prediction for x = 5: 10.0


### Checklist item: Multiple linear regression

**Approach:**
- Protects model fit and representation: adding features should reduce residual error without introducing multicollinearity that destabilizes coefficient estimates.
- Why this is the mathematically right approach: extending to y = w1*x1 + ... + wn*xn + b is still solved by the same least-squares principle, now written in matrix form as y = Xw + b; the optimal weights minimize ||y - Xw||^2, which has the closed-form solution w = (X^T X)^-1 X^T y whenever X^T X is invertible, i.e. features aren't perfectly collinear. More inputs just means more weights in the same weighted-sum-plus-bias structure.
- For this checklist item: Extend to multiple features y = w0 + w1*x1 + ... + wn*xn; check for multicollinearity (high VIF) which inflates coefficient variance.
- Code walkthrough: Check that model.coef_ prints one weight per column (size, bedrooms) in the same order as X's columns, and that model.score(X, y) (R-squared) is high since this toy data is built to be linear.

**Learn more:**
- Website: [scikit-learn: Ordinary Least Squares](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Linear+Regression+Multiple+linear+regression+machine+learning+theory)

**Trade-offs:**
- Adding features raises the model's ceiling versus simple regression, but every added feature is another opportunity for multicollinearity or noise to creep into the coefficient estimates.
- More features generally improve R-squared even when uninformative (overfitting risk), so watch adjusted R-squared, VIF for collinearity, and out-of-sample performance rather than in-sample fit alone.

**Practical software engineering use cases:**
- When to use it: Use this when you have a handful of numeric predictors and need a model whose coefficients a non-ML stakeholder (finance, ops) can read directly.
- When not to use it: Don't use it when your predictors are highly collinear, such as overlapping engineered features, without first checking VIF - the coefficients become unstable and misleading.


In [119]:
# Linear Regression - Multiple linear regression (using scikit-learn)

import numpy as np
from sklearn.linear_model import LinearRegression

# Multiple linear regression = same idea, but with MORE THAN ONE feature (column) in X.
# The model learns one coefficient (weight) per feature:
#   y = coef1 * size + coef2 * bedrooms + intercept

# Step 1: Training data - each row is one house: [size_in_100s_sqft, number_of_bedrooms]
X = np.array([
    [10, 2],    # house 1: 1000 sqft, 2 bedrooms
    [15, 3],    # house 2: 1500 sqft, 3 bedrooms
    [20, 3],    # house 3
    [25, 4],    # house 4
    [30, 4],    # house 5
])
y = np.array([200, 280, 340, 410, 470])              # price in thousands (the target we predict)

# Step 2: Create and train the model in one go (fit returns the model, so we can chain).
model = LinearRegression().fit(X, y)

# Step 3: Look at what was learned - one coefficient PER feature, in the same order as columns.
print("coefficients [size, bedrooms]:", model.coef_)
print("intercept:", model.intercept_)

# Step 4: Predict the price of a NEW house: 2200 sqft (22), 3 bedrooms.
new_house = [[22, 3]]                                # still 2D: a list containing one row
print("predicted price:", model.predict(new_house)[0])

# Step 5: R-squared score = fraction of variance in y the model explains (1.0 = perfect).
print("R^2 on training data:", model.score(X, y))


coefficients [size, bedrooms]: [11.73333333 16.66666667]
intercept: 51.999999999999886
predicted price: 360.1333333333333
R^2 on training data: 0.9994074074074074


### Checklist item: Assumptions

**Approach:**
- Protects evaluation integrity: linearity, homoscedasticity, independence, and normally distributed residuals are what make the model's p-values, confidence intervals, and predictions trustworthy.
- Why this is the mathematically right approach: the Gauss-Markov theorem guarantees OLS is the Best Linear Unbiased Estimator only when its assumptions hold (linearity, independent errors, constant error variance, no perfect collinearity); the least-squares math itself will compute a 'best fit' line regardless of whether these hold, so verifying assumptions is what confirms the optimality guarantee actually applies to your specific case.
- For this checklist item: Linear regression assumes linearity, independence of errors, homoscedasticity, and approximate normality of residuals â€” violating them misleads inference.
- Code walkthrough: Check that residuals.mean() lands near 0 and that the first-half and second-half residual std values are close to each other (confirming homoscedasticity on this synthetic data), and that about 68% of residuals fall within one std, matching the normality assumption.

**Learn more:**
- Website: [scikit-learn: Ordinary Least Squares](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Linear+Regression+Assumptions+machine+learning+theory)

**Trade-offs:**
- Checking assumptions costs a few extra diagnostic plots, but skipping them trades a small time saving now for a model whose p-values and intervals can't be trusted later.
- Violated assumptions don't always mean abandoning linear regression; a transform (log, Box-Cox) or a robust/weighted variant can fix it more cheaply than switching model families.

**Practical software engineering use cases:**
- When to use it: Use this checklist before trusting any p-value, confidence interval, or extrapolation from a shipped linear model, not just once at prototype time.
- When not to use it: Don't skip it just because the model's R-squared looks good - a high R-squared can coexist with badly violated assumptions that make every inference wrong.


In [120]:
# Linear Regression - Assumptions (checked with scikit-learn + numpy)

# Linear regression gives trustworthy results only when these assumptions roughly hold:
#   1. Linearity        - the relationship between X and y is a straight line
#   2. Independence     - errors (residuals) are not correlated with each other
#   3. Homoscedasticity - errors have roughly constant spread across all X values
#   4. Normality        - errors are roughly bell-shaped (matters for confidence intervals)
# The main tool for checking them: RESIDUALS = actual y - predicted y.

import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)                      # random generator with fixed seed (reproducible)

# Create data that IS truly linear: y = 3x + 5 + small random noise
X = np.arange(1, 21).reshape(-1, 1)                  # values 1..20 as a column
y = 3 * X.ravel() + 5 + rng.normal(0, 2, size=20)    # .ravel() flattens X back to 1D for the math

model = LinearRegression().fit(X, y)                 # train the model
residuals = y - model.predict(X)                     # residual = actual - predicted, per sample

# Check 1 (linearity + zero-mean errors): residual mean should be ~0 if the line fits well.
print("mean of residuals (want ~0):", round(residuals.mean(), 4))

# Check 3 (constant spread): compare residual spread in first half vs second half of X.
# If the second half is much larger, errors grow with X -> heteroscedasticity (assumption broken).
print("residual std, first half :", round(residuals[:10].std(), 3))
print("residual std, second half:", round(residuals[10:].std(), 3))

# Check 4 (normality, quick-and-dirty): ~68% of residuals should fall within 1 std of 0.
within_1_std = np.mean(np.abs(residuals) < residuals.std())
print("fraction of residuals within 1 std (want ~0.68):", round(within_1_std, 2))


mean of residuals (want ~0): -0.0
residual std, first half : 1.807
residual std, second half: 1.44
fraction of residuals within 1 std (want ~0.68): 0.6


### Checklist item: Coefficient interpretation

**Approach:**
- Protects user value and decision-making: stakeholders will read "a one-unit increase in X changes Y by beta" literally, so the units and scale of each coefficient must be communicated correctly.
- Why this is the mathematically right approach: because the model is literally y = w1*x1 + w2*x2 + ... + b, calculus says the partial derivative of y with respect to any single xi, holding the others fixed, is exactly wi - that's precisely why 'holding everything else constant, a one-unit increase in xi changes y by wi' is a direct consequence of the equation's structure, not an approximation.
- For this checklist item: Each coefficient wi means: 'holding all other features constant, a one-unit increase in xi changes y by wi units' â€” only valid when assumptions hold.
- Code walkthrough: Check the printed sentence for each feature (e.g. '+1 years_experience changes predicted salary by X k') against model.coef_ for that feature - that mapping from coefficient to plain-English sentence is the whole point of this item.

**Learn more:**
- Website: [scikit-learn: Ordinary Least Squares](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Linear+Regression+Coefficient+interpretation+machine+learning+theory)

**Trade-offs:**
- A linear model's coefficients are far easier to explain to a non-technical stakeholder than a tree ensemble's feature importances, but that interpretability evaporates the moment two features are collinear.
- Coefficients are only cleanly interpretable when features are on comparable, untransformed scales and not collinear; standardized coefficients aid comparison but lose the original units.

**Practical software engineering use cases:**
- When to use it: Use this when a stakeholder asks how much a feature actually moves the outcome and needs a number they can act on, like a pricing lever.
- When not to use it: Don't hand over raw coefficients when features are on wildly different scales or units without translating them first - a big coefficient may just mean a tiny unit.


In [121]:
# Linear Regression - Coefficient interpretation (using scikit-learn)

import numpy as np
from sklearn.linear_model import LinearRegression

# The KEY interpretation rule:
#   "Holding all other features constant, increasing this feature by 1 unit
#    changes the prediction by <coefficient> units."

feature_names = ["years_experience", "certifications"]

X = np.array([
    [1, 0],
    [3, 1],
    [5, 1],
    [7, 2],
    [10, 3],
])
y = np.array([45, 60, 75, 92, 118])                  # salary in thousands

model = LinearRegression().fit(X, y)                 # train

# zip() pairs each feature name with its learned coefficient so we can print them together.
for name, coef in zip(feature_names, model.coef_):
    print(f"{name}: {coef:.2f}")
    print(f"  -> +1 {name} changes predicted salary by {coef:.2f}k, other features held fixed")

# The intercept = prediction when ALL features are 0 (may or may not be meaningful in real life).
print(f"intercept: {model.intercept_:.2f} (predicted salary with 0 experience, 0 certifications)")

# CAUTION: coefficients are in the feature's own units. A "big" coefficient on a feature
# measured in tiny units is not automatically more important - scale features first to compare.


years_experience: 7.50
  -> +1 years_experience changes predicted salary by 7.50k, other features held fixed
certifications: 1.92
  -> +1 certifications changes predicted salary by 1.92k, other features held fixed
intercept: 36.31 (predicted salary with 0 experience, 0 certifications)


### Checklist item: Regularization: Ridge and Lasso

**Approach:**
- Protects model fit and generalization: regularization exists to control variance when features are correlated or numerous relative to sample size, trading a small amount of bias for a large reduction in overfitting risk.
- Why this is the mathematically right approach: Ridge adds a penalty (alpha times the sum of squared weights) to the squared-error loss, which shrinks every weight toward zero without reaching exactly zero, since the penalty's derivative is proportional to the weight itself; Lasso penalizes the sum of absolute weights instead, whose derivative is constant regardless of weight size - that's precisely why Lasso can push a weight all the way to zero, a geometric consequence of the L1 penalty's diamond shape versus L2's circular shape in weight space.
- For this checklist item: Ridge (L2) shrinks coefficients toward zero; Lasso (L1) zeroes out less-important ones â€” both reduce variance at the cost of a small bias.
- Code walkthrough: Compare the three printed coefficient vectors against the true pattern [4, 2, 0, 0, 0]: ridge should shrink all 5 coefficients toward 0 while lasso should push the 3 noise coefficients to exactly 0.

**Learn more:**
- Website: [scikit-learn: Ordinary Least Squares](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Linear+Regression+Regularization%3A+Ridge+and+Lasso+machine+learning+theory)

**Trade-offs:**
- Compared to plain OLS, both penalties trade a small amount of bias for meaningfully lower variance on correlated or high-dimensional data, but neither fixes non-linearity the way a tree-based model would.
- Ridge keeps all features (good when many weakly-informative predictors exist) while Lasso does implicit feature selection (good for sparse, interpretable models), but Lasso can arbitrarily zero out one of two correlated important features.

**Practical software engineering use cases:**
- When to use it: Use this when you have many candidate features (or engineered polynomial/interaction terms) and suspect only a few are truly informative.
- When not to use it: Don't reach for it as a fix for non-linearity or bad features - regularization only tames variance from an already-reasonable linear feature set, it can't rescue a fundamentally wrong model shape.


In [122]:
# Linear Regression - Regularization: Ridge and Lasso (using scikit-learn)

import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Regularization adds a penalty for LARGE coefficients to reduce overfitting:
#   Ridge (L2): penalty = alpha * sum(coef^2)   -> shrinks coefficients toward 0
#   Lasso (L1): penalty = alpha * sum(|coef|)   -> can shrink some coefficients EXACTLY to 0
#                                                  (so Lasso also does feature selection)
# alpha controls penalty strength: 0 = plain linear regression, larger = more shrinkage.

rng = np.random.default_rng(0)
n = 50                                               # number of samples

# Build 5 features, but only the first 2 actually influence y. Features 3-5 are pure noise.
X = rng.normal(size=(n, 5))                          # 50 rows x 5 columns of random values
y = 4 * X[:, 0] + 2 * X[:, 1] + rng.normal(0, 0.5, n)  # y depends on feature 0 and 1 only

# Train all three models on the same data.
plain = LinearRegression().fit(X, y)                 # no penalty
ridge = Ridge(alpha=1.0).fit(X, y)                   # L2 penalty, alpha=1
lasso = Lasso(alpha=0.1).fit(X, y)                   # L1 penalty, alpha=0.1

# Compare the learned coefficients side by side (rounded for readability).
print("true pattern:   [4, 2, 0, 0, 0]")
print("plain linear:  ", np.round(plain.coef_, 2))
print("ridge (L2):    ", np.round(ridge.coef_, 2), " <- all shrunk a little")
print("lasso (L1):    ", np.round(lasso.coef_, 2), " <- noise features pushed to exactly 0")


true pattern:   [4, 2, 0, 0, 0]
plain linear:   [ 3.94  2.03 -0.03 -0.07  0.04]
ridge (L2):     [ 3.82  2.01 -0.02 -0.08  0.05]  <- all shrunk a little
lasso (L1):     [ 3.82  1.95 -0.   -0.    0.  ]  <- noise features pushed to exactly 0


### Checklist item: Residual analysis

**Approach:**
- Protects evaluation integrity: residual plots are the diagnostic tool for catching violated assumptions (non-linearity, heteroscedasticity, outliers) that summary metrics like R-squared can hide.
- Why this is the mathematically right approach: OLS is derived by minimizing the sum of squared residuals, so the residuals it leaves behind are mathematically guaranteed to sum to zero and be uncorrelated with the fitted values - any visible pattern remaining in a residual plot is direct evidence the assumed linear equation form doesn't match the true relationship in the data.
- For this checklist item: Plot residuals vs predicted values; a funnel shape signals heteroscedasticity, and a curve signals a missing non-linear term.
- Code walkthrough: Check that the three printed average residuals form a pattern like [-, +, -] rather than being near 0 in every third - that curved pattern is the tell-tale sign the straight line is missing the true x-squared relationship.

**Learn more:**
- Website: [scikit-learn: Ordinary Least Squares](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Linear+Regression+Residual+analysis+machine+learning+theory)

**Trade-offs:**
- Residual plots cost nothing beyond a single scatter plot, but they're the only diagnostic here that would have caught the curved relationship a summary R-squared number hides.
- A good R-squared with patterned residuals is a red flag, not a pass, so always plot residuals vs. fitted values and vs. each feature before shipping a linear model.

**Practical software engineering use cases:**
- When to use it: Use this as a standard pre-deployment check on every linear model, especially ones feeding a dashboard or automated decision.
- When not to use it: Don't skip it because the model already shipped and works fine - a residual pattern found late is a sign the model has been systematically wrong in production the whole time.


In [123]:
# Linear Regression - Residual analysis (using scikit-learn)

import numpy as np
from sklearn.linear_model import LinearRegression

# Residual = actual y - predicted y. Analyzing residuals tells you WHERE the model fails.
# A good linear fit leaves residuals that look like pure random noise around 0.
# A PATTERN in the residuals means the model is missing something (e.g., a curve).

rng = np.random.default_rng(1)
X = np.linspace(1, 10, 30).reshape(-1, 1)            # 30 evenly spaced points from 1 to 10

# Deliberately create CURVED data (y depends on x^2) and fit a STRAIGHT line to it.
y = 0.5 * X.ravel() ** 2 + rng.normal(0, 1, 30)

model = LinearRegression().fit(X, y)                 # fit a straight line to curved data
residuals = y - model.predict(X)                     # compute residuals

# Split residuals into thirds along X. For a correct model, each third averages ~0.
# Here we will see negative / positive / negative-ish pattern -> a U-shape -> missed curvature.
thirds = np.array_split(residuals, 3)                # split the residual array into 3 chunks
for i, chunk in enumerate(thirds, start=1):
    print(f"average residual in third {i} of X: {chunk.mean():+.2f}")

print()
print("A pattern like [-, +, -] or [+, -, +] means the straight line misses a curve.")
print("Fix: add polynomial features, transform X, or use a non-linear model.")


average residual in third 1 of X: +1.64
average residual in third 2 of X: -3.29
average residual in third 3 of X: +1.65

A pattern like [-, +, -] or [+, -, +] means the straight line misses a curve.
Fix: add polynomial features, transform X, or use a non-linear model.


In [124]:
# Practice: Linear Regression

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Logistic Regression

### Study checklist
- [ ] Binary classification
- [ ] Sigmoid function
- [ ] Decision threshold
- [ ] Odds and log-odds intuition
- [ ] Regularization
- [ ] Interpreting coefficients

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Binary classification

**Approach:**
- Protects model fit: logistic regression models the log-odds of the positive class, so it needs a genuinely binary (or one-vs-rest) target to be well-specified.
- Why this is the mathematically right approach: logistic regression models the log-odds of the positive class as a linear function, log(p/(1-p)) = w*x + b, because log-odds (unlike a raw probability) can range over all real numbers, making it compatible with an unbounded linear equation - solving for p by inverting this equation is exactly what produces the sigmoid function used to recover a valid probability.
- For this checklist item: Logistic regression outputs P(y=1|x) via a linear combination passed through sigmoid; the decision boundary is linear in feature space.
- Code walkthrough: Check that predict_proba's P(pass) rises smoothly with hours studied for the three new students, and that predict()'s hard 0/1 labels match wherever P(pass) crosses 0.5.

**Learn more:**
- Website: [scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Logistic+Regression+Binary+classification+machine+learning+theory)

**Trade-offs:**
- As a linear classifier, logistic regression trains and serves far cheaper than an SVM or gradient-boosted tree, but it cannot represent a boundary that isn't (nearly) a straight line, no matter how much data you feed it.
- It gives a well-calibrated probability out of the box for balanced data, but that calibration degrades under class imbalance unless corrected with class weights or threshold tuning.

**Practical software engineering use cases:**
- When to use it: Use this as your first classifier for a binary outcome (churn/no-churn, approve/deny) when you need calibrated probabilities and an auditable decision boundary.
- When not to use it: Don't use it when the true boundary between classes is known to be highly non-linear, such as image or audio classification, without heavy feature engineering first.


In [125]:
# Logistic Regression - Binary classification (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression

# Logistic regression predicts the PROBABILITY of class 1 (e.g., pass/fail, spam/ham).
# Despite the name, it is a CLASSIFICATION algorithm, not regression.

# Step 1: Training data - hours studied (feature) and pass/fail (target: 0 = fail, 1 = pass).
X = np.array([1, 2, 3, 4, 5, 6, 7, 8]).reshape(-1, 1)   # reshape to 2D (8 rows, 1 column)
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])                  # first 4 failed, last 4 passed

# Step 2: Create and train the model.
model = LogisticRegression()
model.fit(X, y)

# Step 3: Predict CLASSES for new students (uses 0.5 probability cutoff by default).
new_students = np.array([2.5, 4.5, 7.0]).reshape(-1, 1)
print("predicted classes:", model.predict(new_students))       # array of 0s and 1s

# Step 4: Predict PROBABILITIES - often more useful than the hard 0/1 answer.
# predict_proba returns one row per sample: [P(class 0), P(class 1)].
proba = model.predict_proba(new_students)
for hours, p in zip(new_students.ravel(), proba):
    print(f"{hours} hours -> P(fail)={p[0]:.2f}, P(pass)={p[1]:.2f}")


predicted classes: [0 0 1]
2.5 hours -> P(fail)=0.91, P(pass)=0.09
4.5 hours -> P(fail)=0.50, P(pass)=0.50
7.0 hours -> P(fail)=0.05, P(pass)=0.95


### Checklist item: Sigmoid function

**Approach:**
- Protects representation: the sigmoid squashes an unbounded linear score into a valid [0,1] probability, which is what makes the output interpretable as a likelihood of the positive class.
- Why this is the mathematically right approach: the sigmoid, sigma(z) = 1/(1+e^-z), is the exact algebraic inverse of the log-odds transformation, so it isn't an arbitrary squashing function - it's the specific function required to convert a linear score back into a probability consistent with the log-odds model. It also has the convenient calculus property sigma'(z) = sigma(z)(1-sigma(z)), which keeps the training gradient simple to compute.
- For this checklist item: The sigmoid squashes any real number to (0,1) so the output is interpretable as a probability; the 0.5 crossing is the default decision boundary.
- Code walkthrough: Check that expit(z_values) matches the sigmoid shape (near 0, 0.5, near 1 at the extremes and center), then confirm the last two printed lines are identical - proof that predict_proba is just sigmoid(decision_function).

**Learn more:**
- Website: [scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Logistic+Regression+Sigmoid+function+machine+learning+theory)

**Trade-offs:**
- The sigmoid is what makes logistic regression's output a genuine probability rather than an arbitrary score, but that same squashing means very confident (near-0/1) predictions carry almost no gradient signal left to learn from.
- Extreme logits saturate the sigmoid near 0 or 1, which can cause vanishing gradients during training and overconfident predictions at inference, so watch for feature scales that push scores to extremes.

**Practical software engineering use cases:**
- When to use it: Use this understanding whenever you need to explain why the model outputs a probability to a teammate or auditor, or when debugging a probability stuck at 0 or 1.
- When not to use it: Don't assume predict_proba's raw output is well-calibrated across the whole range without checking - extreme, always-saturated scores are a sign to inspect feature scaling first.


In [126]:
# Logistic Regression - Sigmoid function (scipy + scikit-learn)

import numpy as np
from scipy.special import expit                      # expit = the sigmoid function, 1/(1+e^-z)
from sklearn.linear_model import LogisticRegression

# The sigmoid squashes ANY number into the range (0, 1), which lets us read it as a probability:
#   sigmoid(-inf) -> 0     sigmoid(0) -> 0.5     sigmoid(+inf) -> 1

z_values = np.array([-4, -2, 0, 2, 4])               # example raw scores
print("z:        ", z_values)
print("sigmoid(z):", np.round(expit(z_values), 3))   # apply sigmoid to each value

# Inside logistic regression:
#   step 1: compute a raw linear score z = coef * x + intercept   (this is decision_function)
#   step 2: probability = sigmoid(z)                              (this is predict_proba)
X = np.array([1, 2, 3, 4, 5, 6]).reshape(-1, 1)
y = np.array([0, 0, 0, 1, 1, 1])
model = LogisticRegression().fit(X, y)

z = model.decision_function([[3.5]])                 # raw score for x = 3.5
print()
print("raw score z for x=3.5:      ", np.round(z, 3))
print("sigmoid(z) computed by hand: ", np.round(expit(z), 3))
print("model.predict_proba class 1: ", np.round(model.predict_proba([[3.5]])[0, 1], 3))
# The last two lines match: predict_proba IS just sigmoid(decision_function).


z:         [-4 -2  0  2  4]
sigmoid(z): [0.018 0.119 0.5   0.881 0.982]

raw score z for x=3.5:       [0.]
sigmoid(z) computed by hand:  [0.5]
model.predict_proba class 1:  0.5


### Checklist item: Decision threshold

**Approach:**
- Protects deployment reliability and user value: the default 0.5 cutoff is arbitrary and rarely matches the real-world cost of false positives vs. false negatives.
- Why this is the mathematically right approach: because the model outputs P(y=1|x) directly as a number between 0 and 1, converting that into a hard 0/1 decision inherently requires picking a cutoff - 0.5 is the natural default since it's the point where the model considers both classes equally likely (log-odds = 0), but nothing in the underlying math requires using that exact value for the final business decision.
- For this checklist item: Adjust the threshold away from 0.5 to trade precision for recall based on the business cost of false positives vs false negatives.
- Code walkthrough: Check how the same proba_class1 array produces different 0/1 labels across the three thresholds - the points near the boundary should flip from 1 to 0 as the threshold rises from 0.3 to 0.7.

**Learn more:**
- Website: [scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Logistic+Regression+Decision+threshold+machine+learning+theory)

**Trade-offs:**
- Unlike a tree ensemble's implicit decision rules, logistic regression exposes a single explicit threshold you can move after training - a flexibility most other classifiers don't hand you as cleanly.
- Moving the threshold trades precision for recall; pick it using the business cost ratio or a precision-recall curve rather than leaving it at 0.5 by default.

**Practical software engineering use cases:**
- When to use it: Use this whenever you're deploying a classifier into a workflow where false positives and false negatives have different real costs, like fraud review vs. auto-block.
- When not to use it: Don't leave the threshold at the scikit-learn default of 0.5 in a production system without at least once checking it against the actual cost of each error type.


In [127]:
# Logistic Regression - Decision threshold (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression

# model.predict() uses a DEFAULT threshold of 0.5: predict 1 if P(class 1) >= 0.5.
# But 0.5 is a choice, not a law. You can move the threshold to match the business cost:
#   - Lower it (e.g., 0.3) to catch more positives (fewer misses, more false alarms)
#   - Raise it (e.g., 0.7) to only flag when very confident (fewer false alarms, more misses)

X = np.array([1, 2, 3, 4, 5, 6, 7, 8]).reshape(-1, 1)
y = np.array([0, 0, 0, 1, 0, 1, 1, 1])
model = LogisticRegression().fit(X, y)               # train

test_X = np.array([3, 4, 5, 6]).reshape(-1, 1)       # some points near the boundary
proba_class1 = model.predict_proba(test_X)[:, 1]     # [:, 1] takes only the P(class 1) column

# Apply three different thresholds to the SAME probabilities and compare the labels.
for threshold in [0.3, 0.5, 0.7]:
    labels = (proba_class1 >= threshold).astype(int) # True/False -> 1/0
    print(f"threshold {threshold}: probabilities {np.round(proba_class1, 2)} -> labels {labels}")

# Example: for cancer screening you might use 0.3 (missing a case is worse than a false alarm);
# for auto-blocking user accounts you might use 0.9 (false accusations are costly).


threshold 0.3: probabilities [0.21 0.39 0.61 0.79] -> labels [0 1 1 1]
threshold 0.5: probabilities [0.21 0.39 0.61 0.79] -> labels [0 0 1 1]
threshold 0.7: probabilities [0.21 0.39 0.61 0.79] -> labels [0 0 0 1]


### Checklist item: Odds and log-odds intuition

**Approach:**
- Protects coefficient interpretability: log-odds is the space where logistic regression is linear, so understanding it is what lets you translate a coefficient into a multiplicative change in odds.
- Why this is the mathematically right approach: because the model equation is literally log(odds) = w*x + b, exponentiating both sides gives odds = e^b * (e^w)^x, which shows algebraically that increasing x by 1 multiplies the odds by exactly e^w - the odds ratio falls directly out of taking e to the power of the model's own equation, it isn't an empirical approximation.
- For this checklist item: Log-odds = wÂ·x + b; a coefficient wi means: one unit increase in xi multiplies the odds of y=1 by exp(wi) â€” the natural interpretation of logistic regression.
- Code walkthrough: Check that the printed ratio odds5/odds4 matches odds_ratio (e^coef) computed earlier in the cell - that numeric match is the proof that a logistic regression coefficient is a constant multiplier on the odds.

**Learn more:**
- Website: [scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Logistic+Regression+Odds+and+log-odds+intuition+machine+learning+theory)

**Trade-offs:**
- Odds ratios give logistic regression a mathematically clean way to express how much a feature matters, more rigorous than reading a raw coefficient off a linear regression on a binary outcome.
- Odds ratios are easy for non-technical stakeholders to misread as probability changes, so always translate back to a concrete probability example when communicating results.

**Practical software engineering use cases:**
- When to use it: Use this when writing a report or model card that needs to state a one-unit increase in X multiplies the odds of Y by Z, accurately.
- When not to use it: Don't present an odds ratio to a general audience without translating it into a concrete probability example - odds ratios are routinely misread as probability changes.


In [128]:
# Logistic Regression - Odds and log-odds intuition (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression

# Definitions:
#   odds     = P / (1 - P)          e.g., P=0.75 -> odds = 3 ("3 to 1")
#   log-odds = ln(odds)             this is the raw score z inside logistic regression
# Logistic regression is LINEAR in log-odds: each +1 unit of x adds `coef` to the log-odds,
# which MULTIPLIES the odds by e^coef. That number e^coef is called the ODDS RATIO.

X = np.array([1, 2, 3, 4, 5, 6, 7, 8]).reshape(-1, 1)
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])
model = LogisticRegression().fit(X, y)

coef = model.coef_[0][0]                             # the single learned coefficient
odds_ratio = np.exp(coef)                            # e^coef = odds multiplier per +1 in x
print(f"coefficient (log-odds change per +1 hour): {coef:.3f}")
print(f"odds ratio (odds multiplier per +1 hour):  {odds_ratio:.3f}")
print()

# Verify it numerically: odds at x=4 vs x=5 should differ by exactly that ratio.
p4 = model.predict_proba([[4]])[0, 1]                # P(pass) at 4 hours
p5 = model.predict_proba([[5]])[0, 1]                # P(pass) at 5 hours
odds4 = p4 / (1 - p4)                                # convert each probability to odds
odds5 = p5 / (1 - p5)
print(f"odds at 4h: {odds4:.3f}   odds at 5h: {odds5:.3f}")
print(f"ratio odds5/odds4 = {odds5 / odds4:.3f}  (matches e^coef above)")


coefficient (log-odds change per +1 hour): 1.170
odds ratio (odds multiplier per +1 hour):  3.221

odds at 4h: 0.557   odds at 5h: 1.795
ratio odds5/odds4 = 3.221  (matches e^coef above)


### Checklist item: Regularization

**Approach:**
- Protects generalization: L2 (or L1) regularization prevents coefficients from exploding on separable or high-dimensional data, where unregularized logistic regression can fail to converge.
- Why this is the mathematically right approach: adding an L2 penalty (C controls its inverse strength) to the log-loss being minimized keeps weights from growing without bound on separable or high-dimensional data - without it, on perfectly separable data the unconstrained log-loss minimum is achieved by weights that grow to infinity, since the sigmoid can always be pushed closer to 0 or 1 by increasing weight magnitude further.
- For this checklist item: L1/L2 regularization on logistic regression prevents overfitting on small datasets; C (sklearn) is the inverse of lambda â€” smaller C = stronger regularization.
- Code walkthrough: Check that the printed coefficient vector shrinks toward all-near-zero at C=0.01 and grows toward the unregularized values at C=100 - since C is the inverse of regularization strength, smaller C means a stronger penalty.

**Learn more:**
- Website: [scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Logistic+Regression+Regularization+machine+learning+theory)

**Trade-offs:**
- Regularization is on by default in scikit-learn's LogisticRegression, unlike plain LinearRegression, which quietly protects against separable or high-dimensional data at the cost of needing to know to tune C at all.
- Regularization strength directly shrinks and biases coefficients, so heavily regularized models can under-report the true effect size of a feature; don't over-interpret shrunk coefficients as small real-world effects.

**Practical software engineering use cases:**
- When to use it: Use this (and check C) whenever your feature count is large relative to your sample size, or when classes are close to perfectly separable and training might otherwise fail to converge.
- When not to use it: Don't leave C at its default across wildly different datasets without at least a quick sweep - the right regularization strength is data-dependent, not universal.


In [129]:
# Logistic Regression - Regularization (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression

# scikit-learn's LogisticRegression is regularized BY DEFAULT (L2 penalty).
# The knob is C = 1 / regularization strength  (note: INVERSE!):
#   small C (0.01)  -> strong penalty -> small coefficients -> simpler model
#   large C (100)   -> weak penalty   -> coefficients grow freely -> can overfit

rng = np.random.default_rng(3)
n = 40

# 4 features, but only feature 0 truly matters for the class label.
X = rng.normal(size=(n, 4))
y = (X[:, 0] + rng.normal(0, 0.3, n) > 0).astype(int)   # label based on feature 0 + noise

# Train the same model with three different C values and watch the coefficients change.
for C in [0.01, 1, 100]:
    model = LogisticRegression(C=C).fit(X, y)        # C set here; everything else default
    print(f"C={C:<6} coefficients: {np.round(model.coef_[0], 2)}")

print()
print("Small C squashes ALL coefficients toward 0 (strong regularization).")
print("Use penalty='l1' with solver='liblinear' if you want Lasso-style exact zeros.")


C=0.01   coefficients: [ 0.16  0.    0.01 -0.02]
C=1      coefficients: [ 2.44  0.24  0.   -0.22]
C=100    coefficients: [15.18  1.36 -1.04  0.47]

Small C squashes ALL coefficients toward 0 (strong regularization).
Use penalty='l1' with solver='liblinear' if you want Lasso-style exact zeros.


### Checklist item: Interpreting coefficients

**Approach:**
- Protects user value: a logistic coefficient describes a change in log-odds, not probability, so misreading it as a probability change is a common and costly communication error.
- Why this is the mathematically right approach: since the model is log(odds) = w1*x1 + ... + wn*xn + b, the same partial-derivative logic as multiple linear regression applies, but to log-odds rather than to y directly - which is precisely why a logistic coefficient wi tells you the change in log-odds, not the change in probability, per unit of xi.
- For this checklist item: Logistic regression coefficients represent log-odds changes; exponentiate them to get odds ratios, which are easier to explain to non-technical stakeholders.
- Code walkthrough: Check that the sign of each coef matches the direction sentence printed below it (positive coef means 'raises the odds'), and that the odds ratio (np.exp(coef)) is only meaningful because both features were scaled first.

**Learn more:**
- Website: [scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Logistic+Regression+Interpreting+coefficients+machine+learning+theory)

**Trade-offs:**
- Compared to a black-box classifier, a logistic regression coefficient gives you a directly quotable effect size - but only once you remember to scale features first, an extra step tree-based models don't require.
- Interpretation only holds "all else equal," which breaks down when features are correlated, so report interaction or marginal effects when features aren't independent.

**Practical software engineering use cases:**
- When to use it: Use this when a regulator, auditor, or credit/risk team needs to know exactly which features push a prediction up or down and by how much.
- When not to use it: Don't interpret coefficients on unscaled features side by side to claim one matters more than another - that comparison is only valid once features are standardized.


In [130]:
# Logistic Regression - Interpreting coefficients (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Reading a logistic regression coefficient:
#   sign     : positive -> feature pushes prediction TOWARD class 1; negative -> toward class 0
#   e^coef   : odds ratio - how much the odds of class 1 multiply per +1 unit of the feature
# To COMPARE coefficient sizes across features, scale features first (same units).

feature_names = ["income_scaled", "debt_scaled"]

rng = np.random.default_rng(7)
n = 60
income = rng.normal(50, 15, n)                       # income in thousands
debt = rng.normal(20, 8, n)                          # debt in thousands
# Approved if income is high and debt is low (plus noise):
approved = ((income - debt + rng.normal(0, 10, n)) > 30).astype(int)

X = np.column_stack([income, debt])                  # stack the two 1D arrays into 2 columns
X_scaled = StandardScaler().fit_transform(X)         # scale: mean 0, std 1 per column

model = LogisticRegression().fit(X_scaled, approved) # train on the SCALED features

for name, coef in zip(feature_names, model.coef_[0]):
    direction = "raises" if coef > 0 else "lowers"   # sign tells the direction of the effect
    print(f"{name}: coef={coef:+.2f}, odds ratio={np.exp(coef):.2f}")
    print(f"  -> +1 std of this feature {direction} the odds of approval "
          f"by a factor of {np.exp(abs(coef)):.2f}")


income_scaled: coef=+1.82, odds ratio=6.16
  -> +1 std of this feature raises the odds of approval by a factor of 6.16
debt_scaled: coef=-0.74, odds ratio=0.48
  -> +1 std of this feature lowers the odds of approval by a factor of 2.09


In [131]:
# Practice: Logistic Regression

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## K-Nearest Neighbors

### Study checklist
- [ ] Distance-based learning
- [ ] Choosing K
- [ ] Scaling requirement
- [ ] Classification and regression use cases
- [ ] Curse of dimensionality

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Distance-based learning

**Approach:**
- Protects representation: KNN has no training phase and makes every prediction from raw distances, so the quality of the feature space directly determines prediction quality.
- Why this is the mathematically right approach: KNN's prediction rule, averaging or voting among the k closest points by Euclidean distance, sqrt(sum((xi-x'i)^2)), directly encodes the assumption that points close together in feature space should have similar labels; it needs no fitted equation because the 'model' is simply the geometry of the stored training data itself.
- For this checklist item: KNN classifies a new point by majority vote of its K nearest neighbors in feature space; distance is usually Euclidean after scaling.
- Code walkthrough: Check that the 3 neighbors printed by kneighbors() are genuinely the closest points to [2, 2] in X_train, and that the majority label among them matches the model's predict() output.

**Learn more:**
- Website: [scikit-learn: Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Nearest+Neighbors+Distance-based+learning+machine+learning+theory)

**Trade-offs:**
- KNN needs no training step at all, unlike every parametric model in this notebook, but that convenience is repaid at prediction time, when it must scan the stored data instead of applying a few learned weights.
- Prediction cost scales with dataset size since there is no compressed model, which is fine for small reference sets but becomes a latency problem at production scale without approximate nearest-neighbor indexing.

**Practical software engineering use cases:**
- When to use it: Use this for a quick, no-training-required baseline or for recommendation-style find-similar-items features on a small reference set.
- When not to use it: Don't use it as a production real-time classifier over millions of reference rows without an approximate nearest-neighbor index - a naive scan won't meet latency budgets.


In [132]:
# K-Nearest Neighbors - Distance-based learning (using scikit-learn)

import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# KNN has NO training phase in the usual sense - it memorizes the data.
# To classify a new point: find the K closest training points, let them vote.

# Step 1: Training data - [monthly_spend, visits_per_month] and a spending label.
X_train = np.array([[0, 0], [1, 1], [1, 0], [5, 5], [6, 5], [5, 6]])
y_train = np.array(["low", "low", "low", "high", "high", "high"])

# Step 2: Create the classifier with K=3 (each prediction uses the 3 nearest neighbors).
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)                          # "fit" here just stores the data

# Step 3: Classify a new point.
new_point = np.array([[2, 2]])
print("prediction for [2, 2]:", model.predict(new_point)[0])

# Step 4: Look under the hood - which neighbors voted, and how far away are they?
# kneighbors() returns (distances, row-indices-into-X_train) for the K nearest points.
distances, indices = model.kneighbors(new_point)
print()
print("the 3 nearest neighbors were:")
for dist, idx in zip(distances[0], indices[0]):      # [0] because we asked about 1 point
    print(f"  point {X_train[idx]} (label={y_train[idx]}) at distance {dist:.2f}")


prediction for [2, 2]: low

the 3 nearest neighbors were:
  point [1 1] (label=low) at distance 1.41
  point [1 0] (label=low) at distance 2.24
  point [0 0] (label=low) at distance 2.83


### Checklist item: Choosing K

**Approach:**
- Protects model fit: K controls the bias-variance trade-off directly, since a small K overfits to noise and a large K oversmooths and can wash out real local structure.
- Why this is the mathematically right approach: averaging over k neighbors mathematically reduces prediction variance (similar to how averaging k independent measurements reduces noise), while also averaging in more bias from points further away and less similar - K is the direct numeric dial on that bias-variance trade-off, not just a tuning knob without mathematical meaning.
- For this checklist item: Small K means low bias and high variance (noisy boundary); large K means high bias and low variance (smoother boundary); cross-validate to find the sweet spot.
- Code walkthrough: Check that train accuracy is highest around K=1 and generally falls as K grows, while test accuracy should peak at some middle K - that peak, not the training-accuracy trend, is the K you'd actually pick.

**Learn more:**
- Website: [scikit-learn: Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Nearest+Neighbors+Choosing+K+machine+learning+theory)

**Trade-offs:**
- K is essentially the only hyperparameter KNN has, simpler to tune than a tree ensemble's many knobs, but getting that one knob wrong swings the model between overfitting and oversmoothing more dramatically than a single hyperparameter usually would.
- Use cross-validation to pick K rather than a rule of thumb like the square root of n; an odd K also avoids tie votes in binary classification.

**Practical software engineering use cases:**
- When to use it: Use this checklist item whenever you stand up a new KNN model - always sweep K via cross-validation rather than defaulting to K=5.
- When not to use it: Don't pick K by eyeballing training accuracy alone - K=1 will look perfect on training data and mislead you about real generalization.


In [133]:
# K-Nearest Neighbors - Choosing K (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# K controls the bias-variance tradeoff:
#   K too small (1)  -> follows every noisy point -> overfits (jagged boundary)
#   K too large      -> averages over too many points -> underfits (too smooth)
# Standard approach: try several K values and compare accuracy on held-out data.

# Step 1: Generate a synthetic classification dataset (200 samples, 2 informative features).
X, y = make_classification(n_samples=200, n_features=2, n_informative=2,
                           n_redundant=0, random_state=42)

# Step 2: Split into train (70%) and test (30%). The test set simulates unseen data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Step 3: Loop over candidate K values, train, and score each on BOTH sets.
for k in [1, 3, 5, 11, 25, 51]:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)        # accuracy on data it has seen
    test_acc = model.score(X_test, y_test)           # accuracy on unseen data (what matters)
    print(f"K={k:<3} train accuracy={train_acc:.2f}  test accuracy={test_acc:.2f}")

# Reading the output: K=1 usually has perfect train accuracy but weaker test accuracy
# (overfitting). Pick the K with the best TEST accuracy. Odd K avoids tie votes.


K=1   train accuracy=1.00  test accuracy=0.87
K=3   train accuracy=0.93  test accuracy=0.87
K=5   train accuracy=0.91  test accuracy=0.83
K=11  train accuracy=0.87  test accuracy=0.80
K=25  train accuracy=0.87  test accuracy=0.82
K=51  train accuracy=0.86  test accuracy=0.82


### Checklist item: Scaling requirement

**Approach:**
- Protects representation and model fit: KNN's distance metric is dominated by whichever feature has the largest raw numeric range, so unscaled features silently bias which neighbors count.
- Why this is the mathematically right approach: because Euclidean distance sums the squared difference in every dimension, a feature whose raw values span 0-100,000 contributes a term thousands of times larger than a feature spanning 0-1, mathematically drowning out the smaller-scale feature regardless of its real predictive value - standardizing every feature to mean 0, std 1 first is what makes each dimension contribute comparably to the sum.
- For this checklist item: Features on different scales (e.g., age=25 vs income=50000) dominate distance calculations; always normalize or standardize before KNN.
- Code walkthrough: Check that the unscaled prediction is dominated by the salary feature (the larger raw numbers) while the scaled prediction changes once age and salary contribute more equally - that's the scaling bug and its fix side by side.

**Learn more:**
- Website: [scikit-learn: Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Nearest+Neighbors+Scaling+requirement+machine+learning+theory)

**Trade-offs:**
- KNN is more sensitive to feature scale than almost any model in this notebook, since raw Euclidean distance (not a learned weight) decides everything - a tree-based model would shrug off the exact same unscaled data.
- Standardize or normalize features before fitting; forgetting this is the single most common KNN bug and it fails silently with no error, just bad neighbors.

**Practical software engineering use cases:**
- When to use it: Use this as a mandatory pre-flight check any time you build or retrain a KNN model, especially after adding a new feature with a different unit.
- When not to use it: Don't skip scaling just this once because the model still runs without error - KNN fails silently here, producing plausible-looking but wrong neighbors.


In [134]:
# K-Nearest Neighbors - Scaling requirement (using scikit-learn)

import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# KNN is built on DISTANCE, so a feature with big numbers dominates the distance
# calculation and silently drowns out the other features. Always scale features for KNN.

# Features: [age (20-65), salary (30,000-120,000)] - wildly different scales!
X_train = np.array([
    [25, 40000], [30, 45000], [35, 50000],           # class 0: younger, lower salary
    [45, 90000], [50, 100000], [55, 110000],         # class 1: older, higher salary
])
y_train = np.array([0, 0, 0, 1, 1, 1])

# Test point: age like class 1 (52), salary like class 0 (48,000). Age says 1, salary says 0.
new_point = np.array([[52, 48000]])

# WITHOUT scaling: salary's huge numbers dominate the distance -> age is ignored.
model_raw = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
print("prediction WITHOUT scaling:", model_raw.predict(new_point)[0], "(salary dominated)")

# WITH scaling: StandardScaler transforms each column to mean 0, std 1 -> fair contribution.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)             # learn mean/std from training data, apply
new_scaled = scaler.transform(new_point)             # apply the SAME transform (never re-fit!)

model_scaled = KNeighborsClassifier(n_neighbors=3).fit(X_scaled, y_train)
print("prediction WITH scaling:   ", model_scaled.predict(new_scaled)[0], "(both features count)")


prediction WITHOUT scaling: 0 (salary dominated)
prediction WITH scaling:    1 (both features count)


### Checklist item: Classification and regression use cases

**Approach:**
- Protects model fit: the "K nearest neighbors vote or average" mechanism generalizes to both discrete labels (majority vote) and continuous targets (mean or median), so the aggregation must match the target type.
- Why this is the mathematically right approach: voting (the mode of k neighbors' labels) is the estimator that minimizes 0/1 classification error among the neighbors, while averaging (the mean of k neighbors' values) is the estimator that minimizes squared error among them - each aggregation rule is mathematically optimal for its respective loss function, not an arbitrary choice.
- For this checklist item: For classification: majority vote of K neighbors. For regression: mean of K neighbor values. Both are non-parametric and instance-based.
- Code walkthrough: Check that the regression prediction for x=3.5 is exactly the mean of its two nearest neighbors' y-values (140 and 160, averaging to 150), confirming regression truly averages while classification truly votes.

**Learn more:**
- Website: [scikit-learn: Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Nearest+Neighbors+Classification+and+regression+use+cases+machine+learning+theory)

**Trade-offs:**
- The same distance-and-average idea covers both classification and regression with almost no code changes, a symmetry most algorithms like logistic regression or SVM don't offer without swapping to a different formulation.
- KNN regression tends to be biased toward the mean near data boundaries, since neighbors at the edge of the feature space are only available on one side.

**Practical software engineering use cases:**
- When to use it: Use the regressor variant for continuous targets like price estimation from comparable listings, and the classifier variant for categorical labels like similar-item tagging.
- When not to use it: Don't use KNN regression near the edges of your feature range, such as extrapolating beyond observed sizes - it can only average existing neighbors, never extrapolate a trend.


In [135]:
# K-Nearest Neighbors - Classification and regression use cases (using scikit-learn)

import numpy as np
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# KNN comes in two flavors:
#   KNeighborsClassifier -> neighbors VOTE on a category
#   KNeighborsRegressor  -> neighbors' values are AVERAGED into a number

# ---- Flavor 1: CLASSIFICATION (predict a category) ----
X_c = np.array([[1], [2], [3], [10], [11], [12]])    # 1 feature
y_c = np.array(["cheap", "cheap", "cheap", "premium", "premium", "premium"])

clf = KNeighborsClassifier(n_neighbors=3).fit(X_c, y_c)
print("classification for x=2.5:", clf.predict([[2.5]])[0])    # 3 nearest are all "cheap"

# ---- Flavor 2: REGRESSION (predict a number) ----
X_r = np.array([[1], [2], [3], [4], [5], [6]])       # e.g., house size
y_r = np.array([100, 120, 140, 160, 180, 200])       # e.g., price

reg = KNeighborsRegressor(n_neighbors=2).fit(X_r, y_r)
# For x=3.5 the 2 nearest neighbors are x=3 (y=140) and x=4 (y=160); prediction = their mean.
print("regression for x=3.5:", reg.predict([[3.5]])[0], "(average of 140 and 160)")

# When to use KNN: small-to-medium datasets, meaningful distances, quick baselines.
# When to avoid: huge datasets (slow at predict time) or many features (see next cell).


classification for x=2.5: cheap
regression for x=3.5: 150.0 (average of 140 and 160)


### Checklist item: Curse of dimensionality

**Approach:**
- Protects representation: in high dimensions, distances between points concentrate and everything looks roughly equidistant, which breaks the core assumption that near points are meaningfully similar.
- Why this is the mathematically right approach: as dimensionality d grows, the ratio of the distance to the nearest point over the distance to the farthest point provably approaches 1 for many distributions, a consequence of how volume concentrates in high-dimensional space - this is a mathematical fact about geometry itself, not a quirk of any particular dataset, which is why no amount of extra data fixes KNN's high-dimensional breakdown on its own.
- For this checklist item: In high dimensions, all points become approximately equidistant â€” KNN loses discrimination power and requires exponentially more data to maintain density.
- Code walkthrough: Check that the printed nearest/farthest ratio creeps from a small number toward 1.0 as dims grows - a ratio near 1 means nearest and farthest become nearly indistinguishable, which is the curse in a single number.

**Learn more:**
- Website: [scikit-learn: Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Nearest+Neighbors+Curse+of+dimensionality+machine+learning+theory)

**Trade-offs:**
- KNN degrades gracefully with more data in low dimensions, but degrades ungracefully with more dimensions no matter how much data you add - a failure mode tree ensembles and linear models are considerably more resistant to.
- Above roughly a few dozen features, consider dimensionality reduction or feature selection before KNN rather than feeding it the raw high-dimensional space.

**Practical software engineering use cases:**
- When to use it: Use this check before shipping a KNN-based feature on any dataset with more than a few dozen columns.
- When not to use it: Don't feed KNN a high-dimensional embedding or one-hot-encoded feature set directly - reduce dimensionality first, or distances will stop being meaningful.


In [136]:
# K-Nearest Neighbors - Curse of dimensionality (numpy demonstration)

import numpy as np

# "Curse of dimensionality": as the number of features grows, ALL points become
# almost equally far apart - so "nearest" neighbor stops being meaningful for KNN.

rng = np.random.default_rng(0)
n_points = 500                                       # random points in a unit cube

for dims in [2, 10, 100, 1000]:                      # try increasingly many dimensions
    points = rng.random((n_points, dims))            # 500 random points, `dims` features each
    query = rng.random(dims)                         # one random query point

    # Euclidean distance from the query to every point:
    # subtract, square, sum across features (axis=1), square-root.
    dists = np.sqrt(((points - query) ** 2).sum(axis=1))

    # The key ratio: nearest distance / farthest distance.
    # Near 0 -> "nearest" is clearly special. Near 1 -> everything is equally far (bad for KNN).
    ratio = dists.min() / dists.max()
    print(f"dims={dims:<5} nearest={dists.min():6.2f}  farthest={dists.max():6.2f}  "
          f"nearest/farthest={ratio:.2f}")

# Output shows the ratio creeping toward 1 as dims grow -> distance loses meaning.
# Practical fixes: select fewer features, or reduce dimensions (e.g., PCA) before KNN.


dims=2     nearest=  0.03  farthest=  1.27  nearest/farthest=0.02
dims=10    nearest=  0.34  farthest=  1.91  nearest/farthest=0.18
dims=100   nearest=  3.39  farthest=  4.60  nearest/farthest=0.74
dims=1000  nearest= 12.32  farthest= 13.63  nearest/farthest=0.90


In [137]:
# Practice: K-Nearest Neighbors

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Naive Bayes

### Study checklist
- [ ] Bayes theorem intuition
- [ ] Conditional independence assumption
- [ ] Gaussian Naive Bayes
- [ ] Multinomial Naive Bayes for text
- [ ] When Naive Bayes works well

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Bayes theorem intuition

**Approach:**
- Protects model fit: Naive Bayes classification is literally an application of Bayes' theorem to invert P(features given class) into the P(class given features) you actually want to predict.
- Why this is the mathematically right approach: Bayes' theorem, P(class|features) = P(features|class) * P(class) / P(features), is a mathematically exact identity derived from the definition of conditional probability, not an approximation - Naive Bayes classification computes the numerator for each class, since the denominator is identical across classes and can be ignored, and picks whichever class makes that number largest.
- For this checklist item: Bayes' theorem: P(class|features) âˆ P(features|class) Ã— P(class) â€” combine a likelihood model with a prior to get posterior class probabilities.
- Code walkthrough: Check that np.exp(model.class_log_prior_) recovers the actual class split in the 4-document training set (2 spam, 2 ham, so both priors near 0.5), then see how those priors combine with word likelihoods to classify 'cheap meeting'.

**Learn more:**
- Website: [scikit-learn: Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Naive+Bayes+Bayes+theorem+intuition+machine+learning+theory)

**Trade-offs:**
- Naive Bayes needs only counting statistics to train, dramatically cheaper than iteratively-optimized models like logistic regression or SVM, at the cost of the strong independence assumption baked into that simplicity.
- It requires a prior P(class); a poorly estimated prior, for example from an unrepresentative training sample, biases every prediction even when the likelihoods are accurate.

**Practical software engineering use cases:**
- When to use it: Use this as your mental model when debugging why a Naive Bayes classifier favors one class - trace it back to the prior and the per-feature likelihoods it learned.
- When not to use it: Don't rely on it if your training data's class proportions don't reflect the real-world deployment distribution - the learned prior will be systematically wrong.


In [138]:
# Naive Bayes - Bayes theorem intuition (using scikit-learn)

import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

# Bayes theorem:  P(spam | words) is proportional to P(words | spam) * P(spam)
#   P(spam)         = "prior"      - how common spam is overall
#   P(words | spam) = "likelihood" - how likely these words are IN spam
# The classifier picks whichever class makes the observed words most probable.

documents = ["cheap offer now", "project meeting today",
             "cheap deal offer", "team meeting notes"]
labels = np.array([1, 0, 1, 0])                      # 1 = spam, 0 = ham

# CountVectorizer turns text into word counts: one column per unique word.
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)              # learns the vocabulary + counts words
print("vocabulary:", vectorizer.get_feature_names_out())

model = MultinomialNB()                              # Naive Bayes for count data
model.fit(X, labels)

# The PRIOR the model learned: log of P(each class). np.exp() undoes the log.
print("learned priors P(ham), P(spam):", np.round(np.exp(model.class_log_prior_), 2))

# Classify a new message. Note: transform() only (the vocabulary is already learned).
new_msg = ["cheap meeting"]
X_new = vectorizer.transform(new_msg)
print("P(ham), P(spam) for 'cheap meeting':", np.round(model.predict_proba(X_new)[0], 2))
print("prediction:", "spam" if model.predict(X_new)[0] == 1 else "ham")


vocabulary: ['cheap' 'deal' 'meeting' 'notes' 'now' 'offer' 'project' 'team' 'today']
learned priors P(ham), P(spam): [0.5 0.5]
P(ham), P(spam) for 'cheap meeting': [0.5 0.5]
prediction: ham


### Checklist item: Conditional independence assumption

**Approach:**
- Protects evaluation integrity: the "naive" assumption that features are independent given the class is almost never literally true, so understanding it tells you when the model's simplicity will hurt vs. help.
- Why this is the mathematically right approach: the independence assumption is what allows P(x1,x2,...,xn|class) to be rewritten as the simple product P(x1|class) * P(x2|class) * ... * P(xn|class) instead of requiring the full joint distribution, which would need exponentially more data to estimate - it's a deliberate simplification trading some accuracy for a model whose parameters can be estimated from very little data.
- For this checklist item: Naive Bayes assumes all features are conditionally independent given the class â€” rarely true in practice, but the classifier often works surprisingly well.
- Code walkthrough: Check that feature_log_prob_ gives one probability per word, independent of any other word in the same message - the multiplication mentioned at the end is literally the independence assumption being applied during scoring.

**Learn more:**
- Website: [scikit-learn: Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Naive+Bayes+Conditional+independence+assumption+machine+learning+theory)

**Trade-offs:**
- The independence assumption is what makes Naive Bayes' math trivially fast, since there's no joint distribution to estimate, unlike models such as logistic regression that implicitly account for feature interactions - the speed and the blind spot come from the same design choice.
- The assumption is violated often in practice yet the classifier still performs well, because it only needs the ranking of class probabilities to be right, not their absolute calibration, so don't trust the raw predicted probabilities.

**Practical software engineering use cases:**
- When to use it: Use Naive Bayes when your features are genuinely close to independent given the label, like distinct words in a document.
- When not to use it: Don't trust its raw predicted probabilities for anything downstream that needs calibration, like a risk score cutoff - the independence assumption skews them even when classification accuracy is fine.


In [139]:
# Naive Bayes - Conditional independence assumption (using scikit-learn)

import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

# The "naive" part: the model assumes every word appears INDEPENDENTLY given the class:
#   P("cheap offer" | spam)  is treated as  P("cheap" | spam) * P("offer" | spam)
# That is rarely literally true ("new" and "york" travel together!), yet the
# CLASSIFICATION still works well, because we only need the right class to win,
# not perfectly calibrated probabilities.

documents = ["cheap offer", "cheap deal", "free offer",
             "project meeting", "meeting notes", "project plan"]
labels = np.array([1, 1, 1, 0, 0, 0])                # 1 = spam, 0 = ham

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)              # text -> word-count matrix
model = MultinomialNB().fit(X, labels)

# feature_log_prob_[class][word] = log P(word | class). exp() converts back to probability.
words = vectorizer.get_feature_names_out()
per_word_prob_spam = np.exp(model.feature_log_prob_[1])   # row 1 = spam class

print("P(word | spam) learned independently for each word:")
for word, p in zip(words, per_word_prob_spam):
    print(f"  {word:<8} {p:.2f}")

print()
print("To score a document, Naive Bayes just MULTIPLIES these per-word probabilities -")
print("that multiplication IS the independence assumption in action.")


P(word | spam) learned independently for each word:
  cheap    0.21
  deal     0.14
  free     0.14
  meeting  0.07
  notes    0.07
  offer    0.21
  plan     0.07
  project  0.07

To score a document, Naive Bayes just MULTIPLIES these per-word probabilities -
that multiplication IS the independence assumption in action.


### Checklist item: Gaussian Naive Bayes

**Approach:**
- Protects representation: this variant assumes each continuous feature is normally distributed within each class, so it's the right choice specifically for continuous, roughly bell-shaped features.
- Why this is the mathematically right approach: assuming each continuous feature follows a normal distribution within each class means its likelihood P(xi|class) has a known closed-form formula, the Gaussian PDF, parameterized only by a mean and variance per class - that's what lets the model be fit by computing simple summary statistics rather than estimating an arbitrary distribution shape.
- For this checklist item: Gaussian NB models each feature's class-conditional distribution as Gaussian; use it for continuous features when the normality assumption is approximately met.
- Code walkthrough: Check that model.theta_ has one row per iris species and one column per measurement - those are the per-class Gaussian means the model assumes each feature is drawn from.

**Learn more:**
- Website: [scikit-learn: Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Naive+Bayes+Gaussian+Naive+Bayes+machine+learning+theory)

**Trade-offs:**
- GaussianNB has essentially no hyperparameters to tune, unlike SVM's C/gamma or a tree's depth, which makes it an unusually fast baseline to stand up - but that simplicity means it can't adapt to a feature distribution that isn't roughly bell-shaped.
- Skewed or multimodal continuous features violate the Gaussian assumption and hurt accuracy, so check feature histograms per class before defaulting to this variant.

**Practical software engineering use cases:**
- When to use it: Use this variant for continuous, roughly bell-shaped sensor or measurement features where you need a near-instant baseline.
- When not to use it: Don't use it on skewed or multimodal continuous features, like income or wait times, without transforming them first - the Gaussian assumption will misfit the tails.


In [140]:
# Naive Bayes - Gaussian Naive Bayes (using scikit-learn)

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# GaussianNB is Naive Bayes for CONTINUOUS features (measurements, sensor values...).
# For each class it learns a mean and variance per feature, and assumes a bell curve.

# Step 1: Load the classic iris dataset (150 flowers, 4 measurements, 3 species).
X, y = load_iris(return_X_y=True)                    # X = features, y = species labels 0/1/2

# Step 2: Hold out 30% for testing. stratify=y keeps class proportions equal in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Step 3: Train. GaussianNB has essentially no hyperparameters to tune - great baseline.
model = GaussianNB()
model.fit(X_train, y_train)

# Step 4: Predict on the unseen test flowers and measure accuracy.
y_pred = model.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, y_pred), 3))

# Step 5: Peek inside - the per-class, per-feature means the model learned (theta_).
print()
print("learned mean of each feature per class:")
print(model.theta_.round(2))                         # rows = classes, columns = features


test accuracy: 0.911

learned mean of each feature per class:
[[4.99 3.43 1.49 0.24]
 [5.95 2.73 4.24 1.31]
 [6.68 3.01 5.63 2.07]]


### Checklist item: Multinomial Naive Bayes for text

**Approach:**
- Protects representation: this variant models feature counts such as word frequencies, making it the natural fit for bag-of-words or term-frequency representations rather than raw continuous measurements.
- Why this is the mathematically right approach: word counts in a document are modeled as draws from a multinomial distribution, whose likelihood formula is a product of each word's probability raised to its count in the document - this specific mathematical form naturally rewards words appearing more often being more informative, matching how word frequency actually behaves in text.
- For this checklist item: Multinomial NB models feature counts (word frequencies); it is the standard baseline for text classification with BoW or TF-IDF features.
- Code walkthrough: Check that the pipeline classifies both brand-new messages correctly despite never seeing those exact sentences before, and that model.classes_ tells you which column of predict_proba corresponds to which label.

**Learn more:**
- Website: [scikit-learn: Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Naive+Bayes+Multinomial+Naive+Bayes+for+text+machine+learning+theory)

**Trade-offs:**
- For text, Multinomial NB trains in a fraction of the time a transformer-based classifier would need, but it treats a document as an unordered bag of counts, discarding structure a sequence model would use.
- It's fast and a strong baseline for text classification, but ignores word order and context entirely, so it plateaus below embedding-based or transformer approaches on tasks needing semantics.

**Practical software engineering use cases:**
- When to use it: Use this for a fast, cheap-to-retrain text classification baseline, such as spam filtering or ticket routing, before justifying an embedding-based model.
- When not to use it: Don't use it where word order or context changes meaning, such as sentiment with negation or sarcasm - bag-of-words counts can't capture that.


In [141]:
# Naive Bayes - Multinomial Naive Bayes for text (using scikit-learn)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# MultinomialNB is the classic choice for text, because word COUNTS fit its math.
# A Pipeline chains vectorizer + model so raw strings go in and predictions come out.

# Step 1: A tiny labeled dataset of messages.
train_texts = [
    "win a free prize now",       "claim your free reward",     # spam
    "limited offer click now",                                   # spam
    "lunch meeting at noon",      "quarterly report attached",  # ham
    "see you at the standup",                                    # ham
]
train_labels = ["spam", "spam", "spam", "ham", "ham", "ham"]

# Step 2: Build the pipeline.
#   CountVectorizer: text -> matrix of word counts
#   MultinomialNB:   word counts -> class probabilities
model = make_pipeline(CountVectorizer(), MultinomialNB())

# Step 3: Train. The pipeline fits the vectorizer AND the classifier in one call.
model.fit(train_texts, train_labels)

# Step 4: Classify brand-new messages - pass raw strings, the pipeline does the rest.
new_texts = ["free prize inside", "standup meeting moved"]
for text, label in zip(new_texts, model.predict(new_texts)):
    print(f"'{text}' -> {label}")

# predict_proba shows confidence; classes_ tells you the column order (alphabetical).
print()
print("classes:", model.classes_)
print("probabilities:", model.predict_proba(new_texts).round(2))


'free prize inside' -> spam
'standup meeting moved' -> ham

classes: ['ham' 'spam']
probabilities: [[0.14 0.86]
 [0.8  0.2 ]]


### Checklist item: When Naive Bayes works well

**Approach:**
- Protects deployment reliability: knowing its sweet spot, high-dimensional sparse features with roughly independent signal like text, prevents wasted effort applying it where the independence assumption fails badly, such as pixels in an image.
- Why this is the mathematically right approach: because Naive Bayes only needs to estimate one probability distribution per feature per class, never a joint distribution across features, its number of parameters grows linearly with feature count rather than exponentially - that specific mathematical property is what lets it work well with very little training data compared to models that must learn feature interactions.
- For this checklist item: Naive Bayes trains in O(n) and predicts in O(k*d); it excels on high-dimensional sparse text data and small training sets where other models overfit.
- Code walkthrough: Check whether Naive Bayes' accuracy trails Logistic Regression's on this correlated-feature dataset - that gap, if present, is the independence assumption being violated in practice.

**Learn more:**
- Website: [scikit-learn: Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Naive+Bayes+When+Naive+Bayes+works+well+machine+learning+theory)

**Trade-offs:**
- Compared to Logistic Regression, Naive Bayes needs far less training data to reach a reasonable accuracy, which is exactly why it's still a strong first baseline before reaching for anything heavier.
- It trains and predicts extremely fast with little data, making it a good baseline or sanity check, but it's rarely the final production model when correlated features carry most of the signal.

**Practical software engineering use cases:**
- When to use it: Use it as the very first model you try on a new text or high-dimensional sparse classification problem, specifically because it's cheap to disprove.
- When not to use it: Don't keep it as the final production model once you have enough data and correlated features that a discriminative model like logistic regression clearly outperforms it.


In [142]:
# Naive Bayes - When Naive Bayes works well (scikit-learn comparison)

from sklearn.datasets import fetch_20newsgroups, make_classification
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

# Naive Bayes shines when:
#   - features are (roughly) independent given the class  -> e.g., words in text
#   - training data is small                              -> it needs few examples
#   - you need a FAST baseline                            -> trains almost instantly
# It struggles when features are strongly CORRELATED - it double-counts their evidence.

# Demo: build data where two features are nearly copies of each other (highly correlated).
X, y = make_classification(n_samples=300, n_features=4, n_informative=2,
                           n_redundant=2,               # 2 features are combos of the others
                           random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

nb = GaussianNB().fit(X_train, y_train)              # assumes independence (violated here)
lr = LogisticRegression().fit(X_train, y_train)      # handles correlated features fine

print("dataset WITH correlated (redundant) features:")
print("  Naive Bayes test accuracy:        ", round(nb.score(X_test, y_test), 3))
print("  Logistic Regression test accuracy:", round(lr.score(X_test, y_test), 3))
print()
print("Rule of thumb: NB = fast, strong text baseline; check correlations elsewhere.")


dataset WITH correlated (redundant) features:
  Naive Bayes test accuracy:         0.922
  Logistic Regression test accuracy: 0.922

Rule of thumb: NB = fast, strong text baseline; check correlations elsewhere.


In [143]:
# Practice: Naive Bayes

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Decision Trees

### Study checklist
- [ ] Splitting criteria
- [ ] Gini and entropy
- [ ] Tree depth
- [ ] Overfitting
- [ ] Pruning
- [ ] Feature importance

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Splitting criteria

**Approach:**
- Protects model fit: the splitting criterion is the objective function the tree greedily optimizes at every node, so it directly shapes what the tree considers the best split.
- Why this is the mathematically right approach: a tree greedily picks, at each node, the feature and threshold producing the largest decrease in impurity, parent impurity minus the weighted average of the two children's impurity - this greedy, locally-optimal choice is what makes tree-building computationally tractable, since checking every possible sequence of splits for the globally best tree is combinatorially infeasible.
- For this checklist item: Evaluate candidate feature/threshold splits by how much they reduce impurity (such as Gini or entropy), then choose the split that makes child nodes more label-pure.
- Code walkthrough: Check that 'after split gini' is lower than 'parent gini' - that positive impurity reduction is exactly what a tree search maximizes when picking which feature to split on.

**Learn more:**
- Website: [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Decision+Trees+Splitting+criteria+machine+learning+theory)

**Trade-offs:**
- A single decision tree can be read and explained rule-by-rule in a way a random forest or gradient boosting ensemble cannot, but that same rule-by-rule structure is what makes it easy to overfit noise in the training data.
- Greedy, criterion-driven splitting is fast but can miss globally better trees that a non-greedy search would find; this is a known limitation, not a bug to fix by tuning the criterion alone.

**Practical software engineering use cases:**
- When to use it: Use the default criterion (gini) unless you have a specific reason to believe entropy's slightly different sensitivity to class purity matters for your data.
- When not to use it: Don't spend tuning time switching between gini and entropy expecting a meaningful accuracy change - that budget is better spent on depth or pruning.


In [144]:
# Decision Trees - Splitting criteria
rows = [
    {"age_group": "young", "label": 0},
    {"age_group": "young", "label": 0},
    {"age_group": "older", "label": 1},
    {"age_group": "older", "label": 1},
    {"age_group": "older", "label": 0},
]

def gini(labels):
    total = len(labels)
    return 1 - sum((labels.count(label) / total) ** 2 for label in set(labels))

parent = gini([row["label"] for row in rows])
left = [row["label"] for row in rows if row["age_group"] == "young"]
right = [row["label"] for row in rows if row["age_group"] == "older"]
weighted = (len(left) * gini(left) + len(right) * gini(right)) / len(rows)
print("parent gini:", round(parent, 3))
print("after split gini:", round(weighted, 3))
print("impurity reduction:", round(parent - weighted, 3))


parent gini: 0.48
after split gini: 0.267
impurity reduction: 0.213


### Checklist item: Gini and entropy

**Approach:**
- Protects model fit and training efficiency: the splitting criterion decides how the tree partitions data at each node, which affects tree shape and, at scale, training time.
- Why this is the mathematically right approach: Gini impurity, 1 - sum(pi^2), and entropy, -sum(pi*log2(pi)), are both mathematically valid measures equal to exactly 0 when a node is pure and maximized when classes are perfectly mixed - they differ only in how sharply they penalize impurity in between, which is why they usually produce very similar trees despite being different formulas.
- For this checklist item: Gini impurity and entropy measure class mixture at a node; the algorithm picks the feature and split point that most reduces this impurity.
- Code walkthrough: Check that gini and entropy both report 0 impurity for the pure group but different non-zero numbers for the 80/20 mix, and that the two criteria's training accuracy on iris end up nearly identical.

**Learn more:**
- Website: [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Decision+Trees+Gini+and+entropy+machine+learning+theory)

**Trade-offs:**
- Both criteria only ever look at one feature at a time when choosing a split, which keeps training fast compared to a model that considers feature interactions directly like an SVM's kernel, but also means a single split can't express an interaction.
- Gini is cheaper to compute since it avoids a logarithm and is scikit-learn's default; entropy is marginally more sensitive to class purity, but in practice the two rarely change final accuracy much, so don't spend tuning budget here versus depth or pruning.

**Practical software engineering use cases:**
- When to use it: Use either criterion when you need a transparent, rule-based model a non-technical reviewer can trace by hand.
- When not to use it: Don't expect either criterion alone to fix an already-overfit or already-underfit tree - the criterion affects splits, not overall model capacity.


In [145]:
# Decision Trees - Gini and entropy (using scikit-learn)

import numpy as np
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

# Both measure IMPURITY of a group of samples (0 = perfectly pure, all one class):
#   gini    = 1 - sum(p_i^2)            (default; slightly faster, no logarithm)
#   entropy = -sum(p_i * log2(p_i))     (information-theory flavor)
# In practice they usually produce very similar trees.

# First, compute both by hand for intuition on a group with 80% / 20% class mix:
p = np.array([0.8, 0.2])                             # class proportions in the group
gini = 1 - np.sum(p ** 2)                            # gini formula
entropy = -np.sum(p * np.log2(p))                    # entropy formula
print(f"group with 80/20 mix -> gini={gini:.3f}, entropy={entropy:.3f}")

p_pure = np.array([1.0])                             # a pure group: only one class
print(f"pure group           -> gini={1 - np.sum(p_pure**2):.3f}, entropy=0.000")
print()

# Now train the same tree with each criterion and compare accuracy on iris.
X, y = load_iris(return_X_y=True)
for criterion in ["gini", "entropy"]:
    model = DecisionTreeClassifier(criterion=criterion, max_depth=3,
                                   random_state=42).fit(X, y)
    print(f"criterion='{criterion}': training accuracy = {model.score(X, y):.3f}")


group with 80/20 mix -> gini=0.320, entropy=0.722
pure group           -> gini=0.000, entropy=0.000

criterion='gini': training accuracy = 0.973
criterion='entropy': training accuracy = 0.973


### Checklist item: Tree depth

**Approach:**
- Protects model fit: depth is the single biggest lever on the bias-variance trade-off for a tree, since shallow trees underfit and deep trees memorize training data.
- Why this is the mathematically right approach: each additional level of depth roughly doubles the number of leaf regions a tree can carve the feature space into, so depth directly, exponentially controls model capacity - that's precisely why it's the single most influential hyperparameter for a tree's position on the bias-variance trade-off.
- For this checklist item: Shallow trees underfit (high bias); deep trees overfit (high variance); control via max_depth, min_samples_leaf, or post-pruning.
- Code walkthrough: Check that train accuracy keeps climbing toward 1.0 as depth increases while test accuracy rises then falls - the depth where test accuracy peaks is the one you'd actually choose.

**Learn more:**
- Website: [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Decision+Trees+Tree+depth+machine+learning+theory)

**Trade-offs:**
- Depth is a single, intuitive knob compared to the dozen-plus hyperparameters of a gradient boosting library, which makes trees easy to reason about, but that same single knob has an outsized effect on whether the tree underfits or memorizes noise.
- Unlimited depth on real data almost always overfits, so set max_depth or min_samples_leaf via cross-validation rather than letting the tree grow until pure.

**Practical software engineering use cases:**
- When to use it: Use this as the first hyperparameter to tune whenever a decision tree's train/test accuracy gap looks large.
- When not to use it: Don't leave max_depth unset (unlimited) in a production model - an unconstrained tree will grow until it memorizes training noise.


In [146]:
# Decision Trees - Tree depth (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# max_depth = how many questions deep the tree may go. It is the single most
# important knob for controlling tree complexity:
#   shallow tree -> few rules  -> may underfit (too simple)
#   deep tree    -> many rules -> may overfit (memorizes noise)

# Step 1: Make a somewhat noisy dataset so the effect is visible.
X, y = make_classification(n_samples=400, n_features=10, n_informative=4,
                           flip_y=0.1,               # flip 10% of labels = label noise
                           random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Step 2: Train trees of increasing depth; compare train vs test accuracy.
print("depth  train_acc  test_acc")
for depth in [1, 2, 4, 8, None]:                     # None = grow until leaves are pure
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    print(f"{str(depth):<6} {model.score(X_train, y_train):.3f}      "
          f"{model.score(X_test, y_test):.3f}")

# Reading the output: train accuracy keeps rising with depth (eventually 1.0 = memorized),
# but test accuracy peaks and then falls. Pick the depth where TEST accuracy peaks.


depth  train_acc  test_acc
1      0.646      0.642
2      0.736      0.733
4      0.796      0.733
8      0.968      0.817
None   1.000      0.775


### Checklist item: Overfitting

**Approach:**
- Protects evaluation integrity: a single unconstrained tree can reach 100% training accuracy by memorizing noise, so the train/test gap is the key signal to watch, not training accuracy alone.
- Why this is the mathematically right approach: an unconstrained tree can keep splitting until every leaf contains a single training point, mathematically achieving zero training error - but a leaf fit to one point has learned that point's specific noise, not the underlying pattern, which is why perfect training accuracy is a warning sign here rather than a goal.
- For this checklist item: A tree with unlimited depth memorizes training labels; overfitting shows as a large gap between train accuracy and val accuracy.
- Code walkthrough: Check that the unrestricted tree's train-minus-test accuracy gap is much larger than the constrained tree's gap - that gap size is the direct, numeric definition of overfitting used throughout this notebook.

**Learn more:**
- Website: [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Decision+Trees+Overfitting+machine+learning+theory)

**Trade-offs:**
- An unconstrained tree can represent almost any function given enough depth, more flexible than a linear model out of the box, but that flexibility is exactly what lets it fit noise instead of signal without constraints.
- Constraining depth or leaf size reduces overfitting but also reduces the model's ability to capture genuine complex interactions; this is exactly the trade-off ensembles like Random Forest are designed to sidestep.

**Practical software engineering use cases:**
- When to use it: Use this check, comparing train vs. test accuracy, as a standard gate before promoting any tree-based model out of experimentation.
- When not to use it: Don't judge a decision tree's quality from training accuracy alone - a huge train/test gap is the tell that the great training score doesn't mean anything.


In [147]:
# Decision Trees - Overfitting (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Overfitting = the model learns the noise in the training data, not the real pattern.
# Symptom: train accuracy is high, test accuracy is clearly lower (a big gap).
# Unrestricted decision trees are famous overfitters - they can memorize anything.

X, y = make_classification(n_samples=300, n_features=20, n_informative=5,
                           flip_y=0.15,              # 15% noisy labels to tempt the tree
                           random_state=7)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)

# Model 1: no limits at all - the tree grows until every training sample is classified.
overfit_tree = DecisionTreeClassifier(random_state=7).fit(X_train, y_train)

# Model 2: constrained tree. min_samples_leaf forces each leaf to hold >= 10 samples,
# so the tree cannot carve out tiny leaves just to fit individual noisy points.
tamed_tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10,
                                    random_state=7).fit(X_train, y_train)

for name, m in [("unrestricted", overfit_tree), ("constrained  ", tamed_tree)]:
    gap = m.score(X_train, y_train) - m.score(X_test, y_test)   # gap = overfitting size
    print(f"{name}: train={m.score(X_train, y_train):.3f}  "
          f"test={m.score(X_test, y_test):.3f}  gap={gap:.3f}")

# The unrestricted tree hits ~1.0 on train but drops on test (big gap = overfit).
# Main taming knobs: max_depth, min_samples_leaf, min_samples_split, ccp_alpha.


unrestricted: train=1.000  test=0.700  gap=0.300
constrained  : train=0.871  test=0.711  gap=0.160


### Checklist item: Pruning

**Approach:**
- Protects model fit and deployment reliability: pruning removes branches that only fit noise, producing a smaller, faster, more generalizable tree without hand-tuning depth by trial and error.
- Why this is the mathematically right approach: cost-complexity pruning explicitly minimizes error(tree) + alpha * (number of leaves), a formula that mathematically trades fit against tree size using one tunable alpha - increasing alpha is guaranteed to only ever remove branches, never add them, which is what makes the resulting sequence of prunings well-defined and searchable.
- For this checklist item: Pruning removes branches that don't improve val performance; pre-pruning (max_depth) is faster; post-pruning (cost-complexity) is more principled.
- Code walkthrough: Check that n_leaves shrinks as ccp_alpha grows, and watch for a point where test accuracy is actually higher than at ccp_alpha=0 - pruning improving test accuracy is the payoff, not just a smaller tree.

**Learn more:**
- Website: [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Decision+Trees+Pruning+machine+learning+theory)

**Trade-offs:**
- Pruning gives you a smaller, faster tree than depth-limiting alone typically produces, at the cost of an extra parameter (ccp_alpha) to search compared to just capping max_depth.
- Cost-complexity pruning requires its own validation-set tuning; skipping it and just capping depth is simpler but usually gives a slightly worse bias-variance trade-off.

**Practical software engineering use cases:**
- When to use it: Use cost-complexity pruning when you want a compact, deployable tree without hand-tuning max_depth by trial and error.
- When not to use it: Don't skip the pruning-vs-depth comparison on a tree that will run on constrained hardware, like an edge device or low-latency service - a pruned tree can be both smaller and more accurate.


In [148]:
# Decision Trees - Pruning (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Pruning = growing a full tree, then CUTTING BACK branches that don't earn their keep.
# scikit-learn implements "cost-complexity pruning" via the ccp_alpha parameter:
#   ccp_alpha = 0     -> no pruning (full tree)
#   larger ccp_alpha  -> branches must justify their complexity or be removed

X, y = make_classification(n_samples=300, n_features=15, n_informative=5,
                           flip_y=0.1, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=3)

# Try increasing amounts of pruning and watch tree size + accuracy change.
print("ccp_alpha  n_leaves  train_acc  test_acc")
for alpha in [0.0, 0.005, 0.02, 0.05]:
    model = DecisionTreeClassifier(ccp_alpha=alpha,  # pruning strength
                                   random_state=3).fit(X_train, y_train)
    print(f"{alpha:<10} {model.get_n_leaves():<9} "
          f"{model.score(X_train, y_train):.3f}      {model.score(X_test, y_test):.3f}")

# Typical pattern: as alpha grows, the tree shrinks (fewer leaves), train accuracy
# falls a little, and test accuracy often IMPROVES before eventually degrading.
# Bonus: model.cost_complexity_pruning_path(X_train, y_train) lists the exact
# alpha values where the tree structure changes, so you can search only those.


ccp_alpha  n_leaves  train_acc  test_acc
0.0        36        1.000      0.711
0.005      28        0.981      0.700
0.02       6         0.833      0.700
0.05       3         0.743      0.733


### Checklist item: Feature importance

**Approach:**
- Protects user value and model debugging: impurity-based importance tells you which features the tree actually relies on, useful for explaining predictions and catching leaked or spurious features.
- Why this is the mathematically right approach: a feature's importance is computed by summing the impurity decrease across every split that used it, weighted by how many training samples reached that split - a direct mathematical readout of how much total impurity reduction the tree credits to that feature, not a separately estimated approximation.
- For this checklist item: Feature importance from trees counts how much each feature reduces impurity across all splits â€” use it to identify top predictors and prune irrelevant features.
- Code walkthrough: Check that the printed importances sum to 1.0 (feature_importances_ is normalized) and that petal measurements outrank sepal measurements - that ranking is what the classic iris dataset is known for.

**Learn more:**
- Website: [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Decision+Trees+Feature+importance+machine+learning+theory)

**Trade-offs:**
- A single tree's feature importances are essentially free to compute since they fall out of training, unlike permutation importance which needs extra re-scoring passes, but that speed comes with the impurity-based bias called out in the walkthrough.
- Impurity-based importance is biased toward high-cardinality and continuous features since they get more possible split points; cross-check with permutation importance before trusting it for high-stakes decisions.

**Practical software engineering use cases:**
- When to use it: Use it for a first-pass explanation of what a tree relies on when a stakeholder asks why the model decided this.
- When not to use it: Don't use raw impurity-based importance to justify removing a feature from a regulated model - cross-check with permutation importance first, since impurity-based importance is biased toward high-cardinality features.


In [149]:
# Decision Trees - Feature importance (using scikit-learn)

import numpy as np
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

# After training, a tree can report how much each feature contributed to its splits.
# feature_importances_ = the total impurity reduction credited to each feature,
# normalized so all importances sum to 1.0.

iris = load_iris()
X, y = iris.data, iris.target

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X, y)                                      # train the tree

importances = model.feature_importances_             # one number per feature, sums to 1

# np.argsort gives the indices that would sort ascending; [::-1] reverses to descending.
order = np.argsort(importances)[::-1]

print("feature importance (most useful first):")
for idx in order:
    bar = "#" * int(importances[idx] * 40)           # crude text bar chart
    print(f"  {iris.feature_names[idx]:<20} {importances[idx]:.3f} {bar}")

# Caveats worth remembering:
#  - importance says "useful to THIS tree", not "causes the outcome"
#  - correlated features SPLIT the credit between them (each looks less important)
#  - features with many unique values can get inflated importance


feature importance (most useful first):
  petal length (cm)    0.586 #######################
  petal width (cm)     0.414 ################
  sepal width (cm)     0.000 
  sepal length (cm)    0.000 


In [150]:
# Practice: Decision Trees

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Random Forest

### Study checklist
- [ ] Bagging
- [ ] Bootstrap samples
- [ ] Feature randomness
- [ ] Out-of-bag score
- [ ] Feature importance
- [ ] Strengths and limitations

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Bagging

**Approach:**
- Protects model fit: bagging reduces variance by averaging many high-variance trees trained on different resamples, which is precisely what tames a single decision tree's overfitting tendency.
- Why this is the mathematically right approach: for independent, identically distributed estimators, the variance of their average is mathematically the individual variance divided by the number of estimators - averaging many bootstrap-trained trees directly applies this statistical fact to reduce a single tree's high variance, at essentially no cost to bias.
- For this checklist item: Bagging trains B trees on bootstrap samples and averages their predictions; the variance reduction comes from averaging uncorrelated trees.
- Code walkthrough: Check that the forest's test accuracy is meaningfully higher than the single tree's - since both use the same base learner, that gap is caused purely by averaging many bootstrap-sampled trees together.

**Learn more:**
- Website: [scikit-learn: Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests-and-other-randomized-tree-ensembles)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Random+Forest+Bagging+machine+learning+theory)

**Trade-offs:**
- Averaging many trees trades the interpretability of a single decision tree for meaningfully better generalization, a trade most practitioners are happy to make once accuracy matters more than a readable rule list.
- Averaging reduces variance but not bias, so bagging a systematically biased base learner won't fix that bias; it only helps when the base learner is high-variance and low-bias, like an unpruned tree.

**Practical software engineering use cases:**
- When to use it: Use Random Forest as your default good-enough-with-little-tuning tabular model when you need solid accuracy without an extensive tuning cycle.
- When not to use it: Don't use it when you specifically need a single readable rule set for a compliance or audit requirement - a forest of hundreds of trees can't be read like one tree.


In [151]:
# Random Forest - Bagging (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Bagging = Bootstrap AGGregatING:
#   1. draw many random samples (with replacement) from the training data
#   2. train one decision tree on each sample
#   3. average / majority-vote their predictions
# Individual trees overfit in DIFFERENT ways; averaging cancels their errors out.

X, y = make_classification(n_samples=500, n_features=20, n_informative=6,
                           flip_y=0.1, random_state=5)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=5)

# Baseline: one deep, unrestricted tree (a classic overfitter).
single_tree = DecisionTreeClassifier(random_state=5).fit(X_train, y_train)

# Random forest: 200 such trees, each on a bootstrap sample, votes combined.
forest = RandomForestClassifier(n_estimators=200,    # number of trees in the forest
                                random_state=5).fit(X_train, y_train)

print("single tree   - test accuracy:", round(single_tree.score(X_test, y_test), 3))
print("random forest - test accuracy:", round(forest.score(X_test, y_test), 3))
print()
print("Same base learner, but averaging many decorrelated trees reduces variance,")
print("so the forest generalizes better than any single tree.")


single tree   - test accuracy: 0.767
random forest - test accuracy: 0.86

Same base learner, but averaging many decorrelated trees reduces variance,
so the forest generalizes better than any single tree.


### Checklist item: Bootstrap samples

**Approach:**
- Protects representation: each tree sees a different resampled (with replacement) subset of the training data, which is what decorrelates the trees enough for averaging to reduce variance.
- Why this is the mathematically right approach: sampling n rows with replacement from an n-row dataset leaves any given row out of the sample with probability (1-1/n)^n, which converges mathematically to 1/e, about 36.8%, as n grows - that's exactly why roughly a third of rows are out-of-bag for any given tree, a precise consequence of the sampling procedure, not a rule of thumb.
- For this checklist item: A bootstrap sample draws n rows with replacement; ~37% of rows are out-of-bag (OOB) and can be used as a free validation set.
- Code walkthrough: Check that the printed bootstrap sample contains duplicate indices and that the 'left out' (out-of-bag) set is non-empty - roughly a third of the original 10 rows should be missing from the sample.

**Learn more:**
- Website: [scikit-learn: Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests-and-other-randomized-tree-ensembles)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Random+Forest+Bootstrap+samples+machine+learning+theory)

**Trade-offs:**
- Bootstrap sampling is what lets each tree in the forest differ from the others without collecting more data, unlike boosting where trees are deliberately made to depend on each other's errors instead of independent resampling.
- Bootstrapping leaves out roughly 37% of rows per tree, which is useful for free validation but means each individual tree is trained on slightly less unique data than the full set.

**Practical software engineering use cases:**
- When to use it: Use the default bootstrap=True setting for standard Random Forest use - it's what makes the ensemble diverse enough to benefit from averaging.
- When not to use it: Don't turn off bootstrapping unless you have a specific reason - without it, every tree sees the same data and the variance-reduction benefit of the ensemble mostly disappears.


In [152]:
# Random Forest - Bootstrap samples (numpy + scikit-learn)

import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

# A bootstrap sample = draw N rows from an N-row dataset WITH REPLACEMENT.
# Some rows appear multiple times; on average ~37% of rows are left out entirely.
# Each tree in the forest trains on its OWN bootstrap sample - that's where
# the trees' diversity comes from.

rng = np.random.default_rng(0)
n = 10
row_ids = np.arange(n)                               # pretend rows 0..9 are our dataset

# Draw one bootstrap sample: pick 10 indices from 0..9, duplicates allowed.
sample = rng.choice(row_ids, size=n, replace=True)
print("bootstrap sample of row ids:", sorted(sample))
print("rows left out ('out-of-bag'):", sorted(set(row_ids) - set(sample)))
print()

# In scikit-learn this happens automatically because bootstrap=True is the default:
X, y = make_classification(n_samples=200, random_state=0)
forest = RandomForestClassifier(n_estimators=50,
                                bootstrap=True,      # each tree gets a bootstrap sample
                                random_state=0).fit(X, y)
print(f"trained {len(forest.estimators_)} trees, each on its own bootstrap sample")
print("(the left-out rows per tree power the OOB score - next cells)")


bootstrap sample of row ids: [0, 0, 0, 1, 2, 3, 5, 6, 8, 8]
rows left out ('out-of-bag'): [4, 7, 9]

trained 50 trees, each on its own bootstrap sample
(the left-out rows per tree power the OOB score - next cells)


### Checklist item: Feature randomness

**Approach:**
- Protects representation: restricting each split to a random feature subset decorrelates trees further than bagging alone, preventing all trees from relying on the same one or two dominant features.
- Why this is the mathematically right approach: if every tree could choose from all features at every split, trees would repeatedly pick the same strongest feature first, making them highly correlated - and the variance-reduction benefit of averaging N estimators only approaches 1/N when they're independent; restricting each split's feature choice is what mathematically decorrelates the trees enough for averaging to work as intended.
- For this checklist item: At each split, only a random subset of features (sqrt(p) for classification) is considered; this decorrelates the trees and further reduces variance.
- Code walkthrough: Check whether the 'sqrt subset' or '20% subset' setting matches or beats using all features - restricting features per split still performs competitively while adding tree diversity.

**Learn more:**
- Website: [scikit-learn: Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests-and-other-randomized-tree-ensembles)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Random+Forest+Feature+randomness+machine+learning+theory)

**Trade-offs:**
- Restricting features per split is a lever unique to Random Forest among the tree-based models here - single decision trees and gradient boosting don't randomize feature choice this way - and it's what decorrelates the ensemble beyond what bagging alone achieves.
- A smaller feature subset per split increases decorrelation and can reduce variance, but each individual tree becomes weaker; there's a sweet spot to tune, not a rule that more randomness is always better.

**Practical software engineering use cases:**
- When to use it: Use the default max_features setting first, and only tune it down further if you suspect one or two dominant features are making your trees too similar.
- When not to use it: Don't set max_features to use all features unless you specifically want to approximate plain bagging - you'll lose most of the decorrelation benefit that makes Random Forest work well.


In [153]:
# Random Forest - Feature randomness (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Random forests add a SECOND layer of randomness beyond bootstrap sampling:
# at every split, each tree may only consider a RANDOM SUBSET of features.
# The knob is max_features:
#   'sqrt' (default for classification) -> consider sqrt(n_features) features per split
#   1.0                                 -> consider ALL features (plain bagging)
# Why it helps: if one feature is very strong, all trees would otherwise split on it
# first and end up nearly identical - averaging identical trees gains nothing.

X, y = make_classification(n_samples=500, n_features=25, n_informative=5,
                           flip_y=0.1, random_state=11)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=11)

for max_features, label in [(1.0, "all features (bagging)"),
                            ("sqrt", "sqrt subset (forest default)"),
                            (0.2, "20% subset")]:
    model = RandomForestClassifier(n_estimators=200,
                                   max_features=max_features,   # features tried per split
                                   random_state=11).fit(X_train, y_train)
    print(f"max_features={str(max_features):<6} ({label}): "
          f"test accuracy = {model.score(X_test, y_test):.3f}")

# The subset versions usually match or beat 'all features' because the trees
# become more DIVERSE, and diverse trees average into a stronger ensemble.


max_features=1.0    (all features (bagging)): test accuracy = 0.827
max_features=sqrt   (sqrt subset (forest default)): test accuracy = 0.793
max_features=0.2    (20% subset): test accuracy = 0.793


### Checklist item: Out-of-bag score

**Approach:**
- Protects evaluation integrity: the out-of-bag score gives a near-free validation estimate using the roughly 37% of data each tree didn't see, without sacrificing a held-out split.
- Why this is the mathematically right approach: because each row is out-of-bag for roughly a third of the trees, by the same 1/e sampling argument, averaging just those trees' predictions for each row produces a prediction never influenced by that row during training - mathematically equivalent in spirit to a held-out validation prediction, without needing a separate split.
- For this checklist item: OOB score uses each tree's OOB samples as its held-out set â€” it approximates cross-validation at no extra cost and is reliable for large forests.
- Code walkthrough: Check that model.oob_score_ is close to the actual held-out test accuracy printed right below it - that closeness is what makes OOB a trustworthy free validation estimate.

**Learn more:**
- Website: [scikit-learn: Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests-and-other-randomized-tree-ensembles)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Random+Forest+Out-of-bag+score+machine+learning+theory)

**Trade-offs:**
- OOB scoring gives Random Forest a built-in validation estimate that a single decision tree or a linear model has no equivalent for, since those don't have an ensemble of held-out rows to draw on.
- It's a convenient proxy but isn't identical to a proper held-out test set, for example it can't catch time-based leakage, so still do a real train/test split for final reporting on production-bound models.

**Practical software engineering use cases:**
- When to use it: Use oob_score=True when your dataset is small enough that setting aside a separate validation split feels wasteful.
- When not to use it: Don't rely on OOB score alone for a final, reportable metric on a time-dependent dataset - it doesn't respect chronological order the way a proper time-based split would.


In [154]:
# Random Forest - Out-of-bag score (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Remember: each tree's bootstrap sample leaves out ~37% of rows ("out-of-bag" = OOB).
# Clever trick: evaluate each row using ONLY the trees that never saw it.
# Result: a free validation score WITHOUT setting aside a separate validation set.

X, y = make_classification(n_samples=600, n_features=20, n_informative=6,
                           flip_y=0.1, random_state=21)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

model = RandomForestClassifier(n_estimators=300,
                               oob_score=True,       # turn on OOB evaluation
                               bootstrap=True,       # OOB requires bootstrap sampling
                               random_state=21)
model.fit(X_train, y_train)                          # OOB score is computed during fit

# Compare: the OOB estimate vs a real held-out test set. They should be close.
print("OOB score (free estimate):   ", round(model.oob_score_, 3))
print("actual test set accuracy:    ", round(model.score(X_test, y_test), 3))
print()
print("OOB is handy when data is too scarce to give up rows for a validation set.")


OOB score (free estimate):    0.8
actual test set accuracy:     0.833

OOB is handy when data is too scarce to give up rows for a validation set.


### Checklist item: Feature importance

**Approach:**
- Protects user value and debugging: averaging impurity-based importance across many trees gives a more stable importance ranking than a single tree's, useful for feature selection and explaining the model.
- Why this is the mathematically right approach: averaging each tree's impurity-decrease-based importance for a feature across all trees in the forest directly applies the same variance-reduction principle as bagging predictions - a stable estimate obtained by averaging many noisy per-tree estimates.
- For this checklist item: Random Forest feature importance averages impurity-based importance across all trees; it can overstate high-cardinality features â€” use permutation importance as a sanity check.
- Code walkthrough: Check that both the impurity-based importances and the permutation importances agree that features 0-2 dominate over features 3-7 - two different methods agreeing is a good sanity check that the ranking is real.

**Learn more:**
- Website: [scikit-learn: Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests-and-other-randomized-tree-ensembles)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Random+Forest+Feature+importance+machine+learning+theory)

**Trade-offs:**
- Averaging importances across hundreds of trees gives a more stable ranking than a single tree's importances would, but it's still the same impurity-based metric underneath, inheriting the same high-cardinality bias.
- It still inherits the single-tree bias toward high-cardinality or continuous features, and importance is diluted among correlated features since each gets partial credit; use permutation importance for a less biased estimate.

**Practical software engineering use cases:**
- When to use it: Use Random Forest's averaged feature importance as a quick, relatively stable way to rank features across a whole team's dashboard.
- When not to use it: Don't use it to make a final call between two correlated features - the ensemble splits credit between them, making each look less important than it really is.


In [155]:
# Random Forest - Feature importance (using scikit-learn)

import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

# Forest importance = average impurity-based importance across all trees (more stable
# than a single tree's). A stronger alternative is PERMUTATION importance:
# shuffle one feature's values and measure how much accuracy drops - a real
# "how much does the model rely on this?" test.

# Build data where we KNOW the answer: features 0-2 informative, 3-7 pure noise.
X, y = make_classification(n_samples=500, n_features=8, n_informative=3,
                           n_redundant=0, shuffle=False,  # keep informative ones first
                           random_state=9)

model = RandomForestClassifier(n_estimators=200, random_state=9).fit(X, y)

# Method 1: built-in impurity-based importances (fast, computed during training).
print("impurity-based importances:")
print(" ", np.round(model.feature_importances_, 3), " <- features 0-2 should dominate")

# Method 2: permutation importance (slower, but measures actual predictive reliance).
perm = permutation_importance(model, X, y,
                              n_repeats=10,          # shuffle each feature 10 times
                              random_state=9)
print("permutation importances (mean accuracy drop when shuffled):")
print(" ", np.round(perm.importances_mean, 3))


impurity-based importances:
  [0.28  0.134 0.337 0.052 0.047 0.056 0.047 0.047]  <- features 0-2 should dominate
permutation importances (mean accuracy drop when shuffled):
  [0.265 0.054 0.32  0.006 0.001 0.008 0.004 0.005]


### Checklist item: Strengths and limitations

**Approach:**
- Protects deployment reliability: knowing where Random Forest struggles, such as extrapolation beyond the training range, very high-dimensional sparse data, or large model size, prevents deploying it where a different model would clearly do better.
- Why this is the mathematically right approach: because a tree's prediction at any point is always the average of training y-values inside the leaf that point falls into, no tree-based average can output a value outside the range seen during training - a direct mathematical consequence of how leaf predictions are computed, not a fixable implementation detail, which is exactly why tree ensembles can't extrapolate.
- For this checklist item: Strengths: robust to outliers, handles mixed types, built-in feature importance, parallelizable. Limitations: slower inference than a single tree, less interpretable, memory-heavy.
- Code walkthrough: Check that the forest's predictions at x=15 and x=20 stay flat near the training range's maximum instead of continuing the true y=5x trend - that flatlining is the concrete evidence of the no-extrapolation limitation.

**Learn more:**
- Website: [scikit-learn: Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests-and-other-randomized-tree-ensembles)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Random+Forest+Strengths+and+limitations+machine+learning+theory)

**Trade-offs:**
- Random Forest typically needs far less hyperparameter tuning than gradient boosting or SVM to get a solid result, which is its main appeal, but that convenience doesn't extend to problems where predictions must extrapolate beyond the training data's range.
- It's robust and low-maintenance with little tuning, but the ensemble of many trees is slower to predict and harder to interpret than a single tree or linear model, and it can't extrapolate trends outside the training data's range.

**Practical software engineering use cases:**
- When to use it: Use Random Forest for tabular problems with noisy or messy features where you want a robust result without much preprocessing.
- When not to use it: Don't use it for a forecasting or pricing model that must extrapolate beyond the range of historical data - its predictions flatten out instead of continuing a trend.


In [156]:
# Random Forest - Strengths and limitations (using scikit-learn)

import time
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# STRENGTHS: strong accuracy with almost no tuning, handles non-linear patterns,
#            robust to outliers and feature scales, gives feature importances.
# LIMITATIONS: slower & bigger than a single tree, less interpretable,
#              and (shown below) CANNOT EXTRAPOLATE beyond the training range -
#              tree predictions are averages of training targets, nothing more.

# Train on x in [0, 10] where truth is y = 5x, then predict at x = 15 and 20.
rng = np.random.default_rng(2)
X_train = rng.uniform(0, 10, 200).reshape(-1, 1)     # training inputs stay within [0, 10]
y_train = 5 * X_train.ravel() + rng.normal(0, 2, 200)

t0 = time.time()                                     # time the training for the record
model = RandomForestRegressor(n_estimators=200, random_state=2).fit(X_train, y_train)
print(f"training time: {time.time() - t0:.2f}s for 200 trees")

X_new = np.array([[5], [10], [15], [20]])            # 5,10 seen range; 15,20 OUTSIDE it
print()
print(" x   true y   forest prediction")
for x_val, pred in zip(X_new.ravel(), model.predict(X_new)):
    print(f"{x_val:>3}   {5 * x_val:>5}    {pred:8.1f}")
# Predictions at 15 and 20 flatline near the max training y (~50): no extrapolation.
# For trends that must extend beyond seen data, prefer linear models or add trend features.


training time: 0.51s for 200 trees

 x   true y   forest prediction
  5      25        23.1
 10      50        46.7
 15      75        46.7
 20     100        46.7


In [157]:
# Practice: Random Forest

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Gradient Boosting

### Study checklist
- [ ] Boosting intuition
- [ ] Sequential weak learners
- [ ] Learning rate
- [ ] Number of estimators
- [ ] Overfitting controls
- [ ] Comparison with Random Forest
- [ ] Stacking and blending

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Boosting intuition

**Approach:**
- Protects model fit: boosting reduces bias by sequentially fitting new learners to the previous ensemble's errors, which is the opposite mechanism from bagging's variance reduction.
- Why this is the mathematically right approach: gradient boosting builds each new tree to predict the negative gradient of the loss function with respect to the current predictions - for squared-error loss this gradient works out to simply the residuals, so 'fit the next tree to the errors' is a specific, mathematically derived case of gradient descent in function space, not just an intuitive heuristic.
- For this checklist item: Build an additive model where each new weak learner focuses on the previous model's residual errors, gradually improving predictions rather than fitting everything at once.
- Code walkthrough: Check that the printed MSE steadily drops as more trees are used (from 1 tree to 100) - staged_predict lets you watch the ensemble improve tree-by-tree instead of only seeing the final result.

**Learn more:**
- Website: [scikit-learn: Gradient-Boosted Trees](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Gradient+Boosting+Boosting+intuition+machine+learning+theory)

**Trade-offs:**
- Boosting typically reaches a higher accuracy ceiling than Random Forest on the same tabular data, since it directly targets the errors bagging's averaging can leave on the table, but that focus on errors also makes it more sensitive to noisy or mislabeled data.
- Because it's sequential and specifically targets its own past mistakes rather than parallel like bagging, boosting is more prone to overfitting if left unchecked, especially on noisy labels.

**Practical software engineering use cases:**
- When to use it: Use gradient boosting when Random Forest's accuracy has plateaued and you're willing to spend tuning time for a higher ceiling.
- When not to use it: Don't reach for it first on a brand-new problem with unvalidated, possibly noisy labels - its error-fitting focus makes it more prone to chasing label noise than Random Forest.


In [158]:
# Gradient Boosting - Boosting intuition (using scikit-learn)

import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# Boosting builds trees ONE AT A TIME. Each new tree is trained on the RESIDUALS
# (errors) of everything built so far, so it focuses on what is still wrong:
#   prediction = tree1 + lr * tree2 + lr * tree3 + ...
# Contrast with random forest, where trees are independent and simply averaged.

rng = np.random.default_rng(4)
X = rng.uniform(0, 10, 200).reshape(-1, 1)           # 200 points on one feature
y = np.sin(X.ravel()) * 3 + rng.normal(0, 0.3, 200)  # wavy target + noise

model = GradientBoostingRegressor(n_estimators=100,  # build 100 small trees in sequence
                                  max_depth=2,       # each tree is deliberately WEAK
                                  learning_rate=0.1, # shrink each tree's contribution
                                  random_state=4)
model.fit(X, y)

# staged_predict yields the model's prediction after tree 1, tree 2, tree 3, ...
# Watch the error FALL as trees are added - each one repairs remaining mistakes.
print("trees used -> mean squared error")
for i, y_pred in enumerate(model.staged_predict(X), start=1):
    if i in (1, 5, 20, 50, 100):                     # print a few checkpoints only
        mse = np.mean((y - y_pred) ** 2)             # mean squared error at this stage
        print(f"{i:>10} -> {mse:.3f}")


trees used -> mean squared error
         1 -> 3.500
         5 -> 2.093
        20 -> 0.621
        50 -> 0.111
       100 -> 0.050


### Checklist item: Sequential weak learners

**Approach:**
- Protects model fit: each new tree is deliberately shallow and targets residual error, so the ensemble builds up complexity gradually and controllably instead of fitting one large complex model at once.
- Why this is the mathematically right approach: each tree's contribution is added to a running sum, F(x) = F_previous(x) + learning_rate * new_tree(x), so the model's total prediction is mathematically a weighted sum of many small correction steps - this additive structure is exactly what a single step of gradient descent looks like, just repeated many times with a new tree standing in for the gradient direction each round.
- For this checklist item: Train weak learners one after another, feeding each stage the mistakes left by earlier stages so the ensemble improves through ordered correction.
- Code walkthrough: Check that MSE drops after each manually-added stump (round 1, 2, 3) and that sklearn's 100-stump GradientBoostingRegressor reaches an even lower MSE - the manual loop does exactly what the library does, just fewer times.

**Learn more:**
- Website: [scikit-learn: Gradient-Boosted Trees](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Gradient+Boosting+Sequential+weak+learners+machine+learning+theory)

**Trade-offs:**
- Each individual tree in the sequence is deliberately weak and cheap, unlike a Random Forest's fully-grown trees, but building them one at a time instead of in parallel is the direct cost of that design.
- Sequential training means trees can't be built in parallel the way Random Forest's can, so training time scales with the number of estimators; this is what motivates the optimized libraries like XGBoost and LightGBM.

**Practical software engineering use cases:**
- When to use it: Use this mental model when explaining why gradient boosting takes longer to train than Random Forest for a similar number of trees.
- When not to use it: Don't expect to parallelize training across trees the way you can with Random Forest - each tree genuinely depends on the previous one's output.


In [159]:
# Gradient Boosting - Sequential weak learners (using scikit-learn)

import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

# "Weak learner" = a model only slightly better than guessing (here: a tiny tree).
# Boosting chains them: each learner fits the RESIDUALS left by the previous ones.
# Below we do ONE round of boosting manually, then confirm sklearn does the same idea.

rng = np.random.default_rng(8)
X = rng.uniform(0, 10, 100).reshape(-1, 1)
y = 2 * X.ravel() + rng.normal(0, 1, 100)

# --- Manual boosting, round by round ---
pred = np.full_like(y, y.mean())                     # round 0: predict the mean for everyone
print(f"round 0 (just the mean): MSE = {np.mean((y - pred) ** 2):.3f}")

for round_num in [1, 2, 3]:
    residuals = y - pred                             # what is still unexplained
    stump = DecisionTreeRegressor(max_depth=1)       # a "stump": one split, very weak
    stump.fit(X, residuals)                          # fit the stump TO THE RESIDUALS
    pred = pred + stump.predict(X)                   # add its correction to our prediction
    print(f"round {round_num} (added a stump):  MSE = {np.mean((y - pred) ** 2):.3f}")

# --- The library does exactly this, many more times, with a learning rate ---
gb = GradientBoostingRegressor(n_estimators=100, max_depth=1, random_state=8).fit(X, y)
print(f"sklearn, 100 stumps:     MSE = {np.mean((y - gb.predict(X)) ** 2):.3f}")


round 0 (just the mean): MSE = 28.462
round 1 (added a stump):  MSE = 7.735
round 2 (added a stump):  MSE = 4.518
round 3 (added a stump):  MSE = 3.934
sklearn, 100 stumps:     MSE = 0.585


### Checklist item: Learning rate

**Approach:**
- Protects model fit: the learning rate scales how much each new tree's correction is trusted, directly trading training speed against generalization.
- Why this is the mathematically right approach: the learning rate multiplies each new tree's contribution to the running sum before adding it in, exactly analogous to the step-size parameter in gradient descent, alpha in w_new = w_old - alpha*gradient - it directly controls how large a correction each boosting round is allowed to make.
- For this checklist item: Shrink each learner's contribution with a learning rate; smaller steps usually need more estimators but can generalize better.
- Code walkthrough: Check whether the small-learning-rate, many-trees combination (0.05, 400) matches or beats the aggressive (1.0, 50) combination on test accuracy - that's the direct trade-off between learning rate and tree count.

**Learn more:**
- Website: [scikit-learn: Gradient-Boosted Trees](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Gradient+Boosting+Learning+rate+machine+learning+theory)

**Trade-offs:**
- Learning rate gives gradient boosting a smoother dial on model complexity than a tree ensemble's depth or leaf-count knobs alone, but tuning it well requires jointly adjusting the number of trees, unlike Random Forest where more trees are close to a free lunch.
- A low learning rate generalizes better but needs many more estimators, and more training time, to reach the same fit, so tune learning rate and number of estimators together, not independently.

**Practical software engineering use cases:**
- When to use it: Use a small learning rate (0.05-0.1) paired with more estimators as the standard, more-generalizable default configuration.
- When not to use it: Don't set a high learning rate for a quick test and then ship that config to production - it tends to overfit faster and needs re-tuning before going live.


In [160]:
# Gradient Boosting - Learning rate (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

# learning_rate scales how much each new tree contributes:
#   high (e.g., 1.0) -> big steps -> learns fast, can overshoot/overfit
#   low  (e.g., 0.05) -> small careful steps -> usually generalizes better,
#                        but needs MORE trees to get there
# learning_rate and n_estimators trade off against each other.

X, y = make_classification(n_samples=600, n_features=20, n_informative=6,
                           flip_y=0.1, random_state=6)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=6)

print("learning_rate  n_estimators  test_accuracy")
for lr, n_trees in [(1.0, 50),      # aggressive: few big steps
                    (0.1, 200),     # the common default zone
                    (0.05, 400)]:   # gentle: many small steps
    model = GradientBoostingClassifier(learning_rate=lr,
                                       n_estimators=n_trees,
                                       random_state=6).fit(X_train, y_train)
    print(f"{lr:<14} {n_trees:<13} {model.score(X_test, y_test):.3f}")

# Practical recipe: fix learning_rate around 0.05-0.1, then increase n_estimators
# with early stopping (covered in the XGBoost/LightGBM cells) until test error stops improving.


learning_rate  n_estimators  test_accuracy
1.0            50            0.689
0.1            200           0.739
0.05           400           0.733


### Checklist item: Number of estimators

**Approach:**
- Protects evaluation integrity: the number of boosting rounds determines when the model transitions from underfit to overfit, since each round keeps reducing training error even after test error starts rising.
- Why this is the mathematically right approach: since gradient boosting's training loss is guaranteed to keep decreasing, or plateau, as more trees are added, mathematically approaching zero given enough trees and depth, the number of estimators isn't bounded by the training objective itself - it has to be chosen using a separate validation signal, which is exactly what early stopping formalizes.
- For this checklist item: Track performance as trees are added: too few estimators underfit, while too many can overfit unless learning rate and regularization control complexity.
- Code walkthrough: Check where train accuracy keeps climbing toward 1.0 while test accuracy plateaus or dips - that divergence point marks roughly how many trees you'd want, found without retraining from scratch each time.

**Learn more:**
- Website: [scikit-learn: Gradient-Boosted Trees](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Gradient+Boosting+Number+of+estimators+machine+learning+theory)

**Trade-offs:**
- Unlike Random Forest, where adding more trees rarely hurts, gradient boosting's accuracy can peak and then decline as more trees are added, since each one keeps fitting whatever error remains, including noise.
- Don't pick a fixed number by guesswork; use early stopping on a validation set so the number of estimators is determined by when generalization stops improving, not by a preset budget.

**Practical software engineering use cases:**
- When to use it: Use early stopping to pick the number of estimators automatically rather than hardcoding a number based on one dataset's behavior.
- When not to use it: Don't assume more trees is safer the way it often is for Random Forest - past the right point, more trees actively hurt test performance here.


In [161]:
# Gradient Boosting - Number of estimators (using scikit-learn)

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# n_estimators = how many trees to add. Unlike random forest (where more trees never
# really hurt), boosting keeps fitting residuals - too many trees can OVERFIT.

X, y = make_classification(n_samples=500, n_features=15, n_informative=4,
                           flip_y=0.2,               # noisy labels make overfitting visible
                           random_state=10)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

# Train ONCE with many trees; staged_predict lets us "replay" every intermediate size.
model = GradientBoostingClassifier(n_estimators=400, learning_rate=0.2,
                                   random_state=10).fit(X_train, y_train)

print("n_trees  train_acc  test_acc")
# Walk both staged prediction generators in parallel (train and test predictions per stage).
for i, (pr_tr, pr_te) in enumerate(zip(model.staged_predict(X_train),
                                       model.staged_predict(X_test)), start=1):
    if i in (10, 50, 100, 200, 400):                 # report a few checkpoints
        print(f"{i:<8} {accuracy_score(y_train, pr_tr):.3f}      "
              f"{accuracy_score(y_test, pr_te):.3f}")

# Train accuracy keeps climbing toward 1.0, while test accuracy plateaus (or dips) -
# that divergence point is roughly the right number of trees.


n_trees  train_acc  test_acc
10       0.886      0.787
50       1.000      0.787
100      1.000      0.767
200      1.000      0.780
400      1.000      0.793


### Checklist item: Overfitting controls

**Approach:**
- Protects generalization: because boosting keeps fitting residuals indefinitely, it needs explicit brakes such as shallow trees, subsampling, regularization, and early stopping that bagging-based methods don't need as urgently.
- Why this is the mathematically right approach: each control, shallower trees, a smaller learning rate, row/column subsampling, reduces the effective capacity or step size of each individual correction, directly shrinking how much any single round can overfit to noise in its residuals or subsample - they attack the same overfitting risk from different mathematical angles: capacity, step size, and sampling variance.
- For this checklist item: Control boosted-tree overfitting with shallow trees, subsampling, column sampling, minimum leaf constraints, regularization, and early stopping.
- Code walkthrough: Check m.n_estimators_ for the tamed model - early stopping should have kept it below the 300 upper limit - and compare its smaller train/test gap to the wild model's.

**Learn more:**
- Website: [scikit-learn: Gradient-Boosted Trees](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Gradient+Boosting+Overfitting+controls+machine+learning+theory)

**Trade-offs:**
- Gradient boosting offers more distinct levers for controlling overfitting (learning rate, depth, subsampling, early stopping) than a Random Forest typically needs, which is powerful but adds real tuning burden a bagged ensemble mostly avoids.
- Each control, whether a lower learning rate, shallower trees, or row/column subsampling, trades some fit for generalization, so tune them jointly via cross-validation rather than maxing out any single one.

**Practical software engineering use cases:**
- When to use it: Use a combination of these controls, not just one, whenever training on real-world, plausibly noisy labels.
- When not to use it: Don't max out every control simultaneously (lowest learning rate, shallowest trees, heaviest subsampling) without validating - that combination can just as easily underfit.


In [162]:
# Gradient Boosting - Overfitting controls (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

# The main brakes available on gradient boosting:
#   learning_rate      - smaller steps generalize better
#   max_depth          - shallower trees = weaker learners (3 is a common sweet spot)
#   subsample < 1.0    - each tree sees a random fraction of rows ("stochastic" boosting)
#   n_iter_no_change   - EARLY STOPPING: quit adding trees when validation stops improving

X, y = make_classification(n_samples=600, n_features=20, n_informative=5,
                           flip_y=0.2, random_state=13)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)

# Model 1: no brakes - deep trees, aggressive learning rate.
wild = GradientBoostingClassifier(n_estimators=300, max_depth=6, learning_rate=0.5,
                                  random_state=13).fit(X_train, y_train)

# Model 2: all the brakes on.
tamed = GradientBoostingClassifier(
    n_estimators=300,            # upper limit; early stopping may use fewer
    max_depth=3,                 # weak trees
    learning_rate=0.05,          # small steps
    subsample=0.8,               # each tree sees a random 80% of rows
    validation_fraction=0.15,    # hold out 15% of training data internally...
    n_iter_no_change=10,         # ...and stop if it fails to improve for 10 rounds
    random_state=13).fit(X_train, y_train)

for name, m in [("no brakes", wild), ("with brakes", tamed)]:
    print(f"{name:<12} trees_used={m.n_estimators_:<4} "
          f"train={m.score(X_train, y_train):.3f}  test={m.score(X_test, y_test):.3f}")


no brakes    trees_used=300  train=1.000  test=0.739
with brakes  trees_used=37   train=0.862  test=0.661


### Checklist item: Comparison with Random Forest

**Approach:**
- Protects model selection: knowing that Random Forest reduces variance through parallel bagging while Gradient Boosting reduces bias through sequential residual-fitting tells you which one to reach for given whether your errors look like underfitting or high variance.
- Why this is the mathematically right approach: Random Forest's averaging step provably reduces variance, per the same 1/N argument as bagging, while leaving each tree's individual bias unchanged; Gradient Boosting's additive residual-fitting step provably reduces bias each round, since each new tree explicitly targets what's still wrong, at the risk of eventually fitting noise - the two methods mathematically attack opposite terms of the bias-variance decomposition of error.
- For this checklist item: Compare bagging versus boosting: Random Forest trains independent trees to reduce variance, while Gradient Boosting trains sequential trees to reduce bias and residual error.
- Code walkthrough: Check whether Gradient Boosting's test accuracy edges out Random Forest's, and compare fit_time between the two - sequential boosting is often slower to train for a comparable or better accuracy.

**Learn more:**
- Website: [scikit-learn: Gradient-Boosted Trees](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Gradient+Boosting+Comparison+with+Random+Forest+machine+learning+theory)

**Trade-offs:**
- Both are ensembles of trees, but the direction of their error-reduction is opposite: Random Forest reduces variance by averaging independent trees, while Gradient Boosting reduces bias by chaining trees that fix each other's mistakes.
- Gradient Boosting usually achieves higher accuracy on tabular data but requires more careful tuning and is more sensitive to noisy labels or outliers; Random Forest is the safer, lower-maintenance default when you need something that works reasonably well with minimal tuning.

**Practical software engineering use cases:**
- When to use it: Use this comparison to decide your default: reach for Random Forest first when you need something robust with little tuning, and gradient boosting when you have budget to tune for extra accuracy.
- When not to use it: Don't assume gradient boosting is a strict upgrade over Random Forest in every project - on a team without time to tune it properly, it can underperform a well-defaulted forest.


In [163]:
# Gradient Boosting - Comparison with Random Forest (using scikit-learn)

import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Head-to-head summary:
#   Random Forest      : trees built INDEPENDENTLY, in parallel, then averaged.
#                        Reduces VARIANCE. Very forgiving of default settings.
#   Gradient Boosting  : trees built SEQUENTIALLY, each fixing prior errors.
#                        Reduces BIAS. Usually higher ceiling, but needs tuning
#                        and can overfit if pushed too far.

X, y = make_classification(n_samples=800, n_features=20, n_informative=8,
                           flip_y=0.05, random_state=17)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=17)

for name, model in [
    ("Random Forest    ", RandomForestClassifier(n_estimators=200, random_state=17)),
    ("Gradient Boosting", GradientBoostingClassifier(n_estimators=200, random_state=17)),
]:
    t0 = time.time()                                 # start a timer
    model.fit(X_train, y_train)                      # train
    fit_seconds = time.time() - t0                   # elapsed training time
    print(f"{name} test_acc={model.score(X_test, y_test):.3f}  fit_time={fit_seconds:.2f}s")

print()
print("Rules of thumb: RF = robust low-effort baseline; GB = tune it for the last")
print("few points of accuracy. On tabular data, tuned boosting usually wins.")


Random Forest     test_acc=0.854  fit_time=1.33s
Gradient Boosting test_acc=0.863  fit_time=2.92s

Rules of thumb: RF = robust low-effort baseline; GB = tune it for the last
few points of accuracy. On tabular data, tuned boosting usually wins.


### Checklist item: Stacking and blending

**Approach:**
- Protects model fit: combining several different base models through a meta-model that learns how to weight their predictions can capture complementary strengths that no single base model has on its own.
- Why this is the mathematically right approach: a meta-model trained on the base models' own predictions is solving a small, separate optimization problem, minimize error using base_model_1_prediction, base_model_2_prediction, ... as its inputs - it can mathematically learn to weight or combine those predictions optimally, including down-weighting a base model that's consistently wrong in a particular region, something a simple unweighted average cannot do.
- For this checklist item: Unlike bagging's simple averaging or boosting's sequential error-correction, stacking trains a separate model specifically to learn the best way to combine the base models' outputs.
- Code walkthrough: Check that the stacked model's test accuracy is at least as high as the best individual base model's, confirming the meta-model learned a genuinely useful combination rather than just diluting a strong model with weaker ones.

**Learn more:**
- Website: [scikit-learn: Stacked generalization](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Gradient+Boosting+Stacking+and+blending+machine+learning+theory)

**Trade-offs:**
- Compared to a single strong model like gradient boosting, stacking costs more training time and complexity, since it needs to train multiple base models plus a meta-model, and it's more expensive to explain than any one model alone.
- Stacking works best when the base models make different kinds of errors; stacking together several very similar models tends to add complexity without meaningfully improving on the best one alone.

**Practical software engineering use cases:**
- When to use it: Use stacking when you have several genuinely different model types, like a linear model, a tree, and an SVM, that each capture different patterns, and the extra accuracy is worth the added complexity.
- When not to use it: Don't reach for stacking as a first approach on a new problem - get a single well-tuned model working first, and only add stacking if you've confirmed the base models actually make complementary mistakes.


In [ ]:
# Gradient Boosting - Stacking and blending (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier

X, y = make_classification(n_samples=500, n_features=10, n_informative=5, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

base_models = [
    ("logreg", LogisticRegression()),
    ("tree", DecisionTreeClassifier(max_depth=4, random_state=0)),
    ("svm", SVC(probability=True, random_state=0)),
]

stack = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())
stack.fit(X_train, y_train)

for name, model in base_models:
    model.fit(X_train, y_train)
    print(f"{name:8s} test accuracy: {model.score(X_test, y_test):.3f}")
print(f"{'stacked':8s} test accuracy: {stack.score(X_test, y_test):.3f}")


In [164]:
# Practice: Gradient Boosting

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## XGBoost / LightGBM / CatBoost

### Study checklist
- [ ] Why gradient boosting libraries are powerful
- [ ] Handling tabular data
- [ ] Important hyperparameters
- [ ] Categorical handling differences
- [ ] Early stopping
- [ ] Model interpretation

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Why gradient boosting libraries are powerful

**Approach:**
- Protects deployment reliability: these libraries add regularization, missing-value handling, and parallelized histogram-based split-finding on top of vanilla gradient boosting, which is why they dominate tabular ML competitions and production systems.
- Why this is the mathematically right approach: these libraries add a regularization term directly into the loss function minimized at each boosting round, an explicit penalty on tree complexity rather than an external constraint, which is what lets them control overfitting more precisely than the vanilla gradient boosting objective alone.
- For this checklist item: Use specialized boosting libraries for production tabular modeling because they add efficient tree building, missing-value handling, regularization, early stopping, and categorical-feature support.
- Code walkthrough: Check that the last predict() call succeeds even though X_missing has NaN values poked into it - sklearn's own GradientBoostingClassifier would raise an error here, which is the concrete advantage being demonstrated.

**Learn more:**
- Website: [XGBoost Documentation](https://xgboost.readthedocs.io/en/stable/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=XGBoost+%2F+LightGBM+%2F+CatBoost+Why+gradient+boosting+libraries+are+powerful+machine+learning+theory)

**Trade-offs:**
- Compared to scikit-learn's own GradientBoostingClassifier, these libraries add histogram-based training and native missing-value handling that can cut training time by an order of magnitude on large tabular datasets.
- The extra machinery, including many hyperparameters and GPU or histogram options, buys speed and accuracy but adds tuning surface area and library-specific quirks, so budget time for hyperparameter search, not just model fitting.

**Practical software engineering use cases:**
- When to use it: Use one of these libraries once your dataset or feature count outgrows what plain scikit-learn's GradientBoostingClassifier can train in reasonable time.
- When not to use it: Don't add this dependency for a small dataset that trains in seconds with sklearn's built-in boosting - the extra library and tuning surface isn't worth it yet.


In [165]:
# XGBoost / LightGBM / CatBoost - Why gradient boosting libraries are powerful
# One-time setup:  pip install xgboost lightgbm catboost

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier                    # XGBoost's scikit-learn-style wrapper

# These libraries implement the SAME boosting idea as sklearn's GradientBoosting,
# but add: histogram-based fast training, native missing-value handling,
# built-in regularization (L1/L2 on leaf weights), early stopping, and GPU support.
# They follow the familiar fit / predict / score API, so switching is painless.

X, y = make_classification(n_samples=1000, n_features=20, n_informative=8,
                           random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

model = XGBClassifier(
    n_estimators=200,        # number of boosting rounds (trees)
    max_depth=4,             # depth of each tree
    learning_rate=0.1,       # shrinkage per tree
    eval_metric="logloss",   # metric used internally during training
    random_state=0,
)
model.fit(X_train, y_train)                          # same API as any sklearn model

print("XGBoost test accuracy:", round(model.score(X_test, y_test), 3))

# Handles missing values natively - sklearn's GradientBoosting would raise an error:
import numpy as np
X_missing = X_test.copy()
X_missing[0, :5] = np.nan                            # poke NaN holes into the first row
print("prediction with NaNs in the input:", model.predict(X_missing[:1]))


XGBoost test accuracy: 0.947
prediction with NaNs in the input: [0]


### Checklist item: Handling tabular data

**Approach:**
- Protects representation: these libraries are specifically optimized for structured tabular data with mixed numeric and categorical columns and missing values, which is why they usually beat generic deep learning on tabular problems.
- Why this is the mathematically right approach: a tree-based split only ever asks whether a value is above or below a threshold, a comparison that requires no assumption about a feature's scale or distribution shape - that's mathematically why trees handle mixed-scale, non-normal tabular features without the preprocessing a distance- or gradient-based model needs.
- For this checklist item: These libraries handle missing values natively, support categorical features, and are optimized for columnar tabular data â€” the default choice before deep learning.
- Code walkthrough: Check that model.fit() runs without any scaling, imputation, or interaction-feature code despite the messy skewed, missing input - that absence of preprocessing code is the point of this cell.

**Learn more:**
- Website: [XGBoost Documentation](https://xgboost.readthedocs.io/en/stable/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=XGBoost+%2F+LightGBM+%2F+CatBoost+Handling+tabular+data+machine+learning+theory)

**Trade-offs:**
- On tabular data, these libraries routinely outperform deep learning trained on the same data, since trees natively exploit the mixed-type, non-smooth structure that neural networks need heavy feature engineering to match.
- Their strength on tabular data doesn't transfer to unstructured data like images, text, or audio, so don't reach for XGBoost on raw pixels or raw text embeddings expecting the same edge.

**Practical software engineering use cases:**
- When to use it: Use these libraries specifically for structured, mixed-type tabular data such as transaction logs, spreadsheets, or SQL exports.
- When not to use it: Don't reach for them on raw images, audio, or unstructured text embeddings expecting the same edge they have on tabular data - that's not their strength.


In [166]:
# XGBoost / LightGBM / CatBoost - Handling tabular data
# One-time setup:  pip install lightgbm

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

# Gradient boosting libraries are the go-to for TABULAR data (rows x columns, like
# spreadsheets / SQL tables) because trees natively handle what tables throw at you:
# mixed numeric scales (no scaling needed), skewed values, missing data, and
# non-linear feature interactions.

rng = np.random.default_rng(1)
n = 800

# Build a realistic messy table with pandas.
df = pd.DataFrame({
    "age": rng.integers(18, 70, n),                          # ordinary integer feature
    "income": np.round(rng.lognormal(10.5, 0.6, n), 0),      # heavily skewed feature
    "visits": rng.poisson(3, n),                             # count feature
})
df.loc[rng.random(n) < 0.10, "income"] = np.nan              # make 10% of income missing

# Target depends non-linearly on the features (an interaction between age and visits).
target = ((df["age"] > 40) & (df["visits"] > 2)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.3,
                                                    random_state=1)

model = LGBMClassifier(n_estimators=200, random_state=1, verbose=-1)  # verbose=-1: quiet
model.fit(X_train, y_train)                          # NaNs and skew handled automatically

print("test accuracy on messy tabular data:", round(model.score(X_test, y_test), 3))
print("(no scaling, no imputation, no manual interaction features were needed)")


test accuracy on messy tabular data: 1.0
(no scaling, no imputation, no manual interaction features were needed)


### Checklist item: Important hyperparameters

**Approach:**
- Protects model fit: a handful of hyperparameters, namely learning rate, max depth or leaves, number of estimators, subsample ratio, and regularization terms, control almost all of the bias-variance behavior, so tuning effort should concentrate there first.
- Why this is the mathematically right approach: each hyperparameter controls a distinct term in the model's effective complexity - tree depth bounds the function space each tree can represent, subsampling introduces randomness that provably reduces variance similar to bagging, and regularization terms directly penalize large leaf weights in the loss function - tuning them adjusts different mathematical levers on the same bias-variance trade-off, not redundant knobs.
- For this checklist item: The three most impactful knobs are n_estimators (trees), learning_rate (step size), and max_depth (complexity); they interact â€” lower lr needs more trees.
- Code walkthrough: Check whether the tuned configuration's test accuracy beats the defaults' - if it doesn't by much, that's informative too: these hyperparameters interact, and untuned defaults are already decent.

**Learn more:**
- Website: [XGBoost Documentation](https://xgboost.readthedocs.io/en/stable/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=XGBoost+%2F+LightGBM+%2F+CatBoost+Important+hyperparameters+machine+learning+theory)

**Trade-offs:**
- These libraries expose many more tunable knobs than scikit-learn's GradientBoostingClassifier, such as column subsampling and multiple regularization terms, which is exactly the extra tuning surface that can push accuracy higher if you're willing to spend the search budget.
- Grid-searching all hyperparameters jointly is expensive; a common practical shortcut is to fix a small learning rate, use early stopping to pick the number of estimators, then tune tree structure and regularization, since full joint search rarely justifies its cost.

**Practical software engineering use cases:**
- When to use it: Use the shortcut recipe (fix a small learning rate, use early stopping for n_estimators, then tune tree structure and regularization) rather than a full grid search across every parameter.
- When not to use it: Don't tune every hyperparameter simultaneously via exhaustive grid search on a large dataset - the combinatorial cost rarely pays for itself over the staged approach.


In [167]:
# XGBoost / LightGBM / CatBoost - Important hyperparameters
# One-time setup:  pip install xgboost

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# The hyperparameters that matter most, in rough order:
#   n_estimators      - number of trees (pair with early stopping, next cells)
#   learning_rate     - step size; lower = safer but needs more trees
#   max_depth         - tree depth; 3-8 typical (deeper = more interactions, more overfit)
#   subsample         - fraction of ROWS each tree sees (e.g., 0.8)
#   colsample_bytree  - fraction of COLUMNS each tree sees (e.g., 0.8)
#   reg_alpha (L1) / reg_lambda (L2) - penalties on leaf weights

X, y = make_classification(n_samples=800, n_features=20, n_informative=6,
                           flip_y=0.1, random_state=2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2)

# Compare untuned defaults vs a lightly tuned configuration.
default_model = XGBClassifier(eval_metric="logloss", random_state=2)
tuned_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,      # smaller steps than the 0.3 default
    max_depth=4,             # slightly shallower than the 6 default
    subsample=0.8,           # row sampling adds randomness -> less overfitting
    colsample_bytree=0.8,    # column sampling does the same
    reg_lambda=2.0,          # a bit more L2 regularization
    eval_metric="logloss",
    random_state=2,
)

for name, m in [("defaults", default_model), ("tuned   ", tuned_model)]:
    m.fit(X_train, y_train)
    print(f"{name}: test accuracy = {m.score(X_test, y_test):.3f}")


defaults: test accuracy = 0.796
tuned   : test accuracy = 0.808


### Checklist item: Categorical handling differences

**Approach:**
- Protects representation: CatBoost natively handles categorical features via ordered target statistics, LightGBM handles them via histogram-based splits, while XGBoost historically requires manual encoding, which changes your preprocessing pipeline depending on which library you pick.
- Why this is the mathematically right approach: native categorical splitting evaluates a statistic, like the target mean, per category and orders categories by it before searching for the best binary split among them - this converts an otherwise combinatorially expensive search, 2^(k-1) possible groupings for k categories, into a linear, sorted search that's mathematically tractable at the scale these libraries operate.
- For this checklist item: XGBoost requires one-hot or label encoding; LightGBM uses native integer indices; CatBoost handles raw string columns with ordered target encoding.
- Code walkthrough: Check that no one-hot encoding code appears anywhere before model.fit() - LightGBM reads the 'category' dtype directly and handles the city column natively, which is the point being shown.

**Learn more:**
- Website: [XGBoost Documentation](https://xgboost.readthedocs.io/en/stable/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=XGBoost+%2F+LightGBM+%2F+CatBoost+Categorical+handling+differences+machine+learning+theory)

**Trade-offs:**
- Native categorical support removes a preprocessing step that older XGBoost versions and non-tree models like SVM or logistic regression cannot skip - those still require one-hot or target encoding before training.
- Native categorical handling saves preprocessing work and can improve accuracy on high-cardinality categoricals, but makes the model harder to port between libraries and can leak target information if not done carefully, which CatBoost mitigates with ordered boosting.

**Practical software engineering use cases:**
- When to use it: Use LightGBM or CatBoost's native categorical support when your data has high-cardinality categorical columns, like city or SKU, that would explode with one-hot encoding.
- When not to use it: Don't mix native categorical dtypes with a library version of XGBoost that doesn't support them without first checking - you'll silently fall back to treating them as numeric.


In [168]:
# XGBoost / LightGBM / CatBoost - Categorical handling differences
# One-time setup:  pip install lightgbm

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

# How each library treats categorical features (like "city" or "browser"):
#   XGBoost  : enable_categorical=True with pandas 'category' dtype (newer versions),
#              else you must one-hot / ordinal-encode first
#   LightGBM : native support - just make the column dtype 'category' (shown below)
#   CatBoost : the best-in-class - pass cat_features=[...]; it uses target statistics
# Native handling beats one-hot when a column has MANY categories (no column explosion).

rng = np.random.default_rng(3)
n = 600
cities = rng.choice(["delhi", "mumbai", "pune", "chennai"], size=n)

df = pd.DataFrame({
    "city": pd.Categorical(cities),                  # dtype 'category' = the magic step
    "age": rng.integers(18, 65, n),
})
# Make the target genuinely depend on the category (delhi/mumbai more likely 1).
city_effect = pd.Series(cities).map({"delhi": 2, "mumbai": 1.5, "pune": 0, "chennai": -1})
target = ((city_effect + rng.normal(0, 1, n)) > 0.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.3,
                                                    random_state=3)

model = LGBMClassifier(n_estimators=150, random_state=3, verbose=-1)
model.fit(X_train, y_train)                          # LightGBM sees the dtype and handles it

print("test accuracy with native categorical handling:", round(model.score(X_test, y_test), 3))
print("(no one-hot encoding was written - the 'category' dtype did the work)")


test accuracy with native categorical handling: 0.811
(no one-hot encoding was written - the 'category' dtype did the work)


### Checklist item: Early stopping

**Approach:**
- Protects evaluation integrity and training efficiency: monitoring a validation metric and halting once it stops improving prevents both overfitting and wasted compute on unnecessary boosting rounds.
- Why this is the mathematically right approach: since training loss decreases monotonically with more boosting rounds, but validation loss only decreases while genuine signal remains to fit, watching for the round where validation loss stops improving pinpoints, mathematically, the model complexity where further fitting captures only noise.
- For this checklist item: Stop training when validation metric stops improving for N rounds; avoids overfitting without a fixed tree count.
- Code walkthrough: Check model.best_iteration + 1 - it should be far below the 1000 allowed trees, showing early stopping found a smaller, adequate ensemble size automatically from the validation set.

**Learn more:**
- Website: [XGBoost Documentation](https://xgboost.readthedocs.io/en/stable/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=XGBoost+%2F+LightGBM+%2F+CatBoost+Early+stopping+machine+learning+theory)

**Trade-offs:**
- Early stopping plays the same role max_depth or ccp_alpha play for a single decision tree, but it tunes model complexity automatically from data rather than requiring you to guess or grid-search a fixed value in advance.
- Early stopping needs its own held-out validation set separate from the final test set, which shrinks the data available for training; on small datasets this trade-off may not be worth it versus a fixed, cross-validated number of rounds.

**Practical software engineering use cases:**
- When to use it: Use early stopping whenever you set n_estimators generously high, so the actual tree count is chosen from data instead of guessed.
- When not to use it: Don't rely on early stopping if your dataset is too small to spare a separate validation split - the stopping decision itself becomes noisy.


In [169]:
# XGBoost / LightGBM / CatBoost - Early stopping
# One-time setup:  pip install xgboost

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Early stopping = set n_estimators generously HIGH, monitor a validation set every
# round, and STOP when validation performance hasn't improved for N rounds.
# You get the right number of trees automatically instead of guessing it.

X, y = make_classification(n_samples=1000, n_features=20, n_informative=6,
                           flip_y=0.15, random_state=4)

# We need THREE splits: train (fit trees), validation (watch for stopping), test (final).
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=4)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25,
                                                  random_state=4)

model = XGBClassifier(
    n_estimators=1000,           # deliberately way too many - stopping will trim it
    learning_rate=0.05,
    early_stopping_rounds=20,    # stop after 20 rounds with no validation improvement
    eval_metric="logloss",
    random_state=4,
)
model.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],                 # the set being monitored each round
          verbose=False)                             # silence per-round printing

print("trees actually kept:", model.best_iteration + 1, "out of 1000 allowed")
print("test accuracy:", round(model.score(X_test, y_test), 3))
# predict()/score() automatically use the best iteration found - nothing extra to do.


trees actually kept: 85 out of 1000 allowed
test accuracy: 0.79


### Checklist item: Model interpretation

**Approach:**
- Protects user value and debugging: feature importance and SHAP values are how you explain a black-box ensemble of hundreds of trees to a stakeholder or auditor.
- Why this is the mathematically right approach: SHAP values are provably the unique attribution method satisfying additivity, symmetry, and consistency, three fairness properties borrowed from cooperative game theory's Shapley value - that mathematical uniqueness is why SHAP is treated as a principled standard rather than one heuristic among many for explaining predictions.
- For this checklist item: Use gain-based feature importance (split information) from the tree structure; cross-check with permutation importance or SHAP for reliability.
- Code walkthrough: Check that the top 5 printed features are exactly feature_0, feature_1, and feature_2 in some order - matching how the synthetic data was built with shuffle=False confirms the importances are identifying real signal.

**Learn more:**
- Website: [XGBoost Documentation](https://xgboost.readthedocs.io/en/stable/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=XGBoost+%2F+LightGBM+%2F+CatBoost+Model+interpretation+machine+learning+theory)

**Trade-offs:**
- A boosted ensemble of hundreds of trees is far less directly readable than a single decision tree, which is the interpretability cost paid for the accuracy gain - importance scores and SHAP exist specifically to buy some of that readability back.
- Built-in gain-based importance is fast but can be misleading, since it's biased toward high-cardinality features like tree-based importance in general; SHAP is more reliable but noticeably more expensive to compute on large models or datasets.

**Practical software engineering use cases:**
- When to use it: Use SHAP when you need a per-prediction explanation for an individual customer or transaction, not just an overall feature ranking.
- When not to use it: Don't default to SHAP on every large-scale batch job without considering cost - it's noticeably more expensive to compute than built-in importances at scale.


In [170]:
# XGBoost / LightGBM / CatBoost - Model interpretation
# One-time setup:  pip install xgboost  (and optionally: pip install shap)

import numpy as np
from sklearn.datasets import make_classification
from xgboost import XGBClassifier

# Boosted models are not black boxes - your interpretation toolkit:
#   1. feature_importances_       - global "how much was each feature used" (quick look)
#   2. importance_type variants   - 'gain' (quality of splits) vs 'weight' (count of splits)
#   3. SHAP values                - per-PREDICTION explanations (the gold standard)

# Data where the truth is known: only features 0-2 are informative, 3-9 are noise.
X, y = make_classification(n_samples=600, n_features=10, n_informative=3,
                           n_redundant=0, shuffle=False, random_state=5)

model = XGBClassifier(n_estimators=150, max_depth=4, eval_metric="logloss",
                      random_state=5).fit(X, y)

# Global importances, sorted from most to least important.
imp = model.feature_importances_                     # normalized, sums to 1
for idx in np.argsort(imp)[::-1][:5]:                # top 5 features
    print(f"feature_{idx}: importance = {imp[idx]:.3f}")
print("(features 0-2 should top the list - matching how the data was built)")

# For per-prediction explanations, SHAP answers "why did THIS row get THIS score?":
#   import shap
#   explainer = shap.TreeExplainer(model)            # fast exact SHAP for tree models
#   shap_values = explainer.shap_values(X[:5])       # contribution of each feature, per row
#   shap.summary_plot(shap_values, X)                # global beeswarm view


feature_0: importance = 0.636
feature_1: importance = 0.072
feature_2: importance = 0.069
feature_9: importance = 0.038
feature_8: importance = 0.035
(features 0-2 should top the list - matching how the data was built)


In [171]:
# Practice: XGBoost / LightGBM / CatBoost

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Support Vector Machines

### Study checklist
- [ ] Maximum margin classifier
- [ ] Kernel trick
- [ ] Linear vs RBF kernel
- [ ] C and gamma
- [ ] Scaling requirement
- [ ] When SVM is useful

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Maximum margin classifier

**Approach:**
- Protects model fit and generalization: maximizing the margin between classes, rather than just finding any separating boundary, is what gives SVM strong generalization on small-to-medium datasets.
- Why this is the mathematically right approach: the margin width is provably 2/||w||, so maximizing the margin is mathematically equivalent to minimizing ||w||^2 subject to every point being correctly classified - a well-posed quadratic optimization problem with a unique solution, which is why SVM's boundary is precisely determined rather than one of many equally-valid separating lines.
- For this checklist item: SVM finds the hyperplane that maximizes the gap (margin) between classes; points on the margin boundary are the support vectors.
- Code walkthrough: Check that model.support_vectors_ only lists points near the boundary between the two clusters, not every training point - those few points are literally all that determined the line's position.

**Learn more:**
- Website: [scikit-learn: Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Support+Vector+Machines+Maximum+margin+classifier+machine+learning+theory)

**Trade-offs:**
- Unlike logistic regression, which fits every point's likelihood, SVM's boundary is determined entirely by the handful of support vectors nearest the margin - most training points could be deleted without changing the result at all.
- A hard margin requires perfectly separable data, which real data rarely is; the soft-margin formulation with slack variables, controlled by C, is what makes SVM usable in practice.

**Practical software engineering use cases:**
- When to use it: Use SVM when you expect a genuinely clean margin between classes and want a boundary defined by only the hardest-to-classify points.
- When not to use it: Don't use it expecting to inspect typical data points' influence on the model - only the support vectors near the margin matter, not the bulk of the training set.


In [172]:
# Support Vector Machines - Maximum margin classifier (using scikit-learn)

import numpy as np
from sklearn.svm import SVC

# An SVM finds the separating line (hyperplane) with the WIDEST possible gap
# ("margin") to the nearest points of each class. Those nearest points are the
# SUPPORT VECTORS - they alone define the boundary; every other point is ignorable.

# Two clearly separated groups of 2D points.
X = np.array([[1, 2], [2, 3], [1, 3],                # class 0 (left cluster)
              [5, 1], [6, 2], [5, 3]])               # class 1 (right cluster)
y = np.array([0, 0, 0, 1, 1, 1])

model = SVC(kernel="linear",                         # straight-line boundary
            C=1.0)                                   # margin hardness (next cells)
model.fit(X, y)

# Which training points ended up as support vectors?
print("support vectors (the points that define the boundary):")
print(model.support_vectors_)
print("their indices in X:", model.support_)
print()

# The learned line is w . x + b = 0; margin width = 2 / ||w||.
w = model.coef_[0]                                   # weight vector of the line
b = model.intercept_[0]                              # offset of the line
margin = 2 / np.linalg.norm(w)                       # ||w|| = length of w
print(f"boundary: {w[0]:.2f}*x0 + {w[1]:.2f}*x1 + {b:.2f} = 0")
print(f"margin width: {margin:.2f}  (SVM chose the line that MAXIMIZES this)")


support vectors (the points that define the boundary):
[[2. 3.]
 [5. 3.]]
their indices in X: [1 5]

boundary: 0.67*x0 + 0.00*x1 + -2.33 = 0
margin width: 3.00  (SVM chose the line that MAXIMIZES this)


### Checklist item: Kernel trick

**Approach:**
- Protects representation: the kernel trick lets SVM find nonlinear boundaries by implicitly computing similarity in a higher-dimensional space, without ever materializing that space and its computational blowup.
- Why this is the mathematically right approach: the SVM optimization only ever needs the dot product between pairs of points, never their raw coordinates directly - a kernel function computes what that dot product would be in a higher-dimensional space without ever calculating the coordinates in that space, which is mathematically why it avoids the computational blowup of actually transforming the data.
- For this checklist item: The kernel trick computes dot products in a high-dimensional (or infinite-dimensional) feature space without explicitly constructing the mapping Ï†(x).
- Code walkthrough: Check that the linear kernel's test accuracy is barely above chance while the RBF kernel's accuracy is high - that gap is the kernel trick succeeding on data no straight line could ever separate.

**Learn more:**
- Website: [scikit-learn: Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Support+Vector+Machines+Kernel+trick+machine+learning+theory)

**Trade-offs:**
- The kernel trick lets SVM match a neural network's ability to learn nonlinear boundaries without ever building the network, but it pays for that flexibility in training time that scales far worse with dataset size than a tree ensemble's does.
- Kernel choice is a modeling decision, not a free lunch; the wrong kernel can underfit, as with a linear kernel on nonlinear data, or overfit, as with RBF and too-small gamma, and kernel SVMs don't scale well past tens of thousands of rows.

**Practical software engineering use cases:**
- When to use it: Use a kernel SVM when you suspect a non-linear boundary but don't want to hand-engineer the features to make it linear.
- When not to use it: Don't use a kernel SVM on a dataset with hundreds of thousands of rows - training cost grows too fast; consider a tree ensemble instead.


In [173]:
# Support Vector Machines - Kernel trick (using scikit-learn)

from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# Some data is NOT separable by any straight line - e.g., one class forming a ring
# around the other. The KERNEL TRICK implicitly maps points into a higher-dimensional
# space where a line CAN separate them, without ever computing that space explicitly.
# The kernel function computes similarity between points as if they lived there.

# make_circles: class 1 = inner circle, class 0 = surrounding ring. No line separates them.
X, y = make_circles(n_samples=300, factor=0.4,       # factor = inner/outer radius ratio
                    noise=0.08, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

# Attempt 1: a linear SVM - doomed on circular data.
linear_svm = SVC(kernel="linear").fit(X_train, y_train)
print("linear kernel test accuracy:", round(linear_svm.score(X_test, y_test), 3),
      " <- barely better than guessing")

# Attempt 2: RBF (radial basis function) kernel - similarity based on distance,
# equivalent to an infinite-dimensional mapping. Circles become trivially separable.
rbf_svm = SVC(kernel="rbf").fit(X_train, y_train)
print("rbf kernel test accuracy:   ", round(rbf_svm.score(X_test, y_test), 3),
      " <- the kernel trick at work")


linear kernel test accuracy: 0.411  <- barely better than guessing
rbf kernel test accuracy:    1.0  <- the kernel trick at work


### Checklist item: Linear vs RBF kernel

**Approach:**
- Protects model fit: this choice is really a bias-variance decision, since a linear kernel assumes a linear boundary with higher bias and lower variance, while RBF assumes a flexible local boundary with lower bias and higher variance.
- Why this is the mathematically right approach: the RBF kernel, exp(-gamma*||x-x'||^2), corresponds to an infinite-dimensional feature space, a mathematical fact from its Taylor series expansion, giving it strictly more representational flexibility than a linear kernel's finite-dimensional dot product - which is exactly why it can fit non-linear boundaries a linear kernel structurally cannot represent.
- For this checklist item: Use linear kernel when #features >> #samples (e.g., text); use RBF when the boundary is non-linear and features are scaled.
- Code walkthrough: Check that linear and RBF perform similarly on the 'linear-ish data' but RBF clearly wins on the 'moons data' - matching kernel choice to whether the true boundary is straight or curved.

**Learn more:**
- Website: [scikit-learn: Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Support+Vector+Machines+Linear+vs+RBF+kernel+machine+learning+theory)

**Trade-offs:**
- A linear-kernel SVM is essentially as fast and nearly as interpretable as logistic regression, while RBF trades both of those away for the ability to fit curved boundaries neither logistic regression nor a linear SVM can represent.
- Try linear first, especially on high-dimensional or sparse data like text where classes are often already near-linearly separable and RBF's extra flexibility mostly adds overfitting risk and training time.

**Practical software engineering use cases:**
- When to use it: Use the linear kernel first on high-dimensional or sparse data like TF-IDF text features, where classes are often close to linearly separable already.
- When not to use it: Don't default to RBF to be safe on such data - the extra flexibility mostly adds overfitting risk and training time without an accuracy payoff.


In [174]:
# Support Vector Machines - Linear vs RBF kernel (using scikit-learn)

from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# Choosing between the two most common kernels:
#   linear : boundary is a straight line/plane. Fast, works well when data is
#            (near-)linearly separable or has MANY features (e.g., text TF-IDF).
#   rbf    : curved, flexible boundary. The default choice when the relationship
#            is non-linear and the feature count is moderate.

# Dataset A: built to be linearly separable-ish.
X_lin, y_lin = make_classification(n_samples=400, n_features=8, n_informative=8,
                                   n_redundant=0, class_sep=2.0, random_state=1)
# Dataset B: two interleaving half-moons - inherently non-linear.
X_moon, y_moon = make_moons(n_samples=400, noise=0.2, random_state=1)

for name, (X, y) in [("linear-ish data", (X_lin, y_lin)),
                     ("moons data     ", (X_moon, y_moon))]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=1)
    lin_acc = SVC(kernel="linear").fit(X_tr, y_tr).score(X_te, y_te)   # train + score
    rbf_acc = SVC(kernel="rbf").fit(X_tr, y_tr).score(X_te, y_te)
    print(f"{name}: linear={lin_acc:.3f}  rbf={rbf_acc:.3f}")

# Expected pattern: on linear-ish data both do well (prefer linear - simpler, faster);
# on the moons, rbf clearly wins. Try linear first; switch to rbf if it underfits.


linear-ish data: linear=0.975  rbf=0.983
moons data     : linear=0.883  rbf=0.933


### Checklist item: C and gamma

**Approach:**
- Protects model fit: C controls the margin and misclassification trade-off, and gamma controls how far a single training point's influence reaches, together defining the effective complexity of an RBF-kernel SVM.
- Why this is the mathematically right approach: C appears in the optimization objective as the penalty weight on margin violations, directly trading margin width for training-point correctness, while gamma controls how quickly the RBF kernel's similarity value decays with distance - both are literal coefficients inside the mathematical objective SVM solves, not external tuning heuristics layered on top of it.
- For this checklist item: C trades off margin width vs misclassifications (small C = wider margin, more errors tolerated); gamma controls the RBF influence radius (small Î³ = smoother boundary).
- Code walkthrough: Check the combination with the highest C and highest gamma - it should show the classic overfit signature of train accuracy near 1.0 paired with a noticeably lower test accuracy.

**Learn more:**
- Website: [scikit-learn: Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Support+Vector+Machines+C+and+gamma+machine+learning+theory)

**Trade-offs:**
- C and gamma give an RBF-kernel SVM two independent complexity dials, more granular control than a tree's single depth parameter, but that granularity means a two-dimensional grid search instead of a one-dimensional one.
- C and gamma interact strongly, for example high C with high gamma both push toward overfitting, so they should be grid-searched together, not tuned one at a time.

**Practical software engineering use cases:**
- When to use it: Use a joint grid search over C and gamma whenever tuning an RBF SVM - they interact strongly enough that tuning one at a time gives misleading results.
- When not to use it: Don't leave both at their library defaults for a production model without at least one validation sweep - the wrong combination can silently overfit or underfit.


In [175]:
# Support Vector Machines - C and gamma (using scikit-learn)

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# The two knobs that control an RBF-kernel SVM:
#   C     - error tolerance. LOW C = wide margin, allows misclassifications (smoother,
#           may underfit). HIGH C = punishes every error (tighter fit, may overfit).
#   gamma - reach of each training point. LOW gamma = far-reaching influence (smooth
#           boundary). HIGH gamma = each point only influences its tiny neighborhood
#           (wiggly boundary that can wrap around single noisy points = overfitting).

X, y = make_moons(n_samples=400, noise=0.3, random_state=2)   # noisy, non-linear data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2)

print("   C     gamma   train_acc  test_acc")
for C in [0.1, 1, 100]:
    for gamma in [0.1, 1, 10]:
        model = SVC(kernel="rbf", C=C, gamma=gamma).fit(X_train, y_train)
        print(f"{C:>6} {gamma:>8}   {model.score(X_train, y_train):.3f}      "
              f"{model.score(X_test, y_test):.3f}")

# Look for the classic overfit signature at high C + high gamma: train accuracy near
# 1.0 while test accuracy drops. In practice, search C and gamma with GridSearchCV.


   C     gamma   train_acc  test_acc
   0.1      0.1   0.832      0.858
   0.1        1   0.886      0.892
   0.1       10   0.907      0.900
     1      0.1   0.871      0.858
     1        1   0.900      0.908
     1       10   0.925      0.917
   100      0.1   0.886      0.900
   100        1   0.925      0.900
   100       10   0.954      0.858


### Checklist item: Scaling requirement

**Approach:**
- Protects representation and model fit: the margin and kernel computations are distance and dot-product based, so a feature with a larger numeric range will dominate the margin calculation regardless of its actual importance.
- Why this is the mathematically right approach: because both the margin calculation and the RBF kernel are built from squared Euclidean distances, summing the squared difference per feature, a feature with a numerically larger range contributes a proportionally larger term to that sum regardless of its real relevance - standardizing removes this scale-dependence from the underlying distance math entirely.
- For this checklist item: SVM distance depends on Euclidean distance; without scaling, a feature with large numeric range (e.g., income) will dominate the margin calculation.
- Code walkthrough: Check that the 'without scaling' test accuracy is clearly worse than the pipeline version's - feature 1's artificially huge scale should be dominating the RBF distance calculation until the scaler fixes it.

**Learn more:**
- Website: [scikit-learn: Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Support+Vector+Machines+Scaling+requirement+machine+learning+theory)

**Trade-offs:**
- SVM shares KNN's sensitivity to feature scale, since both rely on distances, which is one reason tree-based models are often reached for first on messy, unscaled tabular data instead.
- Standardize features before fitting; skipping this is a common silent failure mode where SVM appears to underperform for no obvious reason.

**Practical software engineering use cases:**
- When to use it: Use a Pipeline(StandardScaler, SVC) as the default pattern any time you build an SVM, so scaling is never accidentally skipped.
- When not to use it: Don't fit a scaler on the full dataset before splitting - fit it inside the pipeline on the training fold only, or you'll leak test-set statistics.


In [176]:
# Support Vector Machines - Scaling requirement (using scikit-learn)

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# SVMs work on distances/inner products, so a feature with a huge numeric range
# dominates the geometry - same problem as KNN. ALWAYS scale features for SVMs.
# Best practice: put the scaler and the SVM in one Pipeline so scaling is applied
# consistently and never leaks test-set statistics into training.

X, y = make_classification(n_samples=400, n_features=5, n_informative=3,
                           random_state=8)
X = X * np.array([1, 1000, 1, 1, 1])                 # blow up feature 1's scale on purpose

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=8)

# Model 1: raw features - the oversized feature 1 dominates the RBF distances.
raw_svm = SVC(kernel="rbf").fit(X_train, y_train)
print("test accuracy WITHOUT scaling:", round(raw_svm.score(X_test, y_test), 3))

# Model 2: pipeline = StandardScaler THEN SVC.
# fit(): learns scaling from X_train only, then trains the SVM on scaled data.
# score()/predict(): automatically apply the SAME scaling to new data first.
scaled_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
scaled_svm.fit(X_train, y_train)
print("test accuracy WITH scaling:   ", round(scaled_svm.score(X_test, y_test), 3))


test accuracy WITHOUT scaling: 0.6
test accuracy WITH scaling:    0.833


### Checklist item: When SVM is useful

**Approach:**
- Protects deployment reliability: knowing SVM's sweet spot, namely high-dimensional, small-to-medium sample sizes with a clear margin between classes such as text classification, prevents defaulting to it where a tree ensemble would be both faster and more accurate.
- Why this is the mathematically right approach: SVM's training optimization scales roughly quadratically to cubically with the number of samples, since it involves pairwise relationships between points via the kernel, but its complexity is largely independent of feature count - that specific mathematical scaling is why SVM excels in the high-dimension, modest-sample-size regime and struggles as sample size grows.
- For this checklist item: SVM excels on high-dimensional sparse data (text) and small datasets; it struggles at scale (>100K rows) â€” gradient boosting is preferred then.
- Code walkthrough: Check that a linear kernel still reaches solid test accuracy even with far more features than training rows - that features-greater-than-samples regime is exactly where SVMs are known to do well.

**Learn more:**
- Website: [scikit-learn: Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Support+Vector+Machines+When+SVM+is+useful+machine+learning+theory)

**Trade-offs:**
- In the features-greater-than-samples regime shown here, SVM can outperform a tree ensemble, which tends to need more rows than columns to find reliable splits - but that advantage flips once the dataset grows into the hundreds of thousands of rows.
- SVM training scales poorly, roughly quadratic to cubic in sample size for kernel methods, making it a poor choice once you have hundreds of thousands of rows, even if accuracy would be competitive.

**Practical software engineering use cases:**
- When to use it: Use SVM specifically in the features-greater-than-samples regime, like bioinformatics or short-document text classification.
- When not to use it: Don't use it once your production dataset grows into the hundreds of thousands of rows - training time becomes impractical well before a tree ensemble's would.


In [177]:
# Support Vector Machines - When SVM is useful (using scikit-learn)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# GOOD FIT for SVMs:
#   - small/medium datasets (hundreds to ~tens of thousands of rows)
#   - MANY features relative to samples (text classification, bioinformatics)
#   - a clear margin exists between classes
# POOR FIT:
#   - very large datasets (training is roughly quadratic in sample count)
#   - you need probability estimates (only via slow extra calibration)
#   - heavy class overlap/noise with no real margin

# Demo of the sweet spot: 100 samples but 500 features (features >> samples).
X, y = make_classification(n_samples=100, n_features=500, n_informative=30,
                           random_state=12)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12)

# Linear kernel is the right choice in high dimensions (data is likely separable there,
# and it is much faster than rbf when the feature count is large).
model = make_pipeline(StandardScaler(), SVC(kernel="linear"))
model.fit(X_train, y_train)

print(f"samples={X.shape[0]}, features={X.shape[1]} (features >> samples)")
print("linear SVM test accuracy:", round(model.score(X_test, y_test), 3))
print()
print("For datasets with millions of rows, prefer gradient boosting or linear models.")


samples=100, features=500 (features >> samples)
linear SVM test accuracy: 0.567

For datasets with millions of rows, prefer gradient boosting or linear models.


In [178]:
# Practice: Support Vector Machines

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## K-Means

### Study checklist
- [ ] Centroids
- [ ] Choosing K
- [ ] Elbow method
- [ ] Silhouette score
- [ ] Scaling requirement
- [ ] Limitations

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Centroids

**Approach:**
- Protects model fit: centroids are the parameters K-Means actually learns, the mean position of each cluster, so understanding them is understanding what the algorithm optimizes, namely minimizing within-cluster squared distance to the centroid.
- Why this is the mathematically right approach: K-Means minimizes the sum of squared distances from each point to its assigned cluster's center; calculus shows this sum is minimized, for a fixed set of assignments, exactly when the center equals the arithmetic mean of the assigned points - the centroid update rule is the exact mathematical minimizer of that objective, not a heuristic.
- For this checklist item: A centroid is the coordinate-wise mean of the points assigned to a cluster; K-Means repeatedly recomputes centroids after assigning points to the nearest one.
- Code walkthrough: Check that each printed centroid is simply the coordinate-wise average of its cluster's points - that hand computation is literally what KMeans does internally for every cluster, every iteration.

**Learn more:**
- Website: [scikit-learn: K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Means+Centroids+machine+learning+theory)

**Trade-offs:**
- Unlike KNN or hierarchical clustering, K-Means never stores or compares individual points after training, only the centroids, which makes prediction on new points extremely cheap but throws away information about a cluster's actual shape.
- Because centroids are means, K-Means implicitly assumes roughly spherical, similarly-sized clusters, so it will place a centroid in a low-density gap between two real clusters if the true cluster shapes are elongated or unequal in size.

**Practical software engineering use cases:**
- When to use it: Use K-Means when you need fast, cheap-to-recompute group centers for something like customer segmentation on roughly round, equal-sized clusters.
- When not to use it: Don't use it when you need to preserve information about individual points post-clustering - only centroids are kept, not the shape of each group.


In [179]:
# K-Means - Centroids
clusters = {
    "cluster_a": [(1, 2), (2, 1), (2, 2)],
    "cluster_b": [(8, 8), (9, 8), (8, 9)],
}
for name, points in clusters.items():
    centroid = tuple(sum(p[i] for p in points) / len(points) for i in range(2))
    print(name, "centroid=", tuple(round(v, 2) for v in centroid))


cluster_a centroid= (1.67, 1.67)
cluster_b centroid= (8.33, 8.33)


### Checklist item: Choosing K

**Approach:**
- Protects model fit and evaluation integrity: unlike supervised learning, there's no ground-truth label to validate against, so K itself is a hyperparameter you must justify with unsupervised metrics or domain knowledge.
- Why this is the mathematically right approach: inertia, the sum of squared distances to centroids, is mathematically guaranteed to never increase as K grows, reaching zero when K equals the number of points - since the objective alone can't identify a 'correct' K, choosing it requires an external criterion, like the elbow method, silhouette score, or domain knowledge, layered on top of the pure optimization.
- For this checklist item: Fit K-Means with several K values and compare inertia or validation criteria; increasing K lowers inertia, so choose a value that balances compact clusters with simplicity.
- Code walkthrough: Check the size of the inertia drop from k=1 to k=2 versus k=2 to k=3 - a big early drop followed by small ones is the pattern you're looking for when picking K.

**Learn more:**
- Website: [scikit-learn: K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Means+Choosing+K+machine+learning+theory)

**Trade-offs:**
- K-Means requires committing to a cluster count up front, unlike DBSCAN which infers the number of clusters from density - simpler to reason about, but it pushes the burden of choosing K entirely onto you.
- The elbow method and silhouette score often disagree or give ambiguous answers on real data, so treat K as a business or domain decision informed by these metrics, not one they definitively answer.

**Practical software engineering use cases:**
- When to use it: Use this as a required step before every K-Means run - never ship a clustering model with an arbitrarily chosen K.
- When not to use it: Don't let a single metric like inertia pick K for you automatically - combine it with silhouette score and domain judgment before finalizing.


In [180]:
# K-Means - Choosing K
inertia_by_k = {1: 120.0, 2: 38.0, 3: 31.0, 4: 29.0}
for k, inertia in inertia_by_k.items():
    print(f"k={k} inertia={inertia}")
print("large gain from k=1 to k=2; smaller gains after that")


k=1 inertia=120.0
k=2 inertia=38.0
k=3 inertia=31.0
k=4 inertia=29.0
large gain from k=1 to k=2; smaller gains after that


### Checklist item: Elbow method

**Approach:**
- Protects evaluation integrity: plotting within-cluster sum of squares against K gives a visual heuristic for the point of diminishing returns from adding more clusters.
- Why this is the mathematically right approach: plotting inertia against K and looking for a bend approximates finding where the marginal reduction in the sum-of-squares objective from adding another cluster starts diminishing sharply - a rough, visual stand-in for a formal cost-benefit trade-off between model fit and model complexity.
- For this checklist item: Plot inertia across K values and look for the point where additional clusters give much smaller improvement; that bend is the elbow heuristic.
- Code walkthrough: Check that the drop from k=2 is clearly the largest single decrease in the printed list - that's what makes k=2 the elbow candidate rather than a later k.

**Learn more:**
- Website: [scikit-learn: K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Means+Elbow+method+machine+learning+theory)

**Trade-offs:**
- The elbow method costs nothing beyond re-running K-Means at a few different K values, far cheaper than silhouette score's pairwise-distance computation, but it trades that cheapness for a visual judgment call instead of a single number.
- The elbow is often subjective and not sharply defined on real, noisy data, making this method a rough guide rather than a precise answer, so pair it with silhouette score or domain constraints.

**Practical software engineering use cases:**
- When to use it: Use the elbow method as a fast, first-pass sanity check on a plausible range for K before deeper analysis.
- When not to use it: Don't treat the elbow as a strict, unambiguous answer on noisy real-world data - it often shows no sharp bend at all.


In [181]:
# K-Means - Elbow method
inertia = [120.0, 38.0, 31.0, 29.0]
for k in range(2, len(inertia) + 1):
    drop = inertia[k - 2] - inertia[k - 1]
    print(f"adding cluster {k}: inertia drop={drop}")
print("elbow candidate: k=2")


adding cluster 2: inertia drop=82.0
adding cluster 3: inertia drop=7.0
adding cluster 4: inertia drop=2.0
elbow candidate: k=2


### Checklist item: Silhouette score

**Approach:**
- Protects evaluation integrity: silhouette score quantifies both cohesion, how close points are to their own cluster, and separation, how far from other clusters, giving a more principled single-number metric than the elbow method alone.
- Why this is the mathematically right approach: for each point, the silhouette score is (b-a)/max(a,b), where a is the mean distance to points in its own cluster and b is the mean distance to the nearest other cluster - this formula is bounded between -1 and 1 by construction, giving a normalized, comparable measure of both cohesion and separation in one number.
- For this checklist item: Silhouette score measures how much closer a point is to its own cluster vs the nearest other cluster; values near 1 indicate well-separated clusters.
- Code walkthrough: Check that b (nearest other cluster distance) is much larger than a (own-cluster distance) here, which is why the computed silhouette comes out close to 1 - a well-clustered point.

**Learn more:**
- Website: [scikit-learn: K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Means+Silhouette+score+machine+learning+theory)

**Trade-offs:**
- Silhouette score gives a single number you can compare across different K values, unlike the elbow method's visual-only judgment, but computing all pairwise distances makes it considerably more expensive on large datasets.
- It's computationally expensive on large datasets due to pairwise distances and, like the elbow method, still assumes convex or spherical clusters, so it can be misleadingly low for legitimately non-spherical clusters.

**Practical software engineering use cases:**
- When to use it: Use silhouette score when you need a single comparable number across multiple candidate K values, not just a visual heuristic.
- When not to use it: Don't compute it on a very large dataset without sampling first - the pairwise distance computation can become a real bottleneck.


In [182]:
# K-Means - Silhouette score
point = (2, 2)
same_cluster_distances = [1.0, 1.4]
nearest_other_cluster_distances = [8.0, 8.5, 9.0]
a = sum(same_cluster_distances) / len(same_cluster_distances)
b = sum(nearest_other_cluster_distances) / len(nearest_other_cluster_distances)
silhouette = (b - a) / max(a, b)
print("a intra-cluster distance:", round(a, 2))
print("b nearest-cluster distance:", round(b, 2))
print("silhouette:", round(silhouette, 3))


a intra-cluster distance: 1.2
b nearest-cluster distance: 8.5
silhouette: 0.859


### Checklist item: Scaling requirement

**Approach:**
- Protects representation: K-Means clusters based on Euclidean distance to centroids, so an unscaled feature with a larger numeric range will dominate cluster assignment regardless of its real-world importance.
- Why this is the mathematically right approach: because cluster assignment is based purely on squared Euclidean distance to each centroid, exactly the same mathematical sensitivity to feature scale that affects KNN and SVM applies here - a feature's raw numeric range directly determines how much it contributes to that squared-distance sum.
- For this checklist item: K-Means is sensitive to feature scale because it uses Euclidean distance; normalize or standardize all features before clustering.
- Code walkthrough: Check that 'agreement with true groups WITHOUT scaling' is much lower than 'WITH scaling' - income's huge raw scale should be drowning out the actual age-based grouping until scaling fixes it.

**Learn more:**
- Website: [scikit-learn: K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Means+Scaling+requirement+machine+learning+theory)

**Trade-offs:**
- K-Means shares KNN and SVM's dependence on raw Euclidean distance, so it inherits the exact same scaling requirement - a tree-based clustering approach would be unaffected by a feature's raw numeric range.
- Always standardize features before clustering; forgetting this silently produces clusters that mostly reflect whichever feature happens to have the largest scale.

**Practical software engineering use cases:**
- When to use it: Use standardized features every time you cluster with K-Means, as a non-negotiable preprocessing step.
- When not to use it: Don't cluster raw mixed-unit features, like age and income together, without scaling - the largest-scale feature will silently dominate the clustering.


In [183]:
# K-Means - Scaling requirement (using scikit-learn)

import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# K-Means assigns points by EUCLIDEAN DISTANCE, so (like KNN and SVM) a feature
# with a large numeric range dominates the clustering. Scale features first.

rng = np.random.default_rng(9)

# Two TRUE groups that differ only in age; income is random noise on a huge scale.
young = np.column_stack([rng.normal(25, 3, 50),      # ages around 25
                         rng.normal(60000, 15000, 50)])   # income: big numbers, no pattern
old = np.column_stack([rng.normal(55, 3, 50),        # ages around 55
                       rng.normal(60000, 15000, 50)])
X = np.vstack([young, old])                          # rows 0-49 young, 50-99 old
true_group = np.array([0] * 50 + [1] * 50)           # ground truth for checking

def agreement(labels, truth):
    """Fraction of points where clustering matches the true grouping.
    Cluster numbering is arbitrary, so also check the flipped labeling."""
    direct = (labels == truth).mean()
    flipped = (labels == 1 - truth).mean()
    return max(direct, flipped)

# WITHOUT scaling: income's scale (tens of thousands) drowns out age (tens).
raw_labels = KMeans(n_clusters=2, n_init=10, random_state=9).fit_predict(X)
print("agreement with true groups WITHOUT scaling:", round(agreement(raw_labels, true_group), 2))

# WITH scaling: both features contribute equally -> age pattern is found.
X_scaled = StandardScaler().fit_transform(X)         # mean 0, std 1 per column
scaled_labels = KMeans(n_clusters=2, n_init=10, random_state=9).fit_predict(X_scaled)
print("agreement with true groups WITH scaling:   ", round(agreement(scaled_labels, true_group), 2))


agreement with true groups WITHOUT scaling: 0.54
agreement with true groups WITH scaling:    1.0


### Checklist item: Limitations

**Approach:**
- Protects deployment reliability: knowing K-Means assumes spherical, similarly-sized, similarly-dense clusters and requires choosing K up front prevents applying it to data with irregular cluster shapes or an unknown cluster count.
- Why this is the mathematically right approach: because centroids are means and clusters are assigned by nearest centroid, K-Means' implicit mathematical model is a set of spherical, equal-variance Gaussian-like blobs - clusters that are elongated, unequal in size, or non-convex simply don't match the geometry this objective function is built to find, regardless of how many times it's re-run.
- For this checklist item: Scale numeric features, choose a distance measure, fit clusters, then evaluate whether groups are stable and actionable.
- Code walkthrough: Check which two points get assigned to each centroid - the assignment follows simple nearest-centroid distance, which is exactly why K-Means struggles once real clusters aren't compact and round like this toy example.

**Learn more:**
- Website: [scikit-learn: K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=K-Means+Limitations+machine+learning+theory)

**Trade-offs:**
- K-Means is dramatically faster than density-based alternatives like DBSCAN on large datasets, but that speed comes from an algorithm that can only ever produce convex, roughly equal-sized partitions, no matter how the real clusters are shaped.
- It's fast and simple, but for non-spherical clusters, varying density, or unknown K, algorithms like DBSCAN or Gaussian Mixture Models are worth the added complexity.

**Practical software engineering use cases:**
- When to use it: Use K-Means when your clusters are plausibly round and similarly sized, such as segmenting customers by a handful of numeric behavior scores.
- When not to use it: Don't use it for clusters with very different densities or elongated shapes, like geographic hotspots - consider DBSCAN or a Gaussian Mixture Model instead.


In [184]:
# K-Means - Limitations
points = [(0, 0), (0, 4), (4, 0), (4, 4)]
centroids = [(0, 2), (4, 2)]
assignments = {centroid: [] for centroid in centroids}
for point in points:
    nearest = min(centroids, key=lambda c: (point[0] - c[0]) ** 2 + (point[1] - c[1]) ** 2)
    assignments[nearest].append(point)
print(assignments)
print("K-means prefers compact round clusters, so awkward shapes or outliers can distort assignments.")


{(0, 2): [(0, 0), (0, 4)], (4, 2): [(4, 0), (4, 4)]}
K-means prefers compact round clusters, so awkward shapes or outliers can distort assignments.


In [185]:
# Practice: K-Means

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## PCA

### Study checklist
- [ ] Dimensionality reduction
- [ ] Variance explained
- [ ] Principal components
- [ ] Scaling requirement
- [ ] Using PCA before ML models
- [ ] Interpreting components

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Dimensionality reduction

**Approach:**
- Protects representation and downstream model efficiency: PCA compresses correlated features into fewer uncorrelated components, reducing noise and computation for models trained afterward.
- Why this is the mathematically right approach: PCA seeks the linear projection that keeps the maximum possible variance of the original data - the Rayleigh quotient shows this maximum-variance direction is exactly the eigenvector of the data's covariance matrix with the largest eigenvalue, so the 'best' direction to project onto is a provable, closed-form answer, not a searched-for approximation.
- For this checklist item: PCA projects data onto the directions of maximum variance, reducing #features while retaining most information â€” useful before KNN, SVM, or visualization.
- Code walkthrough: Check that X_reduced.shape is (150, 2) instead of the original (150, 4), and that the printed variance-kept percentage is high - that's the compression-vs-information trade PCA is making.

**Learn more:**
- Website: [scikit-learn: Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/decomposition.html#principal-component-analysis-pca)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=PCA+Dimensionality+reduction+machine+learning+theory)

**Trade-offs:**
- PCA finds a lower-dimensional representation without needing any labels, unlike supervised feature selection, which makes it broadly reusable across tasks - but also means it optimizes for variance, not for whatever you'll eventually predict.
- Reduction is lossy by design, so the number of components you keep is a direct trade-off between compression, meaning speed and noise reduction, and information retained, meaning predictive power.

**Practical software engineering use cases:**
- When to use it: Use PCA when you have many correlated numeric features and want to speed up or denoise a downstream model.
- When not to use it: Don't use it when you specifically need to keep every original feature interpretable for a report or regulatory requirement - components mix features together.


In [186]:
# PCA - Dimensionality reduction (using scikit-learn)

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# PCA (Principal Component Analysis) compresses many correlated features into a
# few new features ("components") that keep as much of the data's VARIANCE
# (spread/information) as possible. It is unsupervised - labels are never used.

# Step 1: Load iris - 150 flowers, 4 measured features.
X, y = load_iris(return_X_y=True)
print("original shape:", X.shape)                    # (150, 4) = 150 rows, 4 features

# Step 2: Standardize first (PCA is variance-based; see the scaling cell below).
X_scaled = StandardScaler().fit_transform(X)

# Step 3: Ask PCA for 2 components - i.e., compress 4 dimensions down to 2.
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_scaled)              # fit finds directions; transform projects
print("reduced shape: ", X_reduced.shape)            # (150, 2)

# Step 4: How much information survived the 4 -> 2 compression?
kept = pca.explained_variance_ratio_.sum()           # fraction of total variance kept
print(f"variance kept by 2 components: {kept:.1%}")

# Step 5: The new columns are combinations of ALL original features (not a selection!).
print("\nfirst 3 flowers in the new 2D space:")
print(X_reduced[:3].round(2))


original shape: (150, 4)
reduced shape:  (150, 2)
variance kept by 2 components: 95.8%

first 3 flowers in the new 2D space:
[[-2.26  0.48]
 [-2.08 -0.67]
 [-2.36 -0.34]]


### Checklist item: Variance explained

**Approach:**
- Protects representation and evaluation integrity: it's the primary knob for deciding how many components to keep without silently discarding real signal.
- Why this is the mathematically right approach: the total variance in a dataset equals the sum of all the covariance matrix's eigenvalues, so each component's eigenvalue divided by that total sum gives exactly the fraction of variance that direction accounts for - explained variance ratio is a precise mathematical quantity, not an estimated one.
- For this checklist item: Each principal component explains a fraction of total variance; plot a scree plot to choose the number of components that capture ~90-95% of variance.
- Code walkthrough: Check that the cumulative explained variance list ends at 1.0 (all variance accounted for) and note how quickly it climbs - most of it should already be captured by the first component alone.

**Learn more:**
- Website: [scikit-learn: Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/decomposition.html#principal-component-analysis-pca)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=PCA+Variance+explained+machine+learning+theory)

**Trade-offs:**
- Explained variance gives PCA an objective, quantifiable stopping rule for how many components to keep, a clearer signal than the more subjective elbow-method judgment call used for choosing K in K-Means.
- Too few components lose predictive information and too many defeat the point of dimensionality reduction, so use a cumulative explained-variance threshold, for example 90 to 95 percent, rather than a fixed component count, and re-check it whenever input features change.

**Practical software engineering use cases:**
- When to use it: Use a cumulative variance threshold (like 90-95%) as your rule for picking the number of components, applied consistently across retraining.
- When not to use it: Don't pick a component count once and never revisit it - re-check the threshold whenever the input feature set changes meaningfully.


In [187]:
# PCA - Variance explained
explained_variance = [4.2, 0.8, 0.2]
total = sum(explained_variance)
ratios = [value / total for value in explained_variance]
cumulative = []
running = 0
for ratio in ratios:
    running += ratio
    cumulative.append(running)
print("explained variance ratio:", [round(r, 3) for r in ratios])
print("cumulative explained variance:", [round(r, 3) for r in cumulative])


explained variance ratio: [0.808, 0.154, 0.038]
cumulative explained variance: [0.808, 0.962, 1.0]


### Checklist item: Principal components

**Approach:**
- Protects representation: components are orthogonal directions of maximum variance, ordered by how much variance each explains, which is why the first few usually capture most of the useful structure.
- Why this is the mathematically right approach: the eigenvectors of a real, symmetric matrix, which a covariance matrix always is, are mathematically guaranteed to be orthogonal to each other - that's exactly why principal components always come out at right angles to one another, a consequence of linear algebra, not a design choice PCA makes.
- For this checklist item: Principal components are the eigenvectors of the covariance matrix sorted by eigenvalue; they are orthogonal and form a new coordinate system.
- Code walkthrough: Check that PC1 comes out close to [0.71, 0.71] (the diagonal direction) and that dot(PC1, PC2) prints as essentially 0 - components are always exactly perpendicular by construction.

**Learn more:**
- Website: [scikit-learn: Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/decomposition.html#principal-component-analysis-pca)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=PCA+Principal+components+machine+learning+theory)

**Trade-offs:**
- Because components are mathematically guaranteed to be orthogonal, PCA avoids the redundancy that can creep into hand-engineered features, but that guarantee is also why a component rarely maps onto a single, nameable real-world concept.
- Components are linear combinations of all original features, so they trade interpretability for compactness; a stakeholder can't read "component 2" the way they can read a named feature.

**Practical software engineering use cases:**
- When to use it: Use PCA when you want a guaranteed-uncorrelated set of features for a model sensitive to multicollinearity, like linear regression.
- When not to use it: Don't expect a specific principal component to correspond to a specific real-world concept without inspecting its loadings first - that mapping isn't automatic.


In [188]:
# PCA - Principal components (using scikit-learn)

import numpy as np
from sklearn.decomposition import PCA

# A principal component is a DIRECTION in feature space:
#   PC1 = the direction along which the data varies the most
#   PC2 = the next-most-varying direction, at 90 degrees to PC1
#   ... and so on. Components are always perpendicular to each other.

# Build 2D data that is strongly stretched along the diagonal (y roughly = x).
rng = np.random.default_rng(3)
x1 = rng.normal(0, 2, 200)                           # spread of 2 along a hidden axis
x2 = x1 + rng.normal(0, 0.4, 200)                    # x2 follows x1 with small noise
X = np.column_stack([x1, x2])                        # correlated 2-feature dataset

pca = PCA(n_components=2).fit(X)

# components_ holds one unit-length direction vector per row.
print("PC1 direction:", np.round(pca.components_[0], 2),
      " <- ~[0.71, 0.71], i.e., the diagonal, as expected")
print("PC2 direction:", np.round(pca.components_[1], 2),
      " <- perpendicular to PC1")

# Confirm perpendicularity: the dot product of the two directions is 0.
dot = np.dot(pca.components_[0], pca.components_[1])
print("dot(PC1, PC2) =", round(dot, 10), " (0 means exactly 90 degrees apart)")

# And PC1 should carry nearly all the variance of this stretched cloud:
print("variance explained:", np.round(pca.explained_variance_ratio_, 3))


PC1 direction: [0.7  0.72]  <- ~[0.71, 0.71], i.e., the diagonal, as expected
PC2 direction: [ 0.72 -0.7 ]  <- perpendicular to PC1
dot(PC1, PC2) = 0.0  (0 means exactly 90 degrees apart)
variance explained: [0.991 0.009]


### Checklist item: Scaling requirement

**Approach:**
- Protects representation integrity: PCA finds directions of maximum variance, so an unscaled feature with larger raw units, such as income in dollars vs. age in years, will dominate the components regardless of actual importance.
- Why this is the mathematically right approach: variance is measured in squared units, so a feature recorded in units with naturally larger numbers, like grams versus kilograms, will have a numerically larger variance for the exact same underlying quantity - since PCA finds directions of maximum variance, standardizing first, forcing every feature to variance 1, is what makes the resulting components reflect real structure rather than arbitrary unit choices.
- For this checklist item: PCA computes Euclidean distances in feature space; without standardization, high-variance features dominate all components.
- Code walkthrough: Check that the unscaled PC1 direction is nearly [0, 1] (pointing entirely along the huge-numbered weight column) while the scaled version balances both features - PCA being fooled by units, then corrected.

**Learn more:**
- Website: [scikit-learn: Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/decomposition.html#principal-component-analysis-pca)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=PCA+Scaling+requirement+machine+learning+theory)

**Trade-offs:**
- PCA's sensitivity to feature scale mirrors KNN's and SVM's, since all three rely on distance or variance calculations - but PCA's failure mode is quieter, since it will still run and report a result, just the wrong one.
- Always standardize before PCA unless all features are already on comparable, meaningful scales such as pixel intensities; skipping this makes the components reflect units, not real signal.

**Practical software engineering use cases:**
- When to use it: Use standardized features before PCA any time your inputs are on different units, such as dollars vs. years vs. counts.
- When not to use it: Don't skip scaling just because PCA runs without an error - it will happily return a result that's dominated by whichever feature has the largest raw numbers.


In [189]:
# PCA - Scaling requirement (using scikit-learn)

import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# PCA hunts for directions of maximum VARIANCE. Variance depends on units:
# a feature measured in grams has ~1,000,000x the variance of the same feature
# in kilograms! Without scaling, PCA just points at whichever feature has the
# biggest numbers. Standardize first so every feature starts with variance 1.

rng = np.random.default_rng(5)
n = 200
height_m = rng.normal(1.7, 0.1, n)                   # height in METERS (tiny variance)
weight_g = rng.normal(70000, 10000, n)               # weight in GRAMS (huge variance)
X = np.column_stack([height_m, weight_g])

# WITHOUT scaling: PC1 aligns almost perfectly with the raw weight column.
pca_raw = PCA(n_components=1).fit(X)
print("PC1 direction WITHOUT scaling:", np.round(pca_raw.components_[0], 4))
print("  -> [~0, ~1]: PCA 'discovered' that grams have big numbers. Useless.")

# WITH scaling: both features contribute on equal footing.
X_scaled = StandardScaler().fit_transform(X)         # each column -> mean 0, variance 1
pca_scaled = PCA(n_components=1).fit(X_scaled)
print("PC1 direction WITH scaling:   ", np.round(pca_scaled.components_[0], 4))
print("  -> balanced weights: now PCA reflects actual structure, not units.")


PC1 direction WITHOUT scaling: [0. 1.]
  -> [~0, ~1]: PCA 'discovered' that grams have big numbers. Useless.
PC1 direction WITH scaling:    [0.7071 0.7071]
  -> balanced weights: now PCA reflects actual structure, not units.


### Checklist item: Using PCA before ML models

**Approach:**
- Protects model fit and evaluation integrity: reducing noisy or correlated features before feeding a downstream model can improve generalization and speed, especially for distance-based or linear models.
- Why this is the mathematically right approach: because PCA's components are, by construction, uncorrelated linear combinations of the original features, feeding them into a model sensitive to multicollinearity, like linear regression, removes that specific mathematical problem at the source rather than needing separate collinearity diagnostics.
- For this checklist item: Apply PCA on training data, then transform validation and test with the same projection matrix; PCA reduces noise and collinearity before a linear model.
- Code walkthrough: Check that the 20-component pipeline's accuracy is nearly identical to using all 64 raw pixels - roughly a third of the features for almost the same accuracy is the PCA trade being illustrated.

**Learn more:**
- Website: [scikit-learn: Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/decomposition.html#principal-component-analysis-pca)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=PCA+Using+PCA+before+ML+models+machine+learning+theory)

**Trade-offs:**
- Reducing 64 pixels to 20 components sacrifices almost no logistic regression accuracy here, but that trade is model-dependent - tree-based models often see little benefit from PCA since they already ignore irrelevant or redundant features on their own.
- Fit PCA only on the training fold, never on the full dataset before splitting, to avoid leaking test-set variance into the transformation, and note that PCA can hurt tree-based models, which already handle correlated or irrelevant features natively.

**Practical software engineering use cases:**
- When to use it: Use PCA ahead of a distance-based or linear model (KNN, SVM, logistic regression) trained on a large number of correlated numeric features.
- When not to use it: Don't add PCA in front of a tree-based model expecting an accuracy boost - trees already handle correlated and irrelevant features natively, so PCA often adds complexity with little payoff.


In [190]:
# PCA - Using PCA before ML models (using scikit-learn)

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Why put PCA in front of a model?
#   - fewer features -> faster training, less memory
#   - removes redundancy among correlated features
#   - can reduce overfitting when features >> samples
# The digits dataset: 8x8 pixel images of handwritten digits = 64 pixel features.

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=0, stratify=y)

# Pipeline A: scale -> logistic regression on ALL 64 pixel features.
full_model = make_pipeline(StandardScaler(),
                           LogisticRegression(max_iter=2000))
full_model.fit(X_train, y_train)

# Pipeline B: scale -> PCA down to 20 components -> logistic regression.
# Inside a pipeline, PCA is fitted on TRAINING data only - no test-set leakage.
pca_model = make_pipeline(StandardScaler(),
                          PCA(n_components=20),      # 64 features -> 20 components
                          LogisticRegression(max_iter=2000))
pca_model.fit(X_train, y_train)

print("accuracy with all 64 features: ", round(full_model.score(X_test, y_test), 3))
print("accuracy with 20 PCA components:", round(pca_model.score(X_test, y_test), 3))
print("\n~1/3 of the features, nearly identical accuracy - that's the PCA trade.")


accuracy with all 64 features:  0.972
accuracy with 20 PCA components: 0.952

~1/3 of the features, nearly identical accuracy - that's the PCA trade.


### Checklist item: Interpreting components

**Approach:**
- Protects user value and debugging: examining the loadings, or weights, of original features on each component is how you translate an abstract component back into a human-readable story.
- Why this is the mathematically right approach: a component's loading for each original feature is literally that feature's coefficient in the eigenvector, a real number that can be positive, negative, or near zero regardless of the feature's 'importance' in a human sense - the loadings are an exact mathematical description of the linear combination, but that description doesn't have to correspond to any single, nameable real-world concept.
- For this checklist item: Each PC is a weighted linear combination of original features; the loadings (weights) reveal which original features drive each component.
- Code walkthrough: Check that petal length, petal width, and sepal length all load with the same sign on PC1 (supporting the 'overall flower size' reading in the comments) while PC2 is dominated by sepal width with the opposite sign.

**Learn more:**
- Website: [scikit-learn: Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/decomposition.html#principal-component-analysis-pca)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=PCA+Interpreting+components+machine+learning+theory)

**Trade-offs:**
- Reading loadings lets you recover some real-world meaning from an otherwise abstract component, a step that isn't necessary for a supervised model's coefficients, which already come pre-labeled with a feature name.
- Loadings can be ambiguous when many features contribute similarly, and component signs are arbitrary since PCA doesn't know which direction is positive, so don't over-interpret small loading differences as meaningful.

**Practical software engineering use cases:**
- When to use it: Use loadings inspection when you need to give a component a real-world name, like overall size, for a report or presentation.
- When not to use it: Don't over-interpret a small difference between two loadings as meaningful - component signs are arbitrary and small differences are often just noise.


In [191]:
# PCA - Interpreting components (using scikit-learn)

import numpy as np
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Each component is a weighted mix of the ORIGINAL features. The weights are
# called LOADINGS. Reading them tells you what real-world pattern a component
# captures - e.g., "PC1 is overall flower size".

iris = load_iris()
X_scaled = StandardScaler().fit_transform(iris.data) # standardize first, as always

pca = PCA(n_components=2).fit(X_scaled)

# Print the loading of every original feature on each component.
for pc_index in range(2):
    print(f"PC{pc_index + 1} loadings "
          f"({pca.explained_variance_ratio_[pc_index]:.0%} of variance):")
    for name, loading in zip(iris.feature_names, pca.components_[pc_index]):
        # abs(loading) = strength of contribution; sign = direction of the relationship
        print(f"  {name:<20} {loading:+.2f}")
    print()

# How to read the output:
#   PC1: petal length, petal width, and sepal length all load strongly with the SAME
#        sign -> they rise together -> PC1 is essentially "overall flower size".
#   PC2: dominated by sepal width with the opposite sign -> a shape contrast,
#        separating wide-sepal flowers from the rest.
# CAUTION: signs are arbitrary overall (a component can be flipped); what matters
# is the RELATIVE pattern of the loadings, not whether they are + or -.


PC1 loadings (73% of variance):
  sepal length (cm)    +0.52
  sepal width (cm)     -0.27
  petal length (cm)    +0.58
  petal width (cm)     +0.56

PC2 loadings (23% of variance):
  sepal length (cm)    +0.38
  sepal width (cm)     +0.92
  petal length (cm)    +0.02
  petal width (cm)     +0.07



In [192]:
# Practice: PCA

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



# Advanced Machine Learning


## Imbalanced Classification

### Study checklist
- [ ] Class imbalance problem
- [ ] Precision-recall tradeoff
- [ ] Class weights
- [ ] Oversampling and undersampling
- [ ] SMOTE concept
- [ ] Choosing PR-AUC over ROC-AUC when needed

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Class imbalance problem

**Approach:**
- Protects evaluation integrity: a model can reach high accuracy by simply predicting the majority class every time, so recognizing imbalance early prevents shipping a model that's silently useless on the class that actually matters.
- Why this is the mathematically right approach: accuracy is defined as (correct predictions)/(total predictions); when 95% of labels are one class, a model that ignores the features entirely and always predicts that class achieves 0.95 accuracy by the formula alone - this is a direct mathematical consequence of the accuracy definition under a skewed class distribution, not a flaw specific to any particular model.
- For this checklist item: When 95% of samples are negative, a model predicting all-negative gets 95% accuracy â€” accuracy is misleading; use precision, recall, F1, or PR-AUC.
- Code walkthrough: Check that the 'always predict 0' model reports 0.95 accuracy yet caught 0 of the 5 class-1 cases - that gap between a great-looking accuracy number and zero real recall is the entire problem in one printed line.

**Learn more:**
- Website: [imbalanced-learn User Guide](https://imbalanced-learn.org/stable/user_guide.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Imbalanced+Classification+Class+imbalance+problem+machine+learning+theory)

**Trade-offs:**
- Unlike a balanced classification problem where accuracy alone is a reasonable scorecard, an imbalanced one needs the evaluation strategy itself to change before any modeling fix like resampling or weighting can even be judged fairly.
- There's no single fix; the right combination of resampling, class weighting, and metric choice depends on how severe the imbalance is and the real-world cost of missing the minority class.

**Practical software engineering use cases:**
- When to use it: Use this awareness check the moment you see a classification target with a skewed split, before you even pick a metric.
- When not to use it: Don't report accuracy as your headline metric on a rare-event problem - it will look great while hiding a model that never finds the class you care about.


In [ ]:
# Imbalanced Classification - Class imbalance problem (numpy)

import numpy as np

# The problem: with a skewed class split, a model that just predicts the MAJORITY
# class every time can score high on accuracy while being useless.

actual = np.array([0] * 95 + [1] * 5)                # 95% class 0, 5% class 1 (rare event)
print("class balance:", {"class_0": int((actual == 0).sum()), "class_1": int((actual == 1).sum())})

# A "dumb" model that always predicts the majority class.
dumb_predictions = np.zeros_like(actual)
accuracy = (dumb_predictions == actual).mean()
print(f"\n'always predict 0' accuracy: {accuracy:.2f}  <- looks great, but...")
print(f"it caught {int((dumb_predictions[actual == 1] == 1).sum())} out of "
      f"{int((actual == 1).sum())} of the class we actually care about")

print()
print("Accuracy alone hides this failure completely - it's why imbalanced problems")
print("need different metrics (precision/recall, PR-AUC) covered in the next cells.")


### Checklist item: Precision-recall tradeoff

**Approach:**
- Protects evaluation integrity and user value: under imbalance, this trade-off, not accuracy, reflects what actually matters, namely how many false alarms vs. how many missed rare events the business can tolerate.
- Why this is the mathematically right approach: precision = TP/(TP+FP) and recall = TP/(TP+FN) share the same numerator but different denominators, so raising the classification threshold, fewer positive predictions, tends to decrease FP and TP together in a way that mathematically raises precision while lowering recall - the trade-off is baked into the two formulas' shared structure, not just an empirical pattern.
- For this checklist item: Raising the threshold increases precision but decreases recall; lower it to catch more positives at the cost of more false alarms â€” the trade-off depends on business cost.
- Code walkthrough: Check that precision rises and recall falls as the threshold climbs from 0.2 to 0.8 - find the threshold where the two cross to see the actual trade point for this dataset.

**Learn more:**
- Website: [imbalanced-learn User Guide](https://imbalanced-learn.org/stable/user_guide.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Imbalanced+Classification+Precision-recall+tradeoff+machine+learning+theory)

**Trade-offs:**
- This trade-off exists in every classifier in this notebook, but it becomes the central design decision under imbalance specifically, where the default 0.5 threshold almost never matches the real cost asymmetry between the two error types.
- Optimizing purely for recall usually tanks precision and vice versa, so set the operating point using the real cost of each error type, not a default threshold.

**Practical software engineering use cases:**
- When to use it: Use a precision-recall curve to pick your operating threshold whenever false positives and false negatives have different real costs, like fraud review vs. missed fraud.
- When not to use it: Don't optimize purely for recall without checking the resulting precision - catching everything by flagging everything isn't useful in an operational system with limited review capacity.


In [ ]:
# Imbalanced Classification - Precision-recall tradeoff (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

# Precision and recall move in OPPOSITE directions as the decision threshold changes:
#   lower threshold -> flag more positives -> recall UP, precision DOWN (more false alarms)
#   higher threshold -> flag fewer positives -> precision UP, recall DOWN (more misses)

rng = np.random.default_rng(0)
n = 300
X = rng.normal(size=(n, 2))
y = ((X[:, 0] + X[:, 1]) > 1.8).astype(int)          # rare positive class (~10%)
print("positive class rate:", round(y.mean(), 3))

model = LogisticRegression().fit(X, y)
proba = model.predict_proba(X)[:, 1]                 # P(class 1) for every row

print("threshold  precision  recall")
for threshold in [0.2, 0.4, 0.5, 0.6, 0.8]:
    preds = (proba >= threshold).astype(int)
    p = precision_score(y, preds, zero_division=0)
    r = recall_score(y, preds, zero_division=0)
    print(f"{threshold:<10} {p:.3f}      {r:.3f}")

# Reading the output: as the threshold rises, precision climbs and recall falls.
# The "right" threshold depends on which error (false alarm vs missed case) costs more.


### Checklist item: Class weights

**Approach:**
- Protects model fit: reweighting the loss function to penalize minority-class errors more heavily nudges the decision boundary without altering the training data itself.
- Why this is the mathematically right approach: multiplying the loss contribution of minority-class errors by a weight before summing or averaging the total loss directly changes what set of weights minimizes that total loss - it alters the actual optimization objective's math, not a post-hoc correction applied after training.
- For this checklist item: Pass class_weight='balanced' or {0:1, 1:10} to the model to penalize misclassifying the minority class more heavily during training.
- Code walkthrough: Check that the balanced model's recall is higher than the default model's, usually at some cost to precision - that shift is the loss reweighting doing its job without touching the training data.

**Learn more:**
- Website: [imbalanced-learn User Guide](https://imbalanced-learn.org/stable/user_guide.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Imbalanced+Classification+Class+weights+machine+learning+theory)

**Trade-offs:**
- Class weighting is essentially free computationally, since no new data is created and training time barely changes, unlike SMOTE or oversampling - which is why it's often the first fix to try before reaching for data-level techniques.
- It's simpler and leakage-safer than resampling since there are no synthetic or duplicated rows to accidentally leak across folds, but it doesn't help distance-based algorithms as directly as it helps loss-based ones like logistic regression or trees.

**Practical software engineering use cases:**
- When to use it: Use class_weight='balanced' as your first, cheapest fix for imbalance on a loss-based model like logistic regression, before reaching for resampling.
- When not to use it: Don't rely on class weighting alone for a distance-based model like KNN - it doesn't change how the algorithm computes neighbors at all.


In [ ]:
# Imbalanced Classification - Class weights (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score

# class_weight='balanced' reweights the loss so mistakes on the MINORITY class count
# more (inversely proportional to class frequency) - without touching the data itself.

rng = np.random.default_rng(1)
n = 400
X = rng.normal(size=(n, 2))
y = ((X[:, 0] + X[:, 1]) > 2.2).astype(int)          # rare positive class
print("positive class rate:", round(y.mean(), 3))

# Model 1: default (unweighted) - tends to favor the majority class.
plain_preds = LogisticRegression().fit(X, y).predict(X)

# Model 2: balanced weights - errors on the rare class are penalized more heavily.
weighted_preds = LogisticRegression(class_weight="balanced").fit(X, y).predict(X)

for name, preds in [("default              ", plain_preds),
                    ("class_weight=balanced", weighted_preds)]:
    print(f"{name}: precision={precision_score(y, preds, zero_division=0):.3f}  "
          f"recall={recall_score(y, preds, zero_division=0):.3f}")

# Expect: balanced weights raise recall on the minority class (catches more of them),
# usually at some cost to precision - the model's decision boundary shifts toward class 1.


### Checklist item: Oversampling and undersampling

**Approach:**
- Protects data quality and model fit: both aim to rebalance the training distribution so the model doesn't ignore the minority class, but they do it in opposite directions, duplicating or synthesizing minority examples vs. discarding majority ones.
- Why this is the mathematically right approach: a model's training objective is approximately minimized in expectation over the training distribution it sees; duplicating minority rows or removing majority rows directly changes that empirical distribution the optimizer averages over, shifting the effective loss function's minimum toward better minority-class performance.
- For this checklist item: Undersampling removes majority rows to balance classes (fast, loses data); oversampling duplicates minority rows (slower, risk of overfitting exact copies).
- Code walkthrough: Check that both oversampling and undersampling raise recall above the 'original' baseline, and compare their precision costs against each other - neither is free, but they pay in different currencies.

**Learn more:**
- Website: [imbalanced-learn User Guide](https://imbalanced-learn.org/stable/user_guide.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Imbalanced+Classification+Oversampling+and+undersampling+machine+learning+theory)

**Trade-offs:**
- Both techniques change the training data itself rather than the model or the loss function, which sets them apart from class weighting, and also means they can be combined with any classifier, not just loss-based ones.
- Oversampling risks overfitting to duplicated minority examples; undersampling risks throwing away useful majority-class information, so the right choice depends on how much data you can afford to lose or duplicate.

**Practical software engineering use cases:**
- When to use it: Use undersampling when you have abundant majority-class data and can afford to discard some of it for faster training; use oversampling when majority-class data is scarce too.
- When not to use it: Don't oversample before splitting into train/test - duplicated rows can end up on both sides of the split and leak information.


In [ ]:
# Imbalanced Classification - Oversampling and undersampling (numpy + scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score

rng = np.random.default_rng(2)
n_majority, n_minority = 380, 20                     # ~5% minority - a rare-event dataset
X_majority = rng.normal(0, 1, size=(n_majority, 2))
X_minority = rng.normal(2.5, 1, size=(n_minority, 2))
X = np.vstack([X_majority, X_minority])
y = np.array([0] * n_majority + [1] * n_minority)
print(f"original: {n_majority} majority rows, {n_minority} minority rows")

# Oversampling: duplicate minority rows (with replacement) until classes are balanced.
minority_idx = np.where(y == 1)[0]
oversample_idx = rng.choice(minority_idx, size=n_majority - n_minority, replace=True)
X_over = np.vstack([X, X[oversample_idx]])
y_over = np.concatenate([y, y[oversample_idx]])
print(f"after oversampling: {int((y_over == 0).sum())} class 0, {int((y_over == 1).sum())} class 1")

# Undersampling: randomly drop majority rows until classes are balanced.
majority_idx = np.where(y == 0)[0]
keep_majority = rng.choice(majority_idx, size=n_minority, replace=False)
keep_idx = np.concatenate([keep_majority, minority_idx])
X_under, y_under = X[keep_idx], y[keep_idx]
print(f"after undersampling: {int((y_under == 0).sum())} class 0, {int((y_under == 1).sum())} class 1")

for name, (Xr, yr) in [("original    ", (X, y)), ("oversampled ", (X_over, y_over)),
                       ("undersampled", (X_under, y_under))]:
    preds = LogisticRegression().fit(Xr, yr).predict(X)   # always score on the ORIGINAL data
    print(f"{name}: recall={recall_score(y, preds):.3f}  "
          f"precision={precision_score(y, preds, zero_division=0):.3f}")

# Oversampling keeps all original information but risks overfitting to repeated rows.
# Undersampling avoids duplicate rows but throws away majority-class data.


### Checklist item: SMOTE concept

**Approach:**
- Protects data quality and minority-class model fit: SMOTE exists to give the learner enough minority-class signal to find a real decision boundary instead of defaulting to always predicting the majority class.
- Why this is the mathematically right approach: linear interpolation between two points, a + fraction*(b-a) for a fraction between 0 and 1, is guaranteed by vector-space geometry to produce a new point lying exactly on the line segment between them - which is why SMOTE's synthetic points are plausible new examples in the same local region as real minority points, not points that could fall anywhere at random.
- For this checklist item: SMOTE generates synthetic minority samples by interpolating between existing ones; it adds diversity compared to simple oversampling and reduces overfitting risk.
- Code walkthrough: Check that the printed synthetic SMOTE points are genuinely new coordinates, not equal to any original row, while the 'plain oversampling' example at the bottom repeats existing rows exactly - that contrast is the whole SMOTE idea.

**Learn more:**
- Website: [imbalanced-learn User Guide](https://imbalanced-learn.org/stable/user_guide.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Imbalanced+Classification+SMOTE+concept+machine+learning+theory)

**Trade-offs:**
- SMOTE adds genuinely new synthetic points instead of just repeating existing minority rows, reducing the overfitting risk plain oversampling carries - at the cost of an interpolation step that can create unrealistic points if the minority class isn't smoothly distributed.
- Applying SMOTE before splitting train/test, or before CV folds, leaks synthetic neighbors of test points into training, so fit SMOTE only on the training fold and always validate against the untouched, naturally imbalanced data.

**Practical software engineering use cases:**
- When to use it: Use SMOTE when plain duplication (oversampling) is causing visible overfitting to repeated minority rows.
- When not to use it: Don't apply SMOTE before your train/test split or CV fold - interpolated neighbors of test points will leak into training and inflate your reported performance.


In [ ]:
# Imbalanced Classification - SMOTE concept (manual mini-SMOTE with numpy)

import numpy as np

# SMOTE = Synthetic Minority Oversampling. Instead of duplicating minority rows,
# it creates NEW synthetic points by interpolating between a minority point and
# one of its nearest minority neighbors.

rng = np.random.default_rng(3)
minority_points = np.array([[1.0, 1.0], [1.2, 0.9], [0.8, 1.3], [1.1, 1.1]])

def smote_point(points, rng):
    i, j = rng.choice(len(points), size=2, replace=False)   # a point and a random neighbor
    a, b = points[i], points[j]
    fraction = rng.uniform(0, 1)                     # how far along the line to interpolate
    return a + fraction * (b - a)                     # a brand-new point ON the line a->b

synthetic = np.array([smote_point(minority_points, rng) for _ in range(4)])
print("original minority points:\n", minority_points)
print("synthetic SMOTE points (interpolated, not copies):\n", np.round(synthetic, 2))

# Contrast with plain oversampling: those points would be exact duplicates in this list.
duplicates = minority_points[rng.integers(0, len(minority_points), size=4)]
print("\nplain oversampling would instead just repeat existing rows, e.g.:\n", duplicates)


### Checklist item: Choosing PR-AUC over ROC-AUC when needed

**Approach:**
- Protects evaluation integrity: ROC-AUC can look deceptively good under severe imbalance because it accounts for the huge number of true negatives, while PR-AUC focuses on the trade-off that actually matters for the rare class.
- Why this is the mathematically right approach: ROC-AUC's false-positive-rate axis, FP/(FP+TN), is divided by the very large count of true negatives under severe imbalance, so even a large absolute number of false positives becomes a tiny rate - PR-AUC's precision axis, TP/(TP+FP), has no true-negative term at all, which is exactly why it doesn't get mathematically diluted by an abundance of easy negatives the way ROC-AUC does.
- For this checklist item: When the positive class is rare (<5%), ROC-AUC looks inflated because TN counts dominate; PR-AUC focuses on precision and recall over the minority class only.
- Code walkthrough: Check how much higher ROC-AUC reads compared to PR-AUC on this very rare positive class - that gap is exactly why PR-AUC is the more honest metric when positives are scarce.

**Learn more:**
- Website: [imbalanced-learn User Guide](https://imbalanced-learn.org/stable/user_guide.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Imbalanced+Classification+Choosing+PR-AUC+over+ROC-AUC+when+needed+machine+learning+theory)

**Trade-offs:**
- PR-AUC and ROC-AUC are computed from the exact same predicted probabilities, so switching between them costs nothing computationally - the choice is purely about which one actually reflects performance on the class you care about.
- PR-AUC is the right primary metric when the positive class is rare and what you care about, but it's less interpretable to non-technical stakeholders than accuracy or a simple precision/recall pair at a chosen threshold.

**Practical software engineering use cases:**
- When to use it: Use PR-AUC as your primary offline metric when the positive class is both rare and the one that matters, such as fraud detection.
- When not to use it: Don't report ROC-AUC alone to a stakeholder deciding whether a rare-event model is good enough - a high ROC-AUC can mask genuinely poor performance on the class that matters.


In [ ]:
# Imbalanced Classification - Choosing PR-AUC over ROC-AUC when needed (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

rng = np.random.default_rng(4)
n = 2000
X = rng.normal(size=(n, 2))
y = ((X[:, 0] + X[:, 1]) > 3.0).astype(int)          # very rare positive class
print("positive class rate:", round(y.mean(), 4))    # roughly 1-2%

model = LogisticRegression().fit(X, y)
proba = model.predict_proba(X)[:, 1]

roc_auc = roc_auc_score(y, proba)                    # rewards ranking ALL pairs correctly
pr_auc = average_precision_score(y, proba)           # focuses on the rare positive class

print(f"ROC-AUC: {roc_auc:.3f}  <- inflated by the huge number of easy true negatives")
print(f"PR-AUC:  {pr_auc:.3f}  <- lower, and more honest about performance on the rare class")

# With severe imbalance, ROC-AUC can look great even when the model barely finds
# the positive class - PR-AUC is the metric that actually reflects that struggle.


In [199]:
# Practice: Imbalanced Classification

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Time Series Forecasting

### Study checklist
- [ ] Trend, seasonality, noise
- [ ] Stationarity
- [ ] Lag features
- [ ] Rolling statistics
- [ ] Train/test split for time series
- [ ] ARIMA / Prophet / ML-based forecasting

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Trend, seasonality, noise

**Approach:**
- Protects representation: decomposing a series into these components is what tells you which patterns are structural and worth modeling explicitly versus random and irreducible error.
- Why this is the mathematically right approach: the additive decomposition, value = trend + seasonality + noise, is a specific mathematical model of a time series; estimating trend as a rolling average and seasonality as the average deviation from that trend at each position in the cycle are the standard closed-form ways to estimate each additive term separately.
- For this checklist item: Decompose the series into trend (long-run direction), seasonality (repeating cycles), and noise (random residual) before choosing a modeling approach.
- Code walkthrough: Check that the estimated seasonal pattern repeats a shape similar to the sine wave used to build the data, and that the residual's standard deviation lands close to the injected noise level of 1.

**Learn more:**
- Website: [Forecasting: Principles and Practice](https://otexts.com/fpp3/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Time+Series+Forecasting+Trend%2C+seasonality%2C+noise+machine+learning+theory)

**Trade-offs:**
- Decomposing a series into these three components costs only a rolling-mean calculation, far cheaper than fitting a full forecasting model, but it's diagnostic rather than predictive - it explains the series without yet forecasting it.
- Real series often have multiple overlapping seasonalities or a trend that changes over time, which simple additive or multiplicative decomposition can misrepresent, so inspect the decomposition visually before trusting it.

**Practical software engineering use cases:**
- When to use it: Use decomposition early in any forecasting project to understand what structure actually exists before choosing a model family.
- When not to use it: Don't skip straight to modeling without decomposition on a series with visible seasonality - you risk picking a model or feature set that ignores a pattern you could have engineered for directly.


In [ ]:
# Time Series Forecasting - Trend, seasonality, noise (numpy)

import numpy as np

# A time series is often modeled as: value = trend + seasonality + noise.
#   trend      : the slow, long-term direction (rising/falling)
#   seasonality: a repeating pattern at a fixed period (e.g., every 12 months)
#   noise      : whatever is left over - the irreducible random part

rng = np.random.default_rng(6)
t = np.arange(36)
trend = 0.5 * t                                      # steadily rising
seasonality = 4 * np.sin(2 * np.pi * t / 12)         # repeats every 12 steps
noise = rng.normal(0, 1, 36)
series = 20 + trend + seasonality + noise

# Recover the trend with a rolling mean (smooths out the seasonal wiggle and noise).
window = 12
trend_estimate = np.array([series[max(0, i - window // 2):i + window // 2 + 1].mean()
                           for i in range(len(series))])

# Seasonality = original series minus the estimated trend, averaged by position-in-cycle.
detrended = series - trend_estimate
seasonal_estimate = np.array([detrended[i::12].mean() for i in range(12)])

# Noise = whatever is left after removing both trend and seasonality.
residual = series - trend_estimate - np.tile(seasonal_estimate, 3)

print("first 12 raw values:        ", np.round(series[:12], 1))
print("estimated trend (first 12): ", np.round(trend_estimate[:12], 1))
print("estimated seasonal pattern (12 values):", np.round(seasonal_estimate, 2))
print("residual std (should be close to noise's std=1):", round(residual.std(), 2))


### Checklist item: Stationarity

**Approach:**
- Protects model fit: many classical forecasting models like ARIMA assume the series' statistical properties, such as mean and variance, don't change over time, so checking stationarity determines whether that assumption holds.
- Why this is the mathematically right approach: ARIMA and related models are mathematically derived assuming the series' mean and variance are constant over time, the literal definition of a stationary process - differencing, replacing each value with the change from the previous one, is a specific operation proven to remove a linear trend, converting a common type of non-stationary series into a stationary one.
- For this checklist item: A stationary series has constant mean and variance over time; most forecasting models require stationarity â€” apply differencing or log-transform to achieve it.
- Code walkthrough: Check that the raw series' rolling mean keeps climbing (non-stationary) while the first-difference and log-difference series look much flatter - differencing is what's removing the trend.

**Learn more:**
- Website: [Forecasting: Principles and Practice](https://otexts.com/fpp3/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Time+Series+Forecasting+Stationarity+machine+learning+theory)

**Trade-offs:**
- Stationarity matters much more for classical models like ARIMA than for ML-based or Prophet-style approaches, which can often absorb trend and seasonality directly as features instead of requiring the series to be transformed first.
- Differencing to force stationarity is a standard fix but can remove genuine long-term structure, like trend, that a stakeholder actually cares about forecasting, so difference only as much as needed, checked via a statistical test like ADF.

**Practical software engineering use cases:**
- When to use it: Use a stationarity check before fitting any classical model like ARIMA that assumes constant statistical properties over time.
- When not to use it: Don't difference a series more than necessary just because a statistical test suggests it - over-differencing can remove real trend information a stakeholder needs forecasted.


In [201]:
# Time Series Forecasting - Stationarity
import math

sales = [10, 13, 18, 25, 36, 52]
log_sales = [math.log(v) for v in sales]
first_difference = [None] + [sales[i] - sales[i - 1] for i in range(1, len(sales))]
log_difference = [None] + [log_sales[i] - log_sales[i - 1] for i in range(1, len(log_sales))]

def rolling_mean(values, window):
    return [None] * (window - 1) + [sum(values[i - window + 1:i + 1]) / window for i in range(window - 1, len(values))]

print("sales:", sales)
print("rolling mean before differencing:", [None if v is None else round(v, 2) for v in rolling_mean(sales, 3)])
print("first difference:", first_difference)
print("log difference:", [None if v is None else round(v, 3) for v in log_difference])


sales: [10, 13, 18, 25, 36, 52]
rolling mean before differencing: [None, None, 13.67, 18.67, 26.33, 37.67]
first difference: [None, 3, 5, 7, 11, 16]
log difference: [None, 0.262, 0.325, 0.329, 0.365, 0.368]


### Checklist item: Lag features

**Approach:**
- Protects representation: lagged values turn a temporal problem into a standard supervised regression problem, letting you reuse ordinary ML models like trees and linear models for forecasting.
- Why this is the mathematically right approach: representing y_t as a function of y_(t-1), y_(t-2), and so on converts a sequential dependency into an ordinary supervised regression problem, y = f(lagged features) - this reformulation is mathematically valid specifically because it never uses information from time t or later as an input, preserving the causal ordering the forecast must respect.
- For this checklist item: Create lag_1, lag_7, lag_28 features so the model can learn from past values; the lag window size should match the problem's natural memory horizon.
- Code walkthrough: Check that lag_1's value at each row equals sales from the row before it, and lag_2 from two rows before - those shifted columns are what get fed into an ordinary regression model as features.

**Learn more:**
- Website: [Forecasting: Principles and Practice](https://otexts.com/fpp3/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Time+Series+Forecasting+Lag+features+machine+learning+theory)

**Trade-offs:**
- Turning a series into lag features is what lets you use any standard ML model for forecasting, a flexibility ARIMA's built-in autoregression doesn't need but also doesn't offer for incorporating other features.
- Choosing too few lags misses relevant history and too many adds noise and dimensionality, and every lag feature must be laggable at actual prediction time since you can't use a lag that requires future data.

**Practical software engineering use cases:**
- When to use it: Use lag features when you want to forecast with an ordinary ML model, like a tree or linear regression, instead of a dedicated time-series library.
- When not to use it: Don't include a lag that wouldn't actually be available at prediction time in production, such as yesterday's revenue when yesterday's books aren't closed yet - that's a leakage bug waiting to happen.


In [ ]:
# Time Series Forecasting - Lag features (numpy)

import numpy as np

# A lag feature = a past value of the series, reused as an input feature for "today's" row.
# lag_1 = yesterday's value, lag_2 = the value from two steps ago, etc.

sales = np.array([10, 12, 13, 15, 18, 22, 21, 25])

def lag(series, k):
    return np.concatenate([[np.nan] * k, series[:-k]]) if k > 0 else series

lag_1 = lag(sales, 1)
lag_2 = lag(sales, 2)

print("day:   ", list(range(len(sales))))
print("sales: ", sales)
print("lag_1: ", lag_1)                              # each row = yesterday's sales
print("lag_2: ", lag_2)                              # each row = sales from 2 days ago

# These lagged columns become ordinary numeric FEATURES for a regular ML model -
# this is exactly how "ML-based forecasting" turns a time series into a tabular problem.


### Checklist item: Rolling statistics

**Approach:**
- Protects representation: rolling means, standard deviations, and similar statistics smooth out noise and expose local trend or volatility, which are strong predictive features for many forecasting models.
- Why this is the mathematically right approach: a rolling mean over a window of size w is mathematically a low-pass filter - it averages out fluctuations with a period shorter than w while preserving slower-moving trends, which is precisely why window size directly controls the trade-off between responsiveness and smoothness.
- For this checklist item: Rolling mean and rolling std summarize recent history into a single feature; they smooth noise and help models detect sustained shifts in the series.
- Code walkthrough: Check that rolling_mean smooths out the jumpy raw sales values while rolling_std stays low when sales are steady and rises where the series jumps - those are the level and volatility signals this feature type captures.

**Learn more:**
- Website: [Forecasting: Principles and Practice](https://otexts.com/fpp3/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Time+Series+Forecasting+Rolling+statistics+machine+learning+theory)

**Trade-offs:**
- Rolling statistics summarize recent history into a single number per row, cheaper to compute than a full seasonal decomposition, but that summary can blur together a real trend change and ordinary short-term noise.
- The rolling window size trades responsiveness, since a short window reacts fast but is noisy, against stability, since a long window is smooth but lags real changes, so choose it based on the forecast horizon, not an arbitrary default.

**Practical software engineering use cases:**
- When to use it: Use rolling statistics to capture recent trend or volatility as features for a downstream forecasting or anomaly-detection model.
- When not to use it: Don't pick a rolling window size arbitrarily - a mismatch with your actual forecast horizon can make the feature laggy or overly noisy.


In [ ]:
# Time Series Forecasting - Rolling statistics (numpy)

import numpy as np

# A rolling statistic summarizes the last N values as of each point in time -
# useful for smoothing noise and capturing recent trend/volatility.

sales = np.array([10, 12, 13, 15, 18, 22, 21, 25])
window = 3

def rolling(series, window, func):
    return np.array([np.nan] * (window - 1) +
                    [func(series[i - window + 1:i + 1]) for i in range(window - 1, len(series))])

rolling_mean = rolling(sales, window, np.mean)
rolling_std = rolling(sales, window, np.std)

print("sales:       ", sales)
print("rolling_mean:", np.round(rolling_mean, 2))     # smoothed recent level
print("rolling_std: ", np.round(rolling_std, 2))       # smoothed recent volatility

# A sudden jump in rolling_std (volatility) or rolling_mean (level) is often a more
# reliable signal than a single noisy data point.


### Checklist item: Train/test split for time series

**Approach:**
- Protects evaluation integrity: a time series must be split chronologically, training on the past and testing on the future, because a random split lets the model see the future during training, producing wildly optimistic results.
- Why this is the mathematically right approach: any forecasting model's validity rests on only using past information to predict the future; a random split violates this by definition, since it uses rows chronologically after the test point during training - a chronological split is what makes the evaluation setup mathematically consistent with how the model will actually be used at inference time.
- For this checklist item: Decide what future production data will look like, then split so validation simulates that future rather than leaking information across rows.
- Code walkthrough: Check that train, validation, and test are three non-overlapping, chronologically ordered slices - validation and test both come strictly after train, unlike a random split which would mix them.

**Learn more:**
- Website: [Forecasting: Principles and Practice](https://otexts.com/fpp3/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Time+Series+Forecasting+Train%2Ftest+split+for+time+series+machine+learning+theory)

**Trade-offs:**
- A chronological split costs nothing extra to implement compared to a random split, but skipping it is the single most common way a time-series project's offline metrics end up disconnected from real deployment performance.
- Chronological or walk-forward validation gives fewer independent test folds than random k-fold CV, so metric estimates are noisier; this is the necessary cost of getting a realistic estimate.

**Practical software engineering use cases:**
- When to use it: Use a chronological or walk-forward split for every time-series evaluation, without exception.
- When not to use it: Don't use a random train/test split on time-series data even just to try it - it will leak future information into training and produce an unrealistically optimistic metric.


In [204]:
# Time Series Forecasting - Train/test split for time series
sales_by_month = [100, 105, 108, 115, 117, 121, 130, 133]
train = sales_by_month[:5]
validation = sales_by_month[5:7]
test = sales_by_month[7:]
print("train window:", train)
print("validation future window:", validation)
print("test final future window:", test)


train window: [100, 105, 108, 115, 117]
validation future window: [121, 130]
test final future window: [133]


### Checklist item: ARIMA / Prophet / ML-based forecasting

**Approach:**
- Protects model selection: ARIMA suits stationary, linear, well-understood series, Prophet suits series with strong seasonality or holidays and messy real-world gaps, and ML-based approaches suit series with many exogenous features, so matching the tool to the series' properties avoids wasted tuning.
- Why this is the mathematically right approach: ARIMA's autoregressive term is mathematically a weighted sum of the series' own past values, matching the linear regression structure covered earlier, just applied to lagged versions of the same variable, which is why differencing to reach stationarity first is a prerequisite for its linear-model assumptions to hold.
- For this checklist item: ARIMA fits a linear difference equation to the series; Prophet decomposes trend+seasonality; ML-based forecasting (XGBoost on lag features) often outperforms both on long series.
- Code walkthrough: Check which of the three printed MAE numbers is lowest for this series - since it has strong seasonality here, expect the Prophet-style trend-plus-seasonality approach to edge out the naive ARIMA-style and lag-based ML approach.

**Learn more:**
- Website: [Forecasting: Principles and Practice](https://otexts.com/fpp3/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Time+Series+Forecasting+ARIMA+%2F+Prophet+%2F+ML-based+forecasting+machine+learning+theory)

**Trade-offs:**
- All three approaches were trained and evaluated on the exact same 12-step holdout window here, so the MAE differences you see reflect real modeling trade-offs on this series, not differences in how fairly each was tested.
- ARIMA is interpretable but brittle to structural breaks; Prophet is robust to missing data and easy to use but can underperform on series without clear seasonal structure; ML-based forecasting can incorporate rich features but needs more careful feature engineering and leakage checks.

**Practical software engineering use cases:**
- When to use it: Use Prophet-style methods when your series has strong, regular seasonality and messy real-world gaps; use ML-based forecasting when you have extra predictive features to add.
- When not to use it: Don't default to ARIMA for a series with a structural break, a sudden regime change - its autoregressive assumptions break down exactly when that happens.


In [ ]:
# Time Series Forecasting - ARIMA / Prophet / ML-based forecasting (numpy + scikit-learn)

import numpy as np
from sklearn.linear_model import LinearRegression

# Three families, compared on the SAME series:
#   ARIMA-style   : models the series from its own past values (autoregression)
#   Prophet-style : decomposes into trend + seasonality, fits each with a curve
#   ML-based      : turns lags into tabular features and hands them to a regressor
# Below is a tiny illustrative stand-in for each idea (not the real libraries).

rng = np.random.default_rng(5)
t = np.arange(60)
series = 50 + 0.8 * t + 5 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 1.5, 60)

train, test = series[:48], series[48:]

# ARIMA-style: naive autoregression - next value = last value + average recent trend.
recent_diffs = np.diff(train[-12:])
ar_forecast = train[-1] + np.cumsum(np.full(len(test), recent_diffs.mean()))

# Prophet-style: fit trend (linear) + seasonality (average per-month effect) separately.
trend_model = LinearRegression().fit(np.arange(len(train)).reshape(-1, 1), train)
trend_only = trend_model.predict(np.arange(len(train)).reshape(-1, 1))
seasonal = train - trend_only
month_effect = np.array([seasonal[i::12].mean() for i in range(12)])
future_idx = np.arange(len(train), len(train) + len(test))
prophet_forecast = trend_model.predict(future_idx.reshape(-1, 1)) + month_effect[future_idx % 12]

# ML-based: build lag features (previous 3 values) and fit a regressor on them.
X_lag = np.array([train[i - 3:i] for i in range(3, len(train))])
y_lag = train[3:]
ml_model = LinearRegression().fit(X_lag, y_lag)
history = list(train[-3:])
ml_forecast = []
for _ in range(len(test)):
    pred = ml_model.predict([history[-3:]])[0]
    ml_forecast.append(pred)
    history.append(pred)                             # feed the prediction back in as a new lag
ml_forecast = np.array(ml_forecast)

for name, forecast in [("ARIMA-style   ", ar_forecast), ("Prophet-style ", prophet_forecast),
                       ("ML-based      ", ml_forecast)]:
    mae = np.mean(np.abs(test - forecast))
    print(f"{name}: MAE = {mae:.2f}")

# No single family always wins - ARIMA suits simple autocorrelated series, Prophet suits
# strong seasonality, and ML-based approaches shine when you have extra features to add.


In [206]:
# Practice: Time Series Forecasting

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Recommender Systems

### Study checklist
- [ ] Popularity baseline
- [ ] Content-based filtering
- [ ] Collaborative filtering
- [ ] Matrix factorization concept
- [ ] Cold start problem
- [ ] Evaluation of recommendations

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Popularity baseline

**Approach:**
- Protects evaluation integrity: a recommend-the-most-popular-items baseline is the floor any real recommender must beat, and it's a strong floor because popularity bias is powerful.
- Why this is the mathematically right approach: ranking items purely by interaction count is equivalent to estimating P(interaction | item) using only the item's marginal frequency, ignoring the user entirely - it's the simplest valid maximum-likelihood estimate you can make with zero user-specific information, which is exactly why it's a legitimate floor to compare every richer method against.
- For this checklist item: A popularity baseline recommends the globally most-interacted items to everyone; it is cheap to build and hard to beat on precision@K in cold-start conditions.
- Code walkthrough: Check that the recommendations list is sorted purely by interaction count, identical for every user - that sameness for everyone is exactly the lack of personalization this baseline trades away for simplicity.

**Learn more:**
- Website: [Google: Recommendation Systems course](https://developers.google.com/machine-learning/recommendation)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Recommender+Systems+Popularity+baseline+machine+learning+theory)

**Trade-offs:**
- A popularity baseline needs no user-item interaction matrix to factorize and no similarity computation at all, unlike collaborative or content-based filtering, which is exactly why it's the cheapest possible recommender to stand up.
- It's trivial to implement and surprisingly hard to beat for new users, but it produces zero personalization and reinforces popularity bias, a rich-get-richer effect, if left as the production system.

**Practical software engineering use cases:**
- When to use it: Use a popularity baseline as your very first recommender, and as the permanent fallback for brand-new users.
- When not to use it: Don't leave popularity as your only recommendation logic once you have enough interaction data to personalize - it never improves beyond what's generally popular.


In [207]:
# Recommender Systems - Popularity baseline
interactions = [
    ("u1", "course_a"), ("u2", "course_a"), ("u3", "course_b"),
    ("u4", "course_a"), ("u5", "course_c"), ("u6", "course_b"),
]
counts = {}
for _, item in interactions:
    counts[item] = counts.get(item, 0) + 1
recommendations = sorted(counts.items(), key=lambda pair: pair[1], reverse=True)
print("recommend to every new user:", recommendations)


recommend to every new user: [('course_a', 3), ('course_b', 2), ('course_c', 1)]


### Checklist item: Content-based filtering

**Approach:**
- Protects representation: this approach recommends based on item attributes similar to what a user has liked, so it directly depends on having rich, well-structured item metadata.
- Why this is the mathematically right approach: Jaccard similarity, the size of the intersection divided by the size of the union of two tag sets, is a mathematically well-defined similarity measure that equals 1 for identical sets and 0 for disjoint ones - using it to rank items against a user's tag profile is a direct, principled way to quantify overlap, not an arbitrary heuristic.
- For this checklist item: Content-based filtering recommends items whose feature vectors are similar to items the user has already liked â€” solves cold start for items, not for users.
- Code walkthrough: Check that course_a scores highest for a user profile of ml/python - it shares the most tags with that profile via the jaccard() overlap calculation, with no interaction history needed.

**Learn more:**
- Website: [Google: Recommendation Systems course](https://developers.google.com/machine-learning/recommendation)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Recommender+Systems+Content-based+filtering+machine+learning+theory)

**Trade-offs:**
- Content-based filtering only needs a single user's history to make recommendations, unlike collaborative filtering which needs a community of similar users - a real advantage for a brand-new product with few users so far.
- It handles new items well since no interaction history is needed, but it tends to over-specialize recommendations to a user's past behavior, limiting discovery of genuinely novel items, sometimes called a filter bubble.

**Practical software engineering use cases:**
- When to use it: Use content-based filtering when item metadata is rich and you need to recommend brand-new items with zero interaction history.
- When not to use it: Don't rely on it alone if you want to help users discover things outside their established preferences - it tends to narrow recommendations toward what they already like.


In [208]:
# Recommender Systems - Content-based filtering
user_profile = {"ml", "python"}
item_tags = {
    "course_a": {"ml", "statistics"},
    "course_b": {"python", "data"},
    "course_c": {"frontend", "css"},
}

def jaccard(a, b):
    return len(a & b) / len(a | b)

scores = {item: jaccard(user_profile, tags) for item, tags in item_tags.items()}
print(sorted(scores.items(), key=lambda pair: pair[1], reverse=True))


[('course_a', 0.3333333333333333), ('course_b', 0.3333333333333333), ('course_c', 0.0)]


### Checklist item: Collaborative filtering

**Approach:**
- Protects representation: this approach infers preferences from patterns across many users' behavior rather than item content, so it captures signal content-based methods miss, for example "people who liked X also liked Y" for otherwise unrelated items.
- Why this is the mathematically right approach: finding the user whose interaction set has the highest Jaccard overlap with the target user, then recommending items from that neighbor's set the target hasn't seen, is mathematically an instance of the same nearest-neighbor principle as KNN, just applied to user-item interaction sets instead of numeric feature vectors.
- For this checklist item: Collaborative filtering finds users with similar interaction histories and recommends what those peers liked; pure CF needs enough data per user (cold-start limitation).
- Code walkthrough: Check which neighbor scores highest in the printed similarity list, then confirm the recommended items are exactly that neighbor's liked items minus whatever the target user already likes.

**Learn more:**
- Website: [Google: Recommendation Systems course](https://developers.google.com/machine-learning/recommendation)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Recommender+Systems+Collaborative+filtering+machine+learning+theory)

**Trade-offs:**
- Collaborative filtering can surface non-obvious recommendations that share no descriptive tags at all, something content-based filtering's tag-overlap approach can never do by design, since it only ever looks at item metadata.
- It needs a substantial interaction history to work well and struggles badly with new users or items, the cold start problem, that have no interaction history yet.

**Practical software engineering use cases:**
- When to use it: Use collaborative filtering once you have enough interaction history to find meaningful similarity patterns between users or items.
- When not to use it: Don't use it as your only method for new users or items with no interaction history yet - it has nothing to compute a similarity from.


In [209]:
# Recommender Systems - Collaborative filtering
likes = {
    "alice": {"ml_book", "python_book"},
    "bob": {"ml_book", "stats_book"},
    "carol": {"python_book", "data_book"},
}

def similarity(u1, u2):
    return len(likes[u1] & likes[u2]) / len(likes[u1] | likes[u2])

neighbors = sorted((similarity("alice", user), user) for user in likes if user != "alice")
best_neighbor = neighbors[-1][1]
recommendations = likes[best_neighbor] - likes["alice"]
print("nearest neighbor:", best_neighbor)
print("recommendations:", sorted(recommendations))


nearest neighbor: carol
recommendations: ['data_book']


### Checklist item: Matrix factorization concept

**Approach:**
- Protects representation and model fit: factorizing the sparse user-item interaction matrix into dense latent factors is what lets collaborative filtering generalize to unseen user-item pairs instead of just memorizing observed interactions.
- Why this is the mathematically right approach: approximating the mostly empty user-item ratings matrix R as the product of two smaller matrices, R is approximately U times V transpose, is mathematically the same idea as PCA's low-rank approximation - the specific factorization, found by minimizing squared reconstruction error on the known entries, fills in the unknown entries with the model's best estimate.
- For this checklist item: Matrix factorization decomposes the user-item rating matrix R â‰ˆ U Ã— V^T into low-rank user and item embeddings; missing entries are predicted by dot products.
- Code walkthrough: Check that the predicted matrix roughly reproduces the known ratings where they exist, while also filling in a number for every starred, previously-unrated cell - that fill-in is the entire point of factorizing the matrix.

**Learn more:**
- Website: [Google: Recommendation Systems course](https://developers.google.com/machine-learning/recommendation)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Recommender+Systems+Matrix+factorization+concept+machine+learning+theory)

**Trade-offs:**
- Matrix factorization generalizes to predict entries no simple similarity calculation, like the Jaccard overlap used for collaborative filtering above, would even attempt, since it learns latent structure rather than comparing raw overlaps.
- The number of latent factors is a bias-variance trade-off, since too few underfits nuanced preferences and too many overfits sparse data, and the learned factors themselves aren't directly interpretable.

**Practical software engineering use cases:**
- When to use it: Use matrix factorization when your interaction matrix is large and sparse and you need predictions for pairs with no direct overlap.
- When not to use it: Don't expect the learned latent factors to be individually interpretable to a stakeholder - they're optimized for prediction accuracy, not human-readable meaning.


In [210]:
# Recommender Systems - Matrix factorization concept
ratings = {
    ("alice","ml_book"):5, ("alice","python_book"):4,
    ("bob",  "ml_book"):4, ("bob",  "stats_book"):3,
    ("carol","python_book"):5, ("carol","stats_book"):4,
}
users = ["alice","bob","carol"]
items = ["ml_book","python_book","stats_book"]

print("User-item matrix:")
print(f"{'':12s}", end="")
for it in items: print(f"  {it[:12]:>12}", end="")
print()
for u in users:
    print(f"{u:12s}", end="")
    for it in items:
        print(f"  {str(ratings.get((u,it),'-')):>12}", end="")
    print()

# Simulated low-rank factors k=2
U = {"alice":[0.9,0.2],"bob":[0.8,0.4],"carol":[0.3,0.9]}
V = {"ml_book":[0.9,0.1],"python_book":[0.6,0.7],"stats_book":[0.5,0.8]}
print("\nPredicted (U@V.T) â€” * means unobserved entry:")
print(f"{'':12s}", end="")
for it in items: print(f"  {it[:12]:>12}", end="")
print()
for u in users:
    print(f"{u:12s}", end="")
    for it in items:
        pred = sum(U[u][k]*V[it][k] for k in range(2))
        star = "*" if (u,it) not in ratings else ""
        print(f"  {pred:.2f}{star:1s}", end="")
    print()
print("* = prediction for unrated items")


User-item matrix:
                   ml_book   python_book    stats_book
alice                    5             4             -
bob                      4             -             3
carol                    -             5             4

Predicted (U@V.T) â€” * means unobserved entry:
                   ml_book   python_book    stats_book
alice         0.83   0.68   0.61*
bob           0.76   0.76*  0.72 
carol         0.36*  0.81   0.87 
* = prediction for unrated items


### Checklist item: Cold start problem

**Approach:**
- Protects user value: new users and new items have no interaction history for collaborative filtering to use, so without a fallback strategy they get poor or no recommendations exactly when a good first impression matters most.
- Why this is the mathematically right approach: collaborative filtering and matrix factorization both require prior interaction data to compute a similarity or fit a factorization - for a brand-new user or item with zero recorded interactions, there's mathematically no data for these methods to operate on at all, which is precisely why a fundamentally different, interaction-free method is needed as a fallback.
- For this checklist item: A new user has no history; a new item has no interactions â€” content-based features or popularity baselines bridge the gap until enough signal accumulates.
- Code walkthrough: Check that course_b (matching the new user's 'python' tag) ranks above the others despite this user having zero interaction history - that's the content-based fallback solving cold start.

**Learn more:**
- Website: [Google: Recommendation Systems course](https://developers.google.com/machine-learning/recommendation)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Recommender+Systems+Cold+start+problem+machine+learning+theory)

**Trade-offs:**
- Cold start is specifically a failure mode of collaborative filtering and matrix factorization, not of content-based filtering or the popularity baseline - which is exactly why those two are the natural fallback for new users or items.
- Common mitigations such as a content-based fallback, a popularity baseline, or onboarding questions each add complexity or friction, and the right mix depends on how much new-user or new-item traffic the product actually sees.

**Practical software engineering use cases:**
- When to use it: Use a content-based or popularity fallback specifically for the cold-start slice of your traffic: new users, new items.
- When not to use it: Don't apply your main collaborative-filtering or matrix-factorization model directly to brand-new users - it has no data to work from and will produce poor or arbitrary recommendations.


In [211]:
# Recommender Systems - Cold start problem
new_user_profile = {"python"}
popular_items = ["course_a", "course_b", "course_c"]
item_tags = {
    "course_a": {"ml", "statistics"},
    "course_b": {"python", "data"},
    "course_c": {"frontend", "css"},
}

content_scores = {item: len(new_user_profile & tags) for item, tags in item_tags.items()}
ranked = sorted(content_scores, key=lambda item: (content_scores[item], -popular_items.index(item)), reverse=True)
print("new user fallback ranking:", ranked)
print("uses profile tags first, popularity as tie-breaker")


new user fallback ranking: ['course_b', 'course_a', 'course_c']
uses profile tags first, popularity as tie-breaker


### Checklist item: Evaluation of recommendations

**Approach:**
- Protects evaluation integrity: offline metrics like precision@k, recall@k, and NDCG approximate but don't equal real user satisfaction, so recommender evaluation needs both offline metrics and online A/B testing before trusting a model.
- Why this is the mathematically right approach: precision@k is the count of relevant items in the top k divided by k, and NDCG additionally weights each hit by 1 over log2(rank+1), a discount that mathematically rewards a relevant item appearing at rank 1 far more than at rank 10 - these formulas are specifically designed to score a ranked list, unlike accuracy, which has no concept of order.
- For this checklist item: Offline metrics: precision@K, recall@K, NDCG. Online metrics: click-through rate, add-to-cart, conversion â€” offline and online rankings often disagree.
- Code walkthrough: Check which of the 3 recommended items actually appear in actually_clicked, and confirm precision_at_k and recall_at_k are computed from that same hit list - ndcg additionally rewards hits that rank higher.

**Learn more:**
- Website: [Google: Recommendation Systems course](https://developers.google.com/machine-learning/recommendation)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Recommender+Systems+Evaluation+of+recommendations+machine+learning+theory)

**Trade-offs:**
- Precision@k, recall@k, and NDCG all reuse the same recommended list and click data, but they answer different questions: precision cares about accuracy of what's shown, recall about coverage of what mattered, and NDCG about whether the best items ranked first.
- Offline metrics are cheap and fast to iterate on, but a model that wins offline can lose online, for example by recommending obviously relevant but already-known items, so final validation should always be a live experiment, not just an offline leaderboard.

**Practical software engineering use cases:**
- When to use it: Use offline metrics like precision@k and NDCG for fast iteration, then confirm any win with a live A/B test before rolling out broadly.
- When not to use it: Don't ship a recommender change based on offline metrics alone - a model that wins offline can lose online by recommending items users already know about.


In [212]:
# Recommender Systems - Evaluation of recommendations
recommended = ["course_a", "course_b", "course_c"]
actually_clicked = {"course_b", "course_d"}
k = 3
hits = [1 if item in actually_clicked else 0 for item in recommended[:k]]
precision_at_k = sum(hits) / k
recall_at_k = sum(hits) / len(actually_clicked)

dcg = sum(hit / (rank + 1) for rank, hit in enumerate(hits))
ideal_hits = sorted(hits, reverse=True)
idcg = sum(hit / (rank + 1) for rank, hit in enumerate(ideal_hits))
ndcg = dcg / idcg if idcg else 0

print("precision@3:", round(precision_at_k, 3))
print("recall@3:", round(recall_at_k, 3))
print("simple ndcg@3:", round(ndcg, 3))


precision@3: 0.333
recall@3: 0.5
simple ndcg@3: 0.5


In [213]:
# Practice: Recommender Systems

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Anomaly Detection

### Study checklist
- [ ] Statistical anomaly detection
- [ ] Isolation Forest
- [ ] One-class SVM
- [ ] Autoencoder concept
- [ ] Precision challenge in anomaly detection

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Statistical anomaly detection

**Approach:**
- Protects data quality and evaluation integrity: simple statistical rules like z-score or IQR give a fast, interpretable first line of defense for catching anomalies without training a model.
- Why this is the mathematically right approach: under a normal distribution, roughly 95% of values fall within 2 standard deviations of the mean and about 99.7% within 3, a mathematical property of the Gaussian, not an empirical rule - flagging points beyond a chosen number of standard deviations directly applies this known probability mass to decide how surprising a value is.
- For this checklist item: Flag points beyond mean Â± 2â€“3 std, or outside 1.5Ã—IQR fences, as potential anomalies; quick to compute but fails on multivariate or non-Gaussian distributions.
- Code walkthrough: Check that only the value 99 gets flagged out of [10, 11, 10, 12, 99] - that's the z-score rule catching the one value that's obviously different from the rest.

**Learn more:**
- Website: [scikit-learn: Novelty and Outlier Detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Anomaly+Detection+Statistical+anomaly+detection+machine+learning+theory)

**Trade-offs:**
- A z-score rule requires no training step and no model at all, unlike Isolation Forest or a one-class SVM, making it the cheapest possible anomaly check - but that simplicity is exactly why it can only look at one feature in isolation.
- These methods assume a roughly known, stable distribution, often Gaussian, and typically look at one feature at a time, so they miss multivariate anomalies where individually normal values are jointly anomalous.

**Practical software engineering use cases:**
- When to use it: Use simple z-score or IQR rules as a first, cheap line of defense for monitoring a single well-understood metric, like request latency.
- When not to use it: Don't rely on single-feature statistical rules to catch anomalies only visible across multiple features jointly, like an unusual combination of amount and location in a transaction.


In [214]:
import math
# Anomaly Detection - Statistical anomaly detection
values = [10, 11, 10, 12, 99]
mean = sum(values) / len(values)
std = math.sqrt(sum((v - mean) ** 2 for v in values) / len(values))
anomalies = [v for v in values if abs(v - mean) > 2 * std]
print("anomalies:", anomalies)


anomalies: []


### Checklist item: Isolation Forest

**Approach:**
- Protects model fit: Isolation Forest detects anomalies by how few random splits it takes to isolate a point, exploiting the fact that anomalies are few and different rather than modeling what normal looks like directly.
- Why this is the mathematically right approach: a point that's easy to isolate with few random splits is, by the mathematics of random partitioning, one that sits in a sparse region far from other points - anomalies systematically require fewer splits to isolate than points in dense clusters, which is why average path length across many random trees is a valid, if indirect, measure of how anomalous a point is.
- For this checklist item: Isolation Forest randomly partitions feature space; anomalies are isolated in fewer splits on average â€” it scales to high dimensions and is unsupervised.
- Code walkthrough: Check that the [80, 5] point gets a noticeably lower avg_depth than the tightly-clustered points - needing fewer random splits to isolate a point is exactly the Isolation Forest anomaly signal.

**Learn more:**
- Website: [scikit-learn: Novelty and Outlier Detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Anomaly+Detection+Isolation+Forest+machine+learning+theory)

**Trade-offs:**
- Isolation Forest scales to far more dimensions than the simple z-score rule above, since it doesn't need to assume any particular per-feature distribution, but it trades that flexibility for a less intuitive anomaly score than a plain threshold.
- It doesn't require labeled anomalies, which is good since they're often rare or unavailable, but the contamination parameter, the expected anomaly rate, must be estimated and directly affects the decision threshold.

**Practical software engineering use cases:**
- When to use it: Use Isolation Forest when you don't have labeled anomalies to train against and need an unsupervised method that scales to many features.
- When not to use it: Don't set the contamination parameter without at least a rough estimate of your true anomaly rate - an unrealistic value skews the decision threshold either way.


In [215]:
# Anomaly Detection - Isolation Forest
import random

points = [[10, 11], [11, 10], [10, 10], [12, 11], [80, 5]]

def isolation_depth(point, rows, rng, depth=0):
    if len(rows) <= 1:
        return depth
    usable_cols = [col for col in range(len(point)) if min(row[col] for row in rows) < max(row[col] for row in rows)]
    if not usable_cols:
        return depth
    split_col = rng.choice(usable_cols)
    low = min(row[split_col] for row in rows)
    high = max(row[split_col] for row in rows)
    split = rng.uniform(low, high)
    branch = [row for row in rows if (row[split_col] < split) == (point[split_col] < split)]
    if len(branch) == len(rows):
        return depth
    return isolation_depth(point, branch, rng, depth + 1)

for point in points:
    depths = [isolation_depth(point, points, random.Random(seed)) for seed in range(20)]
    avg_depth = sum(depths) / len(depths)
    label = "anomaly" if avg_depth < 2 else "normal"
    print(point, "avg_depth=", round(avg_depth, 2), label)


[10, 11] avg_depth= 2.95 normal
[11, 10] avg_depth= 3.05 normal
[10, 10] avg_depth= 3.1 normal
[12, 11] avg_depth= 2.75 normal
[80, 5] avg_depth= 1.15 anomaly


### Checklist item: One-class SVM

**Approach:**
- Protects model fit: this method learns a boundary around the normal data region using only normal examples, useful when anomalies are too rare or varied to model directly.
- Why this is the mathematically right approach: One-Class SVM solves an optimization that finds the smallest-volume region in kernel-transformed feature space containing most of the training data - mathematically, it's the same margin-maximization machinery as standard SVM, just applied to separate the data from the origin rather than from another class.
- For this checklist item: One-class SVM learns a boundary around normal training data; points outside that boundary are anomalies â€” sensitive to feature scaling and outliers in the training set.
- Code walkthrough: Check that [80, 5]'s boundary_score falls below the 0.20 threshold while every training-like point stays above it - low average similarity to the normal training points is what flags anomalies here.

**Learn more:**
- Website: [scikit-learn: Novelty and Outlier Detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Anomaly+Detection+One-class+SVM+machine+learning+theory)

**Trade-offs:**
- Compared to Isolation Forest, One-Class SVM defines 'normal' with a more mathematically precise boundary, but building that boundary from similarity to every training point makes it slower to train and score at scale.
- It's sensitive to the choice of kernel, gamma, and nu hyperparameters and, like standard SVM, doesn't scale well to very large datasets.

**Practical software engineering use cases:**
- When to use it: Use One-Class SVM when you have a clean sample of normal data and need a precise mathematical boundary around it.
- When not to use it: Don't use it on a very large training set - the pairwise similarity computation to every training point makes it slow to train and score compared to Isolation Forest.


In [216]:
# Anomaly Detection - One-class SVM
import math

train_points = [[10, 11], [11, 10], [10, 10], [12, 11]]
points = train_points + [[80, 5]]
gamma = 0.05
threshold = 0.20

def rbf_similarity(a, b):
    squared_distance = sum((x - y) ** 2 for x, y in zip(a, b))
    return math.exp(-gamma * squared_distance)

for point in points:
    boundary_score = sum(rbf_similarity(point, train) for train in train_points) / len(train_points)
    label = "normal" if boundary_score >= threshold else "anomaly"
    print(point, "boundary_score=", round(boundary_score, 3), label)


[10, 11] boundary_score= 0.919 normal
[11, 10] boundary_score= 0.94 normal
[10, 10] boundary_score= 0.92 normal
[12, 11] boundary_score= 0.876 normal
[80, 5] boundary_score= 0.0 anomaly


### Checklist item: Autoencoder concept

**Approach:**
- Protects representation and model fit: an autoencoder trained on normal data learns to reconstruct it well, so anomalies unlike anything seen in training reconstruct poorly, and that reconstruction error becomes the anomaly score.
- Why this is the mathematically right approach: training a network to reconstruct its own input through a smaller bottleneck layer forces it to learn a compressed representation of whatever patterns are common in the training data - by construction, an input unlike anything in training can't be represented well by that learned compression, so a mathematically large reconstruction error is the expected signature of an anomaly.
- For this checklist item: Train an encoder-decoder to reconstruct normal examples through a compressed bottleneck; use high reconstruction error as the anomaly score.
- Code walkthrough: Check that (8.0, 1.0)'s reconstruction_error is much larger than the other points' - that point can't be well summarized by a single bottleneck value the way the diagonal-ish points can, which is why it gets flagged.

**Learn more:**
- Website: [scikit-learn: Novelty and Outlier Detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Anomaly+Detection+Autoencoder+concept+machine+learning+theory)

**Trade-offs:**
- Unlike Isolation Forest or One-Class SVM, an autoencoder can learn a highly nonlinear notion of 'normal' by compressing and reconstructing the data itself, at the cost of needing a real training loop instead of a single fit() call.
- It needs enough normal data and training time to learn a good representation, more setup cost than statistical or isolation-based methods, and it can fail to flag anomalies that happen to reconstruct well through the bottleneck.

**Practical software engineering use cases:**
- When to use it: Use an autoencoder when normal behavior has complex, nonlinear structure that simpler statistical or tree-based methods can't capture.
- When not to use it: Don't reach for an autoencoder when you have too little normal data to train a reliable reconstruction - a simpler method will be both cheaper and more robust with limited data.


In [217]:
# Anomaly Detection - Autoencoder concept

points = [(1.0, 1.1), (2.0, 1.9), (3.0, 3.1), (4.0, 3.9), (8.0, 1.0)]
threshold = 3.0

# Toy linear autoencoder: compress each point to one value, then reconstruct x and y from it.
reconstruction_errors = []
for x, y in points:
    bottleneck = (x + y) / 2
    reconstructed = (bottleneck, bottleneck)
    error = (x - reconstructed[0]) ** 2 + (y - reconstructed[1]) ** 2
    reconstruction_errors.append(error)

for point, error in zip(points, reconstruction_errors):
    label = "anomaly" if error > threshold else "normal"
    print(point, "reconstruction_error=", round(error, 2), label)


(1.0, 1.1) reconstruction_error= 0.01 normal
(2.0, 1.9) reconstruction_error= 0.01 normal
(3.0, 3.1) reconstruction_error= 0.01 normal
(4.0, 3.9) reconstruction_error= 0.01 normal
(8.0, 1.0) reconstruction_error= 24.5 anomaly


### Checklist item: Precision challenge in anomaly detection

**Approach:**
- Protects evaluation integrity and user value: because true anomalies are rare, even a small false-positive rate translates into a large volume of false alarms relative to real anomalies, which directly affects operator trust in the system.
- Why this is the mathematically right approach: precision equals TP over (TP+FP); when true anomalies are a tiny fraction of all data, even a small false-positive rate applied to the very large pool of normal points produces an absolute number of false positives that can dwarf the true positives - a direct mathematical consequence of multiplying a small rate by a large base count.
- For this checklist item: Anomaly detection is extreme-class imbalance (maybe 0.1% positives); precision will be low even with good recall â€” use PR-AUC and calibrate alerts for business tolerance.
- Code walkthrough: Check the computed precision value against how many of the model's 'anomaly' flags were actually real - a low precision here means most flagged anomalies are false alarms, the core operational challenge for this item.

**Learn more:**
- Website: [scikit-learn: Novelty and Outlier Detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Anomaly+Detection+Precision+challenge+in+anomaly+detection+machine+learning+theory)

**Trade-offs:**
- This precision challenge applies to every method covered above (statistical rules, Isolation Forest, One-Class SVM, autoencoders) equally, since it's a consequence of the target class being rare, not a weakness specific to any one algorithm.
- Tightening the threshold to cut false alarms will also miss more real anomalies, so this trade-off should be set based on the operational cost of investigating a false alarm vs. missing a real one, not a fixed statistical target.

**Practical software engineering use cases:**
- When to use it: Use this awareness when setting an alert threshold for any anomaly system that routes to a human reviewer.
- When not to use it: Don't tune purely for high recall on a rare-event anomaly system without checking the resulting alert volume - you can overwhelm reviewers with false positives even at a good-looking recall number.


In [218]:
# Anomaly Detection - Precision challenge in anomaly detection
actual = [1, 0, 1, 1, 0]
predicted = [1, 0, 0, 1, 1]
tp = sum(a == 1 and p == 1 for a, p in zip(actual, predicted))
fp = sum(a == 0 and p == 1 for a, p in zip(actual, predicted))
fn = sum(a == 1 and p == 0 for a, p in zip(actual, predicted))
precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
print({"precision": precision, "recall": recall})


{'precision': 0.6666666666666666, 'recall': 0.6666666666666666}


In [219]:
# Practice: Anomaly Detection

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:



## Explainable AI

### Study checklist
- [ ] Feature importance
- [ ] Permutation importance
- [ ] Partial dependence
- [ ] SHAP values
- [ ] Local vs global explanations
- [ ] Risks of over-interpreting explanations
- [ ] Fairness and bias in ML

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.


### Checklist item: Feature importance

**Approach:**
- Protects user value and debugging: a global ranking of which features drive predictions overall is the starting point for explaining a model to stakeholders or auditors, and for catching leaked or spurious features.
- Why this is the mathematically right approach: impurity-based importance sums the exact impurity reduction attributed to a feature across every split that used it in a tree, a direct bookkeeping of numbers the tree-building algorithm already computed - it's a readout of the model's own internal optimization, not a separately estimated approximation.
- For this checklist item: Tree-based feature importance measures how much each feature reduces impurity across all splits â€” fast but biased toward high-cardinality and correlated features.
- Code walkthrough: Check that the printed importances rank feature_0 above feature_1 above feature_2 - matching exactly how the synthetic target was built (2*X0 + 0.3*X1 + noise) confirms the importances measure real signal.

**Learn more:**
- Website: [SHAP Documentation](https://shap.readthedocs.io/en/latest/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Explainable+AI+Feature+importance+machine+learning+theory)

**Trade-offs:**
- Feature importance is essentially free when using a tree-based model since it falls out of training, unlike for a black-box model such as an SVM with an RBF kernel, which has no built-in importance at all and needs a model-agnostic method instead.
- Importance isn't a single well-defined concept; different methods such as impurity-based, permutation, and SHAP can rank the same features differently, so state which method you used when reporting it.

**Practical software engineering use cases:**
- When to use it: Use built-in feature importance for a fast, first-pass answer to what this tree-based model relies on.
- When not to use it: Don't use it as your only evidence when a decision is being audited or challenged - cross-check with permutation importance, since impurity-based scores are biased toward high-cardinality features.


In [ ]:
# Explainable AI - Feature importance (using scikit-learn)

import numpy as np
from sklearn.ensemble import RandomForestClassifier

# "Feature importance" usually means a GLOBAL ranking: across the whole dataset,
# how much did each feature help the model separate the classes?

rng = np.random.default_rng(7)
n = 300
# Build data where feature 0 matters a lot, feature 1 a little, feature 2 not at all.
X = rng.normal(size=(n, 3))
y = ((2 * X[:, 0] + 0.3 * X[:, 1] + rng.normal(0, 0.5, n)) > 0).astype(int)

model = RandomForestClassifier(n_estimators=200, random_state=7).fit(X, y)
importances = model.feature_importances_

print("feature importances (should rank feature_0 > feature_1 > feature_2):")
for i, imp in enumerate(importances):
    print(f"  feature_{i}: {imp:.3f}")

print()
print("Importance says 'how much the MODEL relied on this feature', not")
print("'how much this feature causes the outcome' - see the risks cell later.")


### Checklist item: Permutation importance

**Approach:**
- Protects debugging integrity: measuring how much performance drops when a feature's values are shuffled gives a model-agnostic importance measure that isn't biased toward high-cardinality features the way impurity-based importance is.
- Why this is the mathematically right approach: randomly shuffling a feature's values breaks any statistical relationship between that feature and the target while leaving every other feature and the model itself completely unchanged - so the resulting drop in performance is mathematically isolated to exactly that one feature's contribution, which is what makes this method model-agnostic and unbiased toward feature type.
- For this checklist item: Permutation importance shuffles one feature at a time and measures the drop in val score â€” model-agnostic and more reliable than impurity-based importance.
- Code walkthrough: Check that feature_0's accuracy drop is far larger than feature_1's or feature_2's - a bigger drop when a feature is shuffled means the model relied on it more.

**Learn more:**
- Website: [SHAP Documentation](https://shap.readthedocs.io/en/latest/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Explainable+AI+Permutation+importance+machine+learning+theory)

**Trade-offs:**
- Because it only needs predictions in and scores out, permutation importance works identically for a linear model, a tree, or a neural network - a model-agnostic advantage the impurity-based importance in the Decision Trees section doesn't share.
- It's more computationally expensive since it requires re-scoring the model many times, and it can under-credit correlated features, since shuffling one still leaves its correlated partner intact and hides the damage.

**Practical software engineering use cases:**
- When to use it: Use permutation importance when you need a model-agnostic importance measure that works the same way for a linear model, tree, or neural network.
- When not to use it: Don't run it on a very large dataset without considering cost - it needs many re-scoring passes and can become a real bottleneck compared to a tree's built-in importances.


In [ ]:
# Explainable AI - Permutation importance (using scikit-learn)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

# Permutation importance: shuffle ONE feature's values (breaking its link to y),
# re-score the model, and see how much performance DROPS. A big drop = the model
# relied heavily on that feature; no drop = the model barely used it.

rng = np.random.default_rng(11)
n = 300
X = rng.normal(size=(n, 3))
y = (X[:, 0] + 0.1 * X[:, 1] > 0).astype(int)        # feature 0 matters most, 1 a little, 2 not at all

model = LogisticRegression().fit(X, y)
baseline_score = model.score(X, y)

result = permutation_importance(model, X, y, n_repeats=10, random_state=11)
print("baseline accuracy (no shuffling):", round(baseline_score, 3))
print()
print("accuracy drop when each feature is shuffled (bigger = more important):")
for i, drop in enumerate(result.importances_mean):
    print(f"  feature_{i}: {drop:+.3f}")

print()
print("Interpret this as 'how much the MODEL relies on this feature', not proof")
print("that the feature causally drives the outcome.")


### Checklist item: Partial dependence

**Approach:**
- Protects user value: a partial dependence plot shows the marginal effect of one feature on the prediction, averaged over all other features, which is how you explain what happens as a feature increases to a stakeholder.
- Why this is the mathematically right approach: a partial dependence value at x is defined as the average of the model's prediction over the entire dataset with that one feature forced to x - integrating, or averaging, out every other feature this way is the standard mathematical technique for isolating one variable's marginal effect from a multivariate function.
- For this checklist item: A partial dependence plot shows the marginal effect of one (or two) features on the predicted outcome, averaging over all other features.
- Code walkthrough: Check that the printed average-prediction values trace a U-shape as feature_0 sweeps from -2 to +2 - that shape recovers the true 3*x-squared relationship the model was trained on, without ever being told the formula.

**Learn more:**
- Website: [SHAP Documentation](https://shap.readthedocs.io/en/latest/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Explainable+AI+Partial+dependence+machine+learning+theory)

**Trade-offs:**
- A partial dependence plot shows an entire relationship curve, not just a single importance number, which is more informative than permutation importance but also requires re-predicting across a whole grid of values instead of one shuffle per feature.
- Averaging over other features can be misleading when features are strongly correlated, since it implicitly considers combinations that never occur in real data, so check feature correlation before trusting a PDP.

**Practical software engineering use cases:**
- When to use it: Use a partial dependence plot when you need to show a stakeholder the shape of a relationship, such as risk increasing sharply after age 60, not just a single importance number.
- When not to use it: Don't trust a PDP at face value when its featured variable is strongly correlated with another - it can imply combinations of values that never actually occur in your data.


In [ ]:
# Explainable AI - Partial dependence (using scikit-learn)

import numpy as np
from sklearn.ensemble import RandomForestRegressor

# Partial dependence: vary ONE feature across a range, holding all other feature
# values at their ACTUAL values, and average the model's predictions at each step.
# It shows "what happens to the prediction as this feature changes", on average.

rng = np.random.default_rng(8)
n = 300
X = rng.normal(size=(n, 2))
y = 3 * X[:, 0] ** 2 + X[:, 1] + rng.normal(0, 0.5, n)   # non-linear in feature 0

model = RandomForestRegressor(n_estimators=200, random_state=8).fit(X, y)

# Manually compute partial dependence for feature 0 over a small grid of values.
grid = np.linspace(-2, 2, 5)
pd_values = []
for value in grid:
    X_temp = X.copy()
    X_temp[:, 0] = value                             # force EVERY row's feature 0 to this value
    pd_values.append(model.predict(X_temp).mean())   # average prediction across all rows

print("feature_0 value -> average predicted y (partial dependence):")
for value, pd_val in zip(grid, pd_values):
    print(f"  {value:+.1f} -> {pd_val:.2f}")

print()
print("The U-shape mirrors the true 3*x^2 relationship - PDP recovered it without")
print("ever being told the underlying formula.")


### Checklist item: SHAP values

**Approach:**
- Protects user value and evaluation integrity: SHAP attributes each prediction's deviation from the average to individual features in a theoretically grounded, consistent way based on Shapley values, which is why it's become the standard for both local and global explanations.
- Why this is the mathematically right approach: SHAP values are computed as the average marginal contribution of a feature across every possible order features could be revealed in - Shapley's original game-theory proof shows this exact averaging procedure is the unique way to split a total payoff fairly among players, or a prediction among features, while satisfying additivity, symmetry, and consistency simultaneously.
- For this checklist item: SHAP values attribute each prediction's deviation from the base rate to individual features â€” they satisfy additivity and consistency, making them more trustworthy than importance rankings.
- Code walkthrough: Check that the two printed SHAP values add up, plus baseline, to exactly the full prediction - that additivity, guaranteed by averaging over every possible feature order, is the defining property of Shapley values.

**Learn more:**
- Website: [SHAP Documentation](https://shap.readthedocs.io/en/latest/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Explainable+AI+SHAP+values+machine+learning+theory)

**Trade-offs:**
- Unlike permutation importance or partial dependence, which describe the model's behavior in aggregate, SHAP assigns a specific number to every feature for every individual prediction - a finer granularity that costs considerably more compute to produce.
- Exact SHAP is computationally expensive on large models or datasets, though approximations like TreeSHAP help for tree ensembles specifically, and a mathematically fair attribution isn't automatically a causally correct one; SHAP explains the model, not necessarily the real world.

**Practical software engineering use cases:**
- When to use it: Use SHAP when you need to explain one specific prediction to a customer or auditor, such as why a loan application was denied.
- When not to use it: Don't compute full SHAP values on every prediction in a high-throughput production system without checking cost - it's considerably more expensive than a global importance score.


In [ ]:
# Explainable AI - SHAP values (manual exact Shapley value for 2 features)

import itertools

# SHAP values split a prediction's difference from the average fairly among features,
# using the same math as Shapley values from game theory: average each feature's
# marginal contribution across every possible ORDER features could be "added in".

baseline = 10.0                                      # prediction with NO features known
feature_values = {"age": 40, "income": 90}

# A toy "model": prediction = baseline + effect of age + effect of income + a small
# interaction (extra effect that ONLY appears when both are known together).
def predict(known):
    total = baseline
    if "age" in known:
        total += 0.3 * feature_values["age"]
    if "income" in known:
        total += 0.05 * feature_values["income"]
    if "age" in known and "income" in known:
        total += 2.0                                 # interaction bonus
    return total

features = ["age", "income"]
contributions = {f: [] for f in features}

# Try every ORDER of adding features in, and record each one's marginal contribution.
for order in itertools.permutations(features):
    known = set()
    prev = predict(known)
    for f in order:
        known.add(f)
        now = predict(known)
        contributions[f].append(now - prev)          # this feature's contribution in this order
        prev = now

shap_values = {f: sum(vals) / len(vals) for f, vals in contributions.items()}

print("full prediction:", predict(set(features)))
print("baseline (no features known):", baseline)
print("SHAP values (should sum to prediction - baseline):")
for f, v in shap_values.items():
    print(f"  {f}: {v:.2f}")
print("sum of SHAP values + baseline:", round(baseline + sum(shap_values.values()), 2))


### Checklist item: Local vs global explanations

**Approach:**
- Protects user value: a local explanation answers why this one prediction happened, and a global explanation answers what the model does overall, and picking the wrong one for the audience undermines trust.
- Why this is the mathematically right approach: a global explanation like permutation importance is mathematically an average taken over every row in the dataset, while a local explanation like a single row's SHAP values is computed for that one input specifically - averaging necessarily discards row-specific detail, which is exactly why the two views can mathematically disagree about what matters.
- For this checklist item: Global explanations (feature importance, PDP) describe average model behavior; local explanations (SHAP, LIME) explain individual predictions â€” use both for trust and debugging.
- Code walkthrough: Check that nudging feature_2 changes the prediction a lot for the row with feature_0 < 0 but barely at all for the row with feature_0 > 0 - that row-by-row difference is exactly what one global importance number can't show you.

**Learn more:**
- Website: [SHAP Documentation](https://shap.readthedocs.io/en/latest/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Explainable+AI+Local+vs+global+explanations+machine+learning+theory)

**Trade-offs:**
- Global methods like permutation importance answer 'what does the model rely on overall,' while local methods like SHAP answer 'why did it predict this for this row' - two different questions that can have genuinely different answers, as the code cell shows.
- Global explanations are more efficient to produce and communicate but can hide important local exceptions, for example a feature that matters a lot for one subgroup, that only a local, case-by-case explanation would surface.

**Practical software engineering use cases:**
- When to use it: Use a global explanation for a model-level report to leadership, and a local explanation when someone asks about one specific decision.
- When not to use it: Don't answer an individual's question of why they were denied with a global feature-importance chart - it can hide a subgroup effect that only a local explanation would reveal.


In [ ]:
# Explainable AI - Local vs global explanations (using scikit-learn)

import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# GLOBAL explanation: one summary describing the model behavior OVERALL (averaged
# across every row). LOCAL explanation: why did the model make THIS one prediction?

rng = np.random.default_rng(9)
n = 400
X = rng.normal(size=(n, 3))
# Feature 2 only matters for rows where feature 0 is NEGATIVE (a subgroup effect).
y = X[:, 0] + np.where(X[:, 0] < 0, 4 * X[:, 2], 0) + rng.normal(0, 0.3, n)

model = RandomForestRegressor(n_estimators=300, random_state=9).fit(X, y)

# GLOBAL: permutation importance, averaged over ALL rows.
global_imp = permutation_importance(model, X, y, n_repeats=8, random_state=9).importances_mean
print("GLOBAL importance (averaged over all rows):")
for i, imp in enumerate(global_imp):
    print(f"  feature_{i}: {imp:.3f}")

# LOCAL: compare two specific rows - one where feature_0 is negative, one positive.
row_neg = np.array([[-2.0, 0.0, 1.5]])   # feature_0 < 0: feature_2 SHOULD matter here
row_pos = np.array([[ 2.0, 0.0, 1.5]])   # feature_0 > 0: feature_2 should NOT matter here

print()
for label, row in [("feature_0 < 0", row_neg), ("feature_0 > 0", row_pos)]:
    base = model.predict(row)[0]
    nudged = row.copy(); nudged[0, 2] += 1.0
    delta = model.predict(nudged)[0] - base
    print(f"row with {label}: nudging feature_2 by +1.0 changes the prediction by {delta:+.3f}")

print()
print("The GLOBAL importance is one averaged number, but feature_2's LOCAL effect")
print("depends entirely on the row - large when feature_0 < 0, near zero otherwise.")


### Checklist item: Risks of over-interpreting explanations

**Approach:**
- Protects evaluation integrity and deployment reliability: explanations describe correlational patterns the model learned, not causal truths about the world, so treating them as causal can lead to wrong real-world decisions.
- Why this is the mathematically right approach: all these methods, importance, PDP, SHAP, are defined purely in terms of the model's own input-output behavior - they mathematically cannot reference anything about the real-world causal process that generated the data, which is exactly why a high attribution score proves association with the model's output, never causation in reality.
- For this checklist item: Explanations reflect the model's learned patterns, not ground truth; a high SHAP value means the feature influenced the model, not that it causally drives the outcome.
- Code walkthrough: Check that feature_3 (labeled as pure noise in the printout) still gets a non-zero importance score despite having no real relationship to y - that's the concrete risk of treating any non-zero importance as proof a feature matters.

**Learn more:**
- Website: [SHAP Documentation](https://shap.readthedocs.io/en/latest/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Explainable+AI+Risks+of+over-interpreting+explanations+machine+learning+theory)

**Trade-offs:**
- Every explanation method in this section - importance, permutation, PDP, SHAP - describes what the model learned, not what's actually true about the world, a distinction that gets easier to forget the more sophisticated the method looks.
- More sophisticated explanation methods like SHAP and PDP feel more rigorous and can create false confidence, so always validate an explanation-driven decision against domain knowledge or a controlled experiment before acting on it.

**Practical software engineering use cases:**
- When to use it: Use this caution any time an explanation output, such as importance or SHAP, is about to justify a real-world action or policy change.
- When not to use it: Don't treat a non-zero importance or SHAP value as proof a feature causally drives the outcome - validate with domain knowledge or a controlled experiment before acting on it.


In [ ]:
# Explainable AI - Risks of over-interpreting explanations (using scikit-learn)

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

# Risk: even a PURE NOISE feature (unrelated to the target) can get a non-zero
# importance score just by chance - explanations describe the model, not causality.

rng = np.random.default_rng(10)
n = 200
X = rng.normal(size=(n, 3))
y = (X[:, 0] > 0).astype(int)                        # only feature 0 actually matters
noise_feature = rng.normal(size=n)                   # a feature that is pure random noise
X_with_noise = np.column_stack([X, noise_feature])

model = RandomForestClassifier(n_estimators=200, random_state=10).fit(X_with_noise, y)
imp = permutation_importance(model, X_with_noise, y, n_repeats=10,
                             random_state=10).importances_mean

print("permutation importance, including a feature that is PURE NOISE:")
for i, val in enumerate(imp):
    tag = " <- pure noise, unrelated to y" if i == 3 else ""
    print(f"  feature_{i}: {val:+.4f}{tag}")

print()
print("The noise feature usually gets a small non-zero score by chance alone.")
print("Lesson: a non-zero importance/SHAP value is not proof a feature is causally")
print("real - validate important-looking features with domain knowledge or an experiment.")


### Checklist item: Fairness and bias in ML

**Approach:**
- Protects deployment reliability and user value: an aggregate metric like overall accuracy can look fine while a model performs far worse for one subgroup than another, so checking metrics across groups is what actually surfaces that kind of harm.
- Why this is the mathematically right approach: computing recall, TP/(TP+FN), separately within each group's own rows is mathematically independent of how well the other group is served - a model can minimize overall error averaged across both groups while that average hides a much larger error rate concentrated entirely in one group, since a weighted average by definition can be pulled toward whichever group is larger or easier.
- For this checklist item: Computing the same metric, like recall, separately for each relevant subgroup and comparing them directly is the basic first step of a fairness audit.
- Code walkthrough: Check that recall_a (0.67) and recall_b (0.33) differ substantially despite being computed from the exact same overall dataset - that gap is invisible in a single aggregate recall number.

**Learn more:**
- Website: [Google: Fairness](https://developers.google.com/machine-learning/crash-course/fairness)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Explainable+AI+Fairness+and+bias+in+ML+machine+learning+theory)

**Trade-offs:**
- Compared to reporting one overall metric, computing metrics per subgroup costs more analysis and requires having group membership data available, which isn't always collected or appropriate to collect.
- Improving fairness across groups, for example by adjusting thresholds per group, can trade off against a small amount of overall aggregate performance, so the right balance is a policy decision, not a purely technical one.

**Practical software engineering use cases:**
- When to use it: Use per-group metric comparisons whenever a model's decisions affect people and a sensitive grouping, like demographic categories, is relevant and available to check against.
- When not to use it: Don't rely on a single overall metric to certify a model as 'fair' - a model can look excellent in aggregate while systematically underperforming for a specific subgroup.


In [ ]:
# Explainable AI - Fairness and bias in ML
records = [
    {"group": "A", "actual": 1, "predicted": 1},
    {"group": "A", "actual": 0, "predicted": 0},
    {"group": "A", "actual": 1, "predicted": 1},
    {"group": "A", "actual": 1, "predicted": 0},
    {"group": "B", "actual": 1, "predicted": 0},
    {"group": "B", "actual": 0, "predicted": 0},
    {"group": "B", "actual": 1, "predicted": 0},
    {"group": "B", "actual": 1, "predicted": 1},
]

def recall_for_group(group):
    rows = [r for r in records if r["group"] == group and r["actual"] == 1]
    correct = sum(1 for r in rows if r["predicted"] == 1)
    return correct / len(rows) if rows else None

recall_a = recall_for_group("A")
recall_b = recall_for_group("B")
print(f"recall for group A: {recall_a:.2f}")
print(f"recall for group B: {recall_b:.2f}")
print(f"recall gap: {abs(recall_a - recall_b):.2f}")
print()
print("Overall accuracy can look fine while one subgroup is served far worse than another -")
print("fairness checks specifically compare metrics ACROSS groups, not just in aggregate.")


In [226]:
# Practice: Explainable AI

# Objective:
# Dataset:
# Approach:
# Key assumptions:
# Evaluation metric:
# Result:
# Lessons learned:

